# 評估範例

1. 周天成使用「殺球」得分佔他總得分的百分比是多少？

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'

# 分母使用「周天成所有得分回合」：getpoint_player 只在回合結束球有值。
chou_scoring_points = df[df['getpoint_player'].eq(PLAYER)].copy()

# 分子使用「周天成本人打出殺球且該拍直接得分」的回合結束球。
chou_smash_scoring_shots = chou_scoring_points[
    chou_scoring_points['player'].eq(PLAYER) &
    chou_scoring_points['type'].eq('殺球')
]

total_chou_points = len(chou_scoring_points)
smash_score_count = len(chou_smash_scoring_shots)
smash_score_percentage = (smash_score_count / total_chou_points * 100) if total_chou_points else 0

print(f'周天成總得分數: {total_chou_points}')
print(f'周天成以殺球直接得分數: {smash_score_count}')
print(f'殺球得分佔周天成總得分比例: {smash_score_percentage:.2f}%')

1. 周天成使用「殺球」得分佔他總得分的百分比是多少？

In [ ]:
# 檢查數據集是否有資料
if len(df) > 0:
    # 過濾周天成的主動得分
    chou_active_wins = df[(df['player'] == 'CHOU Tien Chen') & 
                          (df['getpoint_player'] == 'CHOU Tien Chen')]

    # 計算總主動得分
    total_active_wins_chou = len(chou_active_wins)

    # 過濾出使用殺球得分的情況
    smash_wins_chou = chou_active_wins[chou_active_wins['type'] == '殺球']

    # 計算殺球得分的次數
    smash_wins_count = len(smash_wins_chou)

    # 計算百分比，避免除以零
    percentage_smash_wins = (smash_wins_count / total_active_wins_chou * 100) if total_active_wins_chou > 0 else 0

    # 打印結果
    print(f"周天成使用「殺球」得分佔他總主動得分的百分比是：{percentage_smash_wins:.2f}%")
else:
    print("數據集為空，無法進行分析。")


1. 周天成使用「殺球」得分佔他總得分的百分比是多少？

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
group_cols = ['match_id', 'set', 'rally']

# 取每回合最後一拍，避免重複計算
rally_last = (
    df.sort_values(group_cols + ['ball_round'])
      .groupby(group_cols, as_index=False)
      .last()
)

# 主動得分：最後一拍 player == getpoint_player == 周天成
active_win_df = rally_last[
    (rally_last['player'] == PLAYER) &
    (rally_last['getpoint_player'] == PLAYER)
].copy()

total_active_wins = len(active_win_df)
smash_active_wins = (active_win_df['type'] == '殺球').sum()

if total_active_wins == 0:
    print("沒有周天成主動得分回合。")
else:
    pct = smash_active_wins / total_active_wins * 100
    print(f"周天成總主動得分回合數: {total_active_wins}")
    print(f"其中殺球主動得分回合數: {smash_active_wins}")
    print(f"殺球得分佔總主動得分比例: {pct:.2f}%")

2. 周天成最常使用哪個球種直接得分

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'

# 直接得分需同時滿足：擊球者是周天成，且該回合得分者也是周天成。
chou_direct_scores = df[
    df['player'].eq(PLAYER) &
    df['getpoint_player'].eq(PLAYER)
].copy()

shot_score_counts = chou_direct_scores['type'].value_counts()

if shot_score_counts.empty:
    print('沒有周天成直接得分的資料。')
else:
    top_shot = shot_score_counts.idxmax()
    top_count = int(shot_score_counts.max())
    print('周天成各球種直接得分次數:')
    print(shot_score_counts)
    print(f'周天成最常使用「{top_shot}」直接得分，共 {top_count} 次。')

    fig, ax = plt.subplots(figsize=(9, 5))
    shot_score_counts.plot(kind='bar', ax=ax, color='#4C78A8')
    ax.set_title('周天成各球種直接得分次數')
    ax.set_xlabel('球種')
    ax.set_ylabel('直接得分次數')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

3. 針對周天成所有「殺球」，繪製其落點熱區圖。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PLAYER = 'CHOU Tien Chen'

# 「落點」使用 landing_x / landing_y / landing_area，不使用站位或擊球點欄位。
chou_smashes = df[
    df['player'].eq(PLAYER) &
    df['type'].eq('殺球')
].dropna(subset=['landing_x', 'landing_y']).copy()

print(f'周天成殺球總數: {len(chou_smashes)}')
print('周天成殺球落點區域次數:')
print(chou_smashes['landing_area'].value_counts().sort_index())

if chou_smashes.empty:
    print('沒有周天成殺球落點資料。')
else:
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.histplot(
        data=chou_smashes,
        x='landing_x',
        y='landing_y',
        bins=12,
        cmap='Reds',
        cbar=True,
        ax=ax
    )
    ax.set_title('周天成殺球落點熱區圖')
    ax.set_xlabel('landing_x')
    ax.set_ylabel('landing_y')
    plt.tight_layout()

4. 分析周天成在雙方都達18分以上時的球種分布。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
PLAYER_SCORE_COL = 'CHOU Tien Chen_score'
OPPONENT_SCORE_COL = 'Kento MOMOTA_score'

# 使用實際比分欄位判斷「雙方都達 18 分以上」。
critical_chou_shots = df[
    df['player'].eq(PLAYER) &
    (df[PLAYER_SCORE_COL] >= 18) &
    (df[OPPONENT_SCORE_COL] >= 18)
].copy()

shot_distribution = critical_chou_shots['type'].value_counts()
shot_percentage = critical_chou_shots['type'].value_counts(normalize=True).mul(100)

print(f'雙方都達18分以上時，周天成擊球筆數: {len(critical_chou_shots)}')
print('球種次數:')
print(shot_distribution)
print('球種比例(%):')
print(shot_percentage.round(2))

if shot_distribution.empty:
    print('沒有符合條件的資料。')
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    shot_distribution.plot(kind='bar', ax=ax, color='#F58518')
    ax.set_title('雙方18分以上時周天成球種分布')
    ax.set_xlabel('球種')
    ax.set_ylabel('次數')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

5. 幫我繪製周天成得分時球的落點熱區圖

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PLAYER = 'CHOU Tien Chen'

# 題目問「周天成得分時球的落點」，這裡採用周天成本人打出的直接得分球。
chou_scoring_shots = df[
    df['player'].eq(PLAYER) &
    df['getpoint_player'].eq(PLAYER)
].dropna(subset=['landing_x', 'landing_y']).copy()

print(f'周天成直接得分球數: {len(chou_scoring_shots)}')
print('周天成直接得分球落點區域次數:')
print(chou_scoring_shots['landing_area'].value_counts().sort_index())

if chou_scoring_shots.empty:
    print('沒有周天成直接得分球的落點資料。')
else:
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.histplot(
        data=chou_scoring_shots,
        x='landing_x',
        y='landing_y',
        bins=12,
        cmap='Blues',
        cbar=True,
        ax=ax
    )
    ax.set_title('周天成直接得分球落點熱區圖')
    ax.set_xlabel('landing_x')
    ax.set_ylabel('landing_y')
    plt.tight_layout()

5. 幫我繪製周天成得分時球的落點熱區圖

In [ ]:
# 確認數據集的大小
total_rows = len(df)
print(f"Total rows in the dataset: {total_rows}")

# 篩選出周天成得分的記錄
chou_tien_chen_scoring_df = df[df['getpoint_player'] == 'CHOU Tien Chen']

# 確認篩選後的數據集大小
filtered_rows = len(chou_tien_chen_scoring_df)
print(f"Rows where CHOU Tien Chen scores: {filtered_rows}")

# 檢查是否有數據可供繪圖
if filtered_rows > 0:
    # 打印落點座標的基本統計信息
    print("Landing X Coordinate Statistics")
    print(chou_tien_chen_scoring_df['landing_x'].describe())

    print("Landing Y Coordinate Statistics")
    print(chou_tien_chen_scoring_df['landing_y'].describe())

    # 繪製落點的熱區圖
    fig, ax = plt.subplots(figsize=(10, 6))
    
    sns.kdeplot(
        x=chou_tien_chen_scoring_df['landing_x'], 
        y=chou_tien_chen_scoring_df['landing_y'], 
        cmap="Reds", 
        shade=True, 
        shade_lowest=False,  # `shade_lowest` 已被移除，使用 `thresh` 或其他替代參數
        ax=ax
    )
    
    ax.set_title('周天成得分時球的落點熱區圖')
    ax.set_xlabel('落點 X 座標')
    ax.set_ylabel('落點 Y 座標')
    
    plt.tight_layout()

else:
    print("No data available for CHOU Tien Chen's scores.")


6. 在第一場次中，每回合用的拍數，繪圖

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 第一場次使用資料中的最小 match_id
first_match_id = sorted(df['match_id'].dropna().unique())[0]
first_match_df = df[df['match_id'].eq(first_match_id)].copy()

# 每回合拍數使用該 rally 內最大的 ball_round。
rally_lengths = (
    first_match_df
    .groupby(['set', 'rally'], as_index=False)['ball_round']
    .max()
    .rename(columns={'ball_round': 'rally_length'})
)
rally_lengths['rally_index'] = range(1, len(rally_lengths) + 1)

print(f'第一場次 match_id: {first_match_id}')
print(f'第一場次回合數: {len(rally_lengths)}')
print(rally_lengths[['set', 'rally', 'rally_length']].head())
print(rally_lengths['rally_length'].describe())

fig, ax = plt.subplots(figsize=(12, 5))
for set_id, group in rally_lengths.groupby('set'):
    ax.plot(group['rally_index'], group['rally_length'], marker='o', linewidth=1, label=f'Set {set_id}')
ax.set_title('第一場次每回合拍數')
ax.set_xlabel('回合順序')
ax.set_ylabel('拍數')
ax.legend(title='局數')
ax.grid(alpha=0.3)
plt.tight_layout()

6. 在第一場次中，每回合用的拍數，繪圖

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 直接假設第一場比賽的 match_id = 1
match_1_df = df[df['match_id'] == 1].copy()

rally_shot_counts = (
    match_1_df.groupby(['set', 'rally'])['ball_round']
    .max()
    .reset_index(name='shot_count')
)

print("第一場比賽每回合拍數：")
print(rally_shot_counts)

plt.figure(figsize=(12, 6))
plt.bar(
    range(1, len(rally_shot_counts) + 1),
    rally_shot_counts['shot_count'],
    color='#4E79A7'
)
plt.title('第一場比賽每回合拍數')
plt.xlabel('回合序號')
plt.ylabel('拍數')
plt.tight_layout()
plt.show()

7. 繪製周天成殺球時的站點熱區圖

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PLAYER = 'CHOU Tien Chen'

# 「站點」使用 player_location_x / player_location_y / player_location_area。
# 不使用 hit_area，因為 hit_area 是球被擊打的位置，不是球員站位。
chou_smash_positions = df[
    df['player'].eq(PLAYER) &
    df['type'].eq('殺球')
].dropna(subset=['player_location_x', 'player_location_y']).copy()

print(f'周天成殺球筆數: {len(chou_smash_positions)}')
print('周天成殺球時站位區域次數:')
print(chou_smash_positions['player_location_area'].value_counts().sort_index())

if chou_smash_positions.empty:
    print('沒有周天成殺球站位資料。')
else:
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.histplot(
        data=chou_smash_positions,
        x='player_location_x',
        y='player_location_y',
        bins=12,
        cmap='Greens',
        cbar=True,
        ax=ax
    )
    ax.set_title('周天成殺球時站點熱區圖')
    ax.set_xlabel('player_location_x')
    ax.set_ylabel('player_location_y')
    plt.tight_layout()

8. 當周天成在「後場」擊球時，他最常使用哪三種球種?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
BACK_COURT_AREAS = [1, 2, 3, 4]

# 「在後場擊球」指球的擊球點在後場，因此使用 hit_area，不使用 player_location_area。
chou_backcourt_hits = df[
    df['player'].eq(PLAYER) &
    df['hit_area'].isin(BACK_COURT_AREAS)
].copy()

shot_counts = chou_backcourt_hits['type'].value_counts()
top_three_shots = shot_counts.head(3)

print(f'周天成後場擊球總數: {len(chou_backcourt_hits)}')
print('周天成後場擊球球種次數:')
print(shot_counts)
print('最常使用的前三種球種:')
print(top_three_shots)

if top_three_shots.empty:
    print('沒有周天成後場擊球資料。')
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    top_three_shots.plot(kind='bar', ax=ax, color='#54A24B')
    ax.set_title('周天成後場擊球最常使用前三種球種')
    ax.set_xlabel('球種')
    ax.set_ylabel('次數')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()

9. 分析周天成站在前中後場，分別的得分與失分數

In [ ]:
import numpy as np

# ============================================================
# 分析周天成在前場、中場、後場的站位得分數與失分數
# ============================================================

# 1. 驗證數據量
print(f"總資料筆數: {len(df)}")
assert len(df) > 0, "DataFrame 為空！"

# 2. 定義場地區域分類（根據 Court Grid Definitions）
back_court_zones  = [1, 2, 3, 4]                                          # 後場
mid_court_zones   = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]          # 中場
front_court_zones = [17, 18, 19, 20, 21, 22, 23, 24]                      # 前場
# Zones 25-32 為出界，不計入分析

def classify_zone(zone):
    """將 Zone ID 分類為前場/中場/後場，出界或無效回傳 None"""
    if pd.isna(zone):
        return None
    z = int(zone)
    if z in back_court_zones:
        return '後場'
    elif z in mid_court_zones:
        return '中場'
    elif z in front_court_zones:
        return '前場'
    else:
        return None  # 出界或不在有效場內

# ============================================================
# 3. 篩選有得分事件的球（rally 結束球，getpoint_player 不為空）
# ============================================================
df_point = df[df['getpoint_player'].notna()].copy()
print(f"\n有得分事件的球數（rally 結束球）: {len(df_point)}")

# ============================================================
# 4. 取得「周天成的站位區域」
#    - 該球 player == 'CHOU Tien Chen'   → 使用 player_location_area
#    - 該球 player == 'Kento MOMOTA'     → 周天成是 opponent，使用 opponent_location_area
# ============================================================
chou = 'CHOU Tien Chen'

df_point['chou_location_area'] = df_point.apply(
    lambda row: row['player_location_area'] if row['player'] == chou
                else row['opponent_location_area'],
    axis=1
)

# 5. 分類周天成站位場區（使用 pd.notna() 避免 NaN 問題）
df_point['chou_zone_category'] = df_point['chou_location_area'].apply(classify_zone)

print(f"\n周天成站位區域分佈（含出界/NaN）:")
print(df_point['chou_zone_category'].value_counts(dropna=False))

# ============================================================
# 6. 判斷該 rally 結束時，周天成是得分還是失分
# ============================================================
df_point['chou_result'] = df_point['getpoint_player'].apply(
    lambda gp: '得分' if gp == chou else '失分'
)

print(f"\n周天成得失分總覽:")
print(df_point['chou_result'].value_counts())

# ============================================================
# 7. 篩選有效場地區域（排除出界/None）
# ============================================================
df_valid = df_point[df_point['chou_zone_category'].notna()].copy()
print(f"\n有效站位（前/中/後場）的球數: {len(df_valid)}")
print(f"站位在出界區域的球數（已排除）: {len(df_point) - len(df_valid)}")

# ============================================================
# 8. 統計各站位區域的得分數與失分數
# ============================================================
zone_result = df_valid.groupby(['chou_zone_category', 'chou_result']).size().unstack(fill_value=0)

# 確保欄位齊全（避免某區域只有得分或只有失分）
for col in ['得分', '失分']:
    if col not in zone_result.columns:
        zone_result[col] = 0

# 依照前→中→後場順序排序
zone_order = ['前場', '中場', '後場']
zone_result = zone_result.reindex([z for z in zone_order if z in zone_result.index])

# 計算總計與得分率
zone_result['總計']      = zone_result['得分'] + zone_result['失分']
zone_result['得分率(%)'] = (zone_result['得分'] / zone_result['總計'] * 100).round(1)

print(f"\n【周天成各場區站位得失分完整統計】")
print("=" * 55)
print(zone_result[['得分', '失分', '總計', '得分率(%)']].to_string())

# ============================================================
# 9. 視覺化
# ============================================================
zones = zone_result.index.tolist()
x     = np.arange(len(zones))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("周天成各場區站位得失分分析（所有場次）",
             fontsize=16, fontweight='bold')

# ── 圖1：各場區得分數 vs 失分數（分組長條圖）──
ax1 = axes[0]
bars_win  = ax1.bar(x - width/2, zone_result['得分'], width,
                    label='得分', color='#2ECC71', edgecolor='white', linewidth=0.8)
bars_lose = ax1.bar(x + width/2, zone_result['失分'], width,
                    label='失分', color='#E74C3C', edgecolor='white', linewidth=0.8)

# 標示數值於長條頂端
for bar in bars_win:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2., h + 0.3,
             f'{int(h)}', ha='center', va='bottom', fontsize=13,
             fontweight='bold', color='#1A8A4A')

for bar in bars_lose:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2., h + 0.3,
             f'{int(h)}', ha='center', va='bottom', fontsize=13,
             fontweight='bold', color='#A93226')

ax1.set_xlabel('站位場區', fontsize=12)
ax1.set_ylabel('球數（局末球次數）', fontsize=12)
ax1.set_title('各場區站位：得分數 vs 失分數', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(zones, fontsize=12)
ax1.legend(fontsize=11)
ax1.set_ylim(0, max(zone_result[['得分', '失分']].values.flatten()) * 1.2)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── 圖2：各場區站位的得分率（水平條形圖）──
ax2 = axes[1]
rate_colors = ['#2ECC71' if r >= 50 else '#E74C3C'
               for r in zone_result['得分率(%)']]
bars_rate = ax2.barh(zones, zone_result['得分率(%)'],
                     color=rate_colors, edgecolor='white',
                     linewidth=0.8, height=0.5)

# 標示得分率與得失分明細
for bar, rate, win, lose in zip(
        bars_rate, zone_result['得分率(%)'],
        zone_result['得分'], zone_result['失分']):
    w = bar.get_width()
    ax2.text(w + 1.0, bar.get_y() + bar.get_height() / 2.,
             f'{rate}%  (得 {int(win)} / 失 {int(lose)})',
             ha='left', va='center', fontsize=11, fontweight='bold')

# 50% 基準線
ax2.axvline(x=50, color='gray', linestyle='--',
            linewidth=1.5, alpha=0.7, label='50% 基準線')

ax2.set_xlabel('得分率 (%)', fontsize=12)
ax2.set_title('各場區站位得分率', fontsize=13, fontweight='bold')
ax2.set_xlim(0, 115)   # 留右側空間給標籤
ax2.legend(fontsize=10, loc='lower right')
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_yticklabels(zones, fontsize=12)

plt.tight_layout()
print("\n✅ 圖表已生成完畢")

10. 當周天成站在前場時，他最主要的得分球種?

In [ ]:
# 確保資料集不為空
if len(df) > 0:
    # 過濾周天成在前場位置且是得分者的資料
    front_court_zones = [17, 18, 19, 20, 21, 22, 23, 24]
    filtered_df = df[
        (df['player'] == 'CHOU Tien Chen') &
        (df['player_location_area'].isin(front_court_zones)) &
        (df['getpoint_player'] == df['player'])
    ]

    # 確保過濾後的資料集不為空
    if len(filtered_df) > 0:
        # 計算得分的球種出現次數
        type_counts = filtered_df['type'].value_counts()
        
        # 找出最主要的得分方式
        primary_score_method = type_counts.idxmax()

        # 印出周天成在前場最主要的得分方式
        print(f"周天成在前場最主要的得分方式是: {primary_score_method}，共得分 {type_counts.max()} 次。")
    else:
        print("沒有發現符合條件的得分數據。")
else:
    print("資料集為空。")

11. 計算周天成站在前場擊球時的落點分布。

In [ ]:
# 確認數據是否存在
if len(df) > 0:
    # 過濾周天成站在前場擊球的數據
    front_court_areas = [17, 18, 19, 20, 21, 22, 23, 24]
    filtered_df = df[(df['player'] == 'CHOU Tien Chen') & (df['player_location_area'].isin(front_court_areas))]

    if len(filtered_df) > 0:
        # 計算各落點區域的出現頻率
        landing_area_counts = filtered_df['landing_area'].value_counts().sort_index()

        # 可視化落點分布
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.bar(landing_area_counts.index, landing_area_counts.values, color='skyblue')
        ax.set_xlabel('落點區域代碼', fontsize=12)
        ax.set_ylabel('頻率', fontsize=12)
        ax.set_title('周天成在前場擊球的落點分布', fontsize=14)
        ax.set_xticks(landing_area_counts.index)  # 使用正確的區域代碼作為ticks
        ax.set_xticklabels(landing_area_counts.index)
        plt.xticks(rotation=45)
        plt.tight_layout()

        print(f"周天成在前場擊球時的落點頻率如下：\n{landing_area_counts}")
    else:
        print("無周天成在前場擊球的相關數據。")
else:
    print("數據框 df 為空。")

12. 分析周天成站在前中後場擊球時，自己分別的移動距離

In [ ]:
import numpy as np

# 確認數據是否存在
if len(df) > 0:
    # 計算移動距離
    df['move_distance'] = np.sqrt(df['player_move_x']**2 + df['player_move_y']**2)

    # 前場位置定義
    front_court_zones = [17, 18, 19, 20, 21, 22, 23, 24]
    # 中場位置定義
    mid_court_zones = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
    # 後場位置定義
    back_court_zones = [1, 2, 3, 4]

    # 過濾周天成的數據
    chou_df = df[df['player'] == 'CHOU Tien Chen']

    # 計算各位置的平均移動距離
    front_court_distance = chou_df[chou_df['player_location_area'].isin(front_court_zones)]['move_distance'].mean()
    mid_court_distance = chou_df[chou_df['player_location_area'].isin(mid_court_zones)]['move_distance'].mean()
    back_court_distance = chou_df[chou_df['player_location_area'].isin(back_court_zones)]['move_distance'].mean()

    print(f"周天成在前場的平均移動距離: {front_court_distance:.2f}")
    print(f"周天成在中場的平均移動距離: {mid_court_distance:.2f}")
    print(f"周天成在後場的平均移動距離: {back_court_distance:.2f}")
else:
    print("數據不包含任何行為記錄。")

13. 幫我統整周天成所有球種的使用比例，以及每種球種的「得分次數」和「失誤次數」。

In [ ]:
# 確認數據是否存在
if len(df) > 0:
    # 篩選出周天成的數據
    chou_df = df[df['player'] == 'CHOU Tien Chen']

    # 計算球種使用比例
    type_counts = chou_df['type'].value_counts(normalize=True) * 100
    type_counts = type_counts.sort_index()

    # 計算每種球種的得分次數
    win_counts = chou_df[(chou_df['getpoint_player'] == 'CHOU Tien Chen')].groupby('type')['type'].count()

    # 計算每種球種的失誤次數
    lose_counts = chou_df[(chou_df['getpoint_player'] != 'CHOU Tien Chen') & (~chou_df['lose_reason'].isna())].groupby('type')['type'].count()

    # 圖表合併三個信息
    fig, ax1 = plt.subplots(figsize=(12, 8))

    # 繪製使用比例
    ax1.bar(type_counts.index, type_counts.values, color='b', alpha=0.6, label='使用比例 (%)')
    ax1.set_xlabel('球種')
    ax1.set_ylabel('使用比例 (%)', color='b')
    ax1.legend(loc='upper left')

    # 第二軸，共用 x 軸
    ax2 = ax1.twinx()
    ax2.plot(win_counts.index, win_counts.values, label='得分次數', color='g', marker='o')
    ax2.plot(lose_counts.index, lose_counts.values, label='失誤次數', color='r', marker='x')
    ax2.set_ylabel('次數', color='k')
    ax2.legend(loc='upper right')

    plt.title('周天成球種使用比例、得分次數和失誤次數')
    plt.xticks(rotation=45)
    plt.tight_layout()

    # Print important information
    print("球種使用比例 (%):")
    print(type_counts)
    print("\n每種球種的得分次數:")
    print(win_counts)
    print("\n每種球種的失誤次數:")
    print(lose_counts)

    # 返回繪製的圖表
    fig

13. 幫我統整周天成所有球種的使用比例，以及每種球種的「得分次數」和「失誤次數」。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'

chou_df = df[df['player'] == PLAYER].copy()

# 用最後一拍失分近似失誤
chou_df['is_error'] = chou_df['getpoint_player'] == OPPONENT
chou_df['is_score'] = chou_df['getpoint_player'] == PLAYER

summary = chou_df.groupby('type').agg(
    使用次數=('type', 'count'),
    得分次數=('is_score', 'sum'),
    失誤次數=('is_error', 'sum')
).reset_index()

summary['使用比例'] = summary['使用次數'] / summary['使用次數'].sum()

print("周天成各球種的使用比例、得分次數與失誤次數：")
print(summary)

14. 幫我畫周天成失分原因的圓餅圖

In [ ]:
# ========================
# 分析周天成失分原因分佈
# ========================

player_name = 'CHOU Tien Chen'
opponent_name = 'Kento MOMOTA'

# 1. 驗證數據量
print(f"Total rows: {len(df)}")
assert len(df) > 0, "DataFrame is empty!"

# 2. 篩選所有「周天成失分」的球（getpoint_player == 'Kento MOMOTA'）
# 這些球是每個 rally 的最後一球，且對手得分
chou_lost_df = df[df['getpoint_player'] == opponent_name].copy()
print(f"\n周天成失分總次數（rally結束球）: {len(chou_lost_df)}")

# 3. 確認 lose_reason / win_reason 覆蓋情況
print(f"\n有 lose_reason 的球數: {chou_lost_df['lose_reason'].notna().sum()}")
print(f"有 win_reason 的球數: {chou_lost_df['win_reason'].notna().sum()}")
print(f"兩者皆無的球數: {(chou_lost_df['lose_reason'].isna() & chou_lost_df['win_reason'].isna()).sum()}")

# 4. 顯示這些失分球的 player 分佈（是周天成打失誤，還是桃田打致勝）
print(f"\n失分球由誰打出:")
print(chou_lost_df['player'].value_counts())

# 5. 分類失分原因
# 類別一：周天成自身打球失誤（lose_reason 有值，player == CHOU）
chou_error_df = chou_lost_df[
    (chou_lost_df['player'] == player_name) &
    (chou_lost_df['lose_reason'].notna())
]

# 類別二：被對手主動致勝（player == MOMOTA，win_reason 有值）
opponent_winner_df = chou_lost_df[
    (chou_lost_df['player'] == opponent_name) &
    (chou_lost_df['win_reason'].notna())
]

# 類別三：其他（兩者皆無或特殊情況）
other_df = chou_lost_df[
    ~(
        ((chou_lost_df['player'] == player_name) & (chou_lost_df['lose_reason'].notna())) |
        ((chou_lost_df['player'] == opponent_name) & (chou_lost_df['win_reason'].notna()))
    )
]

print(f"\n=== 失分類別統計 ===")
print(f"周天成主動失誤（lose_reason）: {len(chou_error_df)}")
print(f"  - 細分: {chou_error_df['lose_reason'].value_counts().to_dict()}")
print(f"被桃田致勝球得分（win_reason）: {len(opponent_winner_df)}")
print(f"  - 細分: {opponent_winner_df['win_reason'].value_counts().to_dict()}")
print(f"其他/未分類: {len(other_df)}")
if len(other_df) > 0:
    print("其他的 player 與 lose/win reason:")
    print(other_df[['match_id','set','rally','ball_round','player','lose_reason','win_reason']].head(20))

# ========================
# 6. 建立詳細失分原因標籤
# ========================
def classify_loss_reason(row):
    """
    將每一個「周天成失分」的球分類為具體失分原因
    """
    if row['player'] == player_name and pd.notna(row['lose_reason']):
        # 周天成自身打球失誤，細分 lose_reason
        return f"主動失誤：{row['lose_reason']}"
    elif row['player'] == opponent_name and pd.notna(row['win_reason']):
        # 被桃田主動致勝
        return f"被致勝：{row['win_reason']}"
    else:
        return "其他失分"

chou_lost_df['loss_category'] = chou_lost_df.apply(classify_loss_reason, axis=1)

# 統計各類別
loss_counts = chou_lost_df['loss_category'].value_counts()
print(f"\n=== 詳細失分原因統計 ===")
print(loss_counts)
print(f"總計: {loss_counts.sum()}")

# ========================
# 7. 高層次分類（合併為大類）
# ========================
def high_level_category(label):
    if label.startswith('主動失誤'):
        return label  # 保留細分
    elif label.startswith('被致勝'):
        return label  # 保留細分
    else:
        return '其他失分'

# 合併小比例類別（< 3%）為「其他」
total = loss_counts.sum()
loss_pct = loss_counts / total * 100

# 判斷是否有微小比例需合併
threshold = 3.0  # 低於3%合併
small_cats = loss_pct[loss_pct < threshold].index.tolist()
print(f"\n比例 < {threshold}% 的類別（將合併為『其他』）: {small_cats}")

# 合併小類別
loss_counts_merged = loss_counts.copy().astype(object)
if small_cats:
    other_count = loss_counts[small_cats].sum()
    loss_counts_merged = loss_counts_merged.drop(small_cats)
    if '其他失分' in loss_counts_merged.index:
        loss_counts_merged['其他失分'] += other_count
    else:
        loss_counts_merged['其他失分'] = other_count

loss_counts_merged = loss_counts_merged.sort_values(ascending=False)
print(f"\n=== 合併後失分原因統計 ===")
print(loss_counts_merged)

# ========================
# 8. 視覺化 - 圓餅圖
# ========================

# 顏色配置：主動失誤系列用暖色，被致勝系列用冷色
import numpy as np

categories = loss_counts_merged.index.tolist()
values = loss_counts_merged.values.tolist()
total_points = sum(values)

# 設定顏色
color_map = {
    '主動失誤：出界': '#E74C3C',
    '主動失誤：未過網': '#E67E22',
    '主動失誤：掛網': '#F39C12',
    '主動失誤：落點判斷失誤': '#D35400',
    '主動失誤：犯規': '#C0392B',
    '被致勝：落地致勝': '#2980B9',
    '被致勝：落地判斷失誤': '#1ABC9C',
    '其他失分': '#95A5A6',
}
colors = [color_map.get(cat, '#BDC3C7') for cat in categories]

# 計算百分比標籤
pct_vals = [v / total_points * 100 for v in values]

# 找最大類別 explode（突出顯示）
max_idx = pct_vals.index(max(pct_vals))
explode = [0.05 if i == max_idx else 0 for i in range(len(categories))]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# --- 左側：圓餅圖 ---
ax1 = axes[0]
wedges, texts, autotexts = ax1.pie(
    values,
    labels=None,  # 標籤放在圖例
    colors=colors,
    explode=explode,
    autopct=lambda pct: f'{pct:.1f}%\n({int(round(pct/100*total_points))}分)',
    pctdistance=0.75,
    startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=10)
)

# 調整百分比文字顏色
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight('bold')
    autotext.set_color('white')

# 標題
ax1.set_title(
    f'周天成失分原因分佈\n（全場次，共 {total_points} 分失分）',
    fontsize=14, fontweight='bold', pad=20
)

# 圖例
legend_labels = [f'{cat} ({v}分, {p:.1f}%)' 
                 for cat, v, p in zip(categories, values, pct_vals)]
ax1.legend(
    wedges, legend_labels,
    title='失分原因',
    loc='lower center',
    bbox_to_anchor=(0.5, -0.35),
    fontsize=9,
    title_fontsize=10,
    ncol=1
)

# --- 右側：橫條圖（輔助視覺）---
ax2 = axes[1]

# 依數值排序（從大到小）
sorted_pairs = sorted(zip(values, categories, colors, pct_vals), reverse=True)
sorted_vals, sorted_cats, sorted_colors, sorted_pcts = zip(*sorted_pairs)

bars = ax2.barh(
    range(len(sorted_cats)),
    sorted_vals,
    color=sorted_colors,
    edgecolor='white',
    linewidth=1.5,
    height=0.6
)

# 在條形右端加數值標籤
for i, (bar, val, pct) in enumerate(zip(bars, sorted_vals, sorted_pcts)):
    ax2.text(
        val + 0.3, bar.get_y() + bar.get_height() / 2,
        f'{val} 分 ({pct:.1f}%)',
        va='center', ha='left', fontsize=10, fontweight='bold', color='#2C3E50'
    )

ax2.set_yticks(range(len(sorted_cats)))
ax2.set_yticklabels(sorted_cats, fontsize=10)
ax2.set_xlabel('失分次數（分）', fontsize=11)
ax2.set_title('各失分原因細項排名', fontsize=13, fontweight='bold')
ax2.set_xlim(0, max(sorted_vals) * 1.35)
ax2.invert_yaxis()  # 最大值在上
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# 加上總計標註
ax2.axvline(x=np.mean(list(sorted_vals)), color='gray', linestyle=':', alpha=0.6, label=f'平均: {np.mean(list(sorted_vals)):.1f}分')
ax2.legend(fontsize=9)

plt.suptitle('周天成失分原因全面分析', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()

print("\n✅ 圖表已生成")
print(f"\n📊 失分摘要:")
print(f"   總失分: {total_points} 分")
print(f"   主動失誤總計: {sum(v for c, v in zip(categories, values) if '主動失誤' in c)} 分")
print(f"   被對手致勝總計: {sum(v for c, v in zip(categories, values) if '被致勝' in c)} 分")


14. 幫我畫周天成失分原因的圓餅圖

In [ ]:
# 確保有資料可用
if len(df) > 0:
    # 找出周天成失分的情況
    chou_lost_points = df[(df['player'] == 'CHOU Tien Chen') & (df['getpoint_player'] == 'Kento MOMOTA')]
    
    # 過濾lose_reason不為NaN的資料
    chou_lose_reasons = chou_lost_points.dropna(subset=['lose_reason'])

    # 計算每個失分原因的次數
    lose_reason_counts = chou_lose_reasons['lose_reason'].value_counts()

    # 確保有失分原因資料
    if not lose_reason_counts.empty:
        # 繪製圓餅圖
        fig, ax = plt.subplots()
        ax.pie(lose_reason_counts, labels=lose_reason_counts.index, autopct='%1.1f%%', startangle=90, counterclock=False)
        ax.set_title('周天成的失分原因分佈')
        plt.tight_layout()
    else:
        print("沒有失分原因的相關資料。")
else:
    print("資料集為空。")

15. 當周天成的對手處於「前場」時，周天成打球的落點分布

In [ ]:
import platform
import matplotlib.pyplot as plt
import seaborn as sns

# 字體設定
s = platform.system()
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang TC'] if s=='Darwin' else ['Microsoft JhengHei', 'SimHei'] if s=='Windows' else ['WenQuanYi Zen Hei']
plt.rcParams['axes.unicode_minus'] = False

# 確保數據框有數據
if len(df) > 0:
    # 篩選條件：周天成為 player 且對手在前場區域
    filtered_df = df[(df['player'] == 'CHOU Tien Chen') & (df['opponent_location_area'].isin([17, 18, 19, 20, 21, 22, 23, 24]))]

    # 確保篩選後數據框有數據
    if len(filtered_df) > 0:
        # 繪製散點圖顯示擊球落點
        plt.figure(figsize=(12, 8))
        sns.scatterplot(data=filtered_df, x='landing_x', y='landing_y', hue='landing_area', palette='viridis', edgecolor=None)
        plt.title('周天成當對手在前場時的擊球落點分布')
        plt.xlabel('落點 X 座標')
        plt.ylabel('落點 Y 座標')
        plt.grid(True)
        plt.tight_layout()
        plt.legend(title='落點區域代碼', loc='upper right')
        fig = plt.gcf()  # 獲得當前圖表對象以便外部操作
    else:
        print("沒有符合條件的數據可以進行分析。")
else:
    print("數據框為空，無法進行分析。")

16. 給我周天成最常見的失分原因

In [ ]:
# ============================================================
# 周天成失分原因分佈分析
# ============================================================

# 驗證數據量
print(f"總資料筆數: {len(df)}")
print(f"match_id 列表: {sorted(df['match_id'].unique())}")

# ============================================================
# Step 1: 篩選有得分記錄的球 (rally 結束那一球)
# ============================================================
scored_df = df[df['getpoint_player'].notna()].copy()
print(f"\n有得分記錄的球數: {len(scored_df)}")

# ============================================================
# Step 2: 篩選「周天成失分」的球
# 條件: getpoint_player != 'CHOU Tien Chen'，即對手(Kento MOMOTA)得分
# ============================================================
chou_lose_df = scored_df[scored_df['getpoint_player'] != 'CHOU Tien Chen'].copy()
print(f"周天成失分的球數: {len(chou_lose_df)}")

# ============================================================
# Step 3: 建立失分原因分類
# ============================================================
# 根據 player (誰打這球) + lose_reason/win_reason 分類

def classify_lose_reason(row):
    """
    分類周天成失分原因:
    
    Case A: 周天成自己打這球 → 主動失誤
        - 出界 / 掛網 / 未過網 / 落點判斷失誤 / 犯規
    Case B: 對手(Kento MOMOTA)打這球 → 被對手得分
        - 落地致勝 → 對手主動得分
        - 落地判斷失誤 → 周天成判斷失誤
    """
    player = row['player']
    lose_reason = row['lose_reason']
    win_reason = row['win_reason']
    
    if player == 'CHOU Tien Chen':
        # 周天成自己打這球造成失分 → 主動失誤
        if pd.notna(lose_reason):
            if lose_reason == '出界':
                return '主動失誤 - 出界'
            elif lose_reason == '掛網':
                return '主動失誤 - 掛網'
            elif lose_reason == '未過網':
                return '主動失誤 - 未過網'
            elif lose_reason == '落點判斷失誤':
                return '主動失誤 - 落點判斷失誤'
            elif lose_reason == '犯規':
                return '主動失誤 - 犯規'
            else:
                return f'主動失誤 - {lose_reason}'
        elif pd.notna(win_reason):
            return f'其他 - {win_reason}'
        else:
            return '主動失誤 - 未知'
    
    elif player == 'Kento MOMOTA':
        # 對手打這球得分 → 被對手得分
        if pd.notna(win_reason):
            if win_reason == '落地致勝':
                return '被對手得分 - 落地致勝'
            elif win_reason == '落地判斷失誤':
                return '被對手得分 - 落點判斷失誤'
            else:
                return f'被對手得分 - {win_reason}'
        elif pd.notna(lose_reason):
            # 周天成接球失誤
            return f'被迫失誤 - {lose_reason}'
        else:
            return '被對手得分 - 未知'
    
    else:
        return '未知'

chou_lose_df['lose_category'] = chou_lose_df.apply(classify_lose_reason, axis=1)

# ============================================================
# Step 4: 統計各失分原因
# ============================================================
lose_counts = chou_lose_df['lose_category'].value_counts().reset_index()
lose_counts.columns = ['失分原因', '次數']
lose_counts['佔比(%)'] = (lose_counts['次數'] / len(chou_lose_df) * 100).round(2)

print("\n========================================")
print("📊 周天成失分原因統計 (由高到低)")
print("========================================")
print(lose_counts.to_string(index=False))
print(f"\n總失分次數: {len(chou_lose_df)}")

# ============================================================
# Step 5: 建立「大類別」統計 (合併子類別)
# ============================================================
def classify_major_category(category):
    if '主動失誤' in category:
        return '主動失誤'
    elif '被對手得分' in category:
        return '被對手得分'
    elif '被迫失誤' in category:
        return '被迫失誤'
    else:
        return '其他'

chou_lose_df['major_category'] = chou_lose_df['lose_category'].apply(classify_major_category)
major_counts = chou_lose_df['major_category'].value_counts().reset_index()
major_counts.columns = ['大類別', '次數']
major_counts['佔比(%)'] = (major_counts['次數'] / len(chou_lose_df) * 100).round(2)

print("\n========================================")
print("📊 周天成失分大類別統計")
print("========================================")
print(major_counts.to_string(index=False))

# ============================================================
# Step 6: 視覺化 - 雙圖表
# ============================================================
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle('周天成失分原因分佈分析 (所有場次)', fontsize=18, fontweight='bold', y=1.01)

# ────────────────────────────────────────────────────────
# 圖1: 大類別 - 甜甜圈圖 (Donut Chart)
# ────────────────────────────────────────────────────────
ax1 = axes[0]

major_labels = major_counts['大類別'].tolist()
major_values = major_counts['次數'].tolist()

colors_major = ['#E74C3C', '#3498DB', '#F39C12', '#95A5A6'][:len(major_labels)]

wedges, texts, autotexts = ax1.pie(
    major_values,
    labels=major_labels,
    autopct=lambda pct: f'{pct:.1f}%\n({int(round(pct/100*sum(major_values)))}次)',
    colors=colors_major,
    startangle=90,
    wedgeprops=dict(width=0.6, edgecolor='white', linewidth=2),
    textprops={'fontsize': 11}
)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')

# 中心文字
ax1.text(0, 0, f'總失分\n{len(chou_lose_df)}次', 
         ha='center', va='center', fontsize=14, fontweight='bold', color='#2C3E50')
ax1.set_title('失分大類別分佈', fontsize=14, fontweight='bold', pad=20)

# ────────────────────────────────────────────────────────
# 圖2: 細項失分原因 - 水平長條圖
# ────────────────────────────────────────────────────────
ax2 = axes[1]

# 按次數由高到低排列 (低→高，水平圖由下而上)
lose_counts_sorted = lose_counts.sort_values('次數', ascending=True)

# 根據大類別設定顏色
def get_color(label):
    if '主動失誤' in label:
        return '#E74C3C'
    elif '被對手得分' in label:
        return '#3498DB'
    elif '被迫失誤' in label:
        return '#F39C12'
    else:
        return '#95A5A6'

bar_colors = [get_color(label) for label in lose_counts_sorted['失分原因']]

bars = ax2.barh(
    lose_counts_sorted['失分原因'],
    lose_counts_sorted['次數'],
    color=bar_colors,
    edgecolor='white',
    linewidth=0.8,
    height=0.65
)

# 在長條上標示數值和佔比
for bar, (_, row) in zip(bars, lose_counts_sorted.iterrows()):
    width = bar.get_width()
    ax2.text(
        width + 0.3, bar.get_y() + bar.get_height()/2,
        f'{int(width)}次 ({row["佔比(%)"]}%)',
        va='center', ha='left', fontsize=10, fontweight='bold', color='#2C3E50'
    )

ax2.set_xlabel('出現次數', fontsize=12)
ax2.set_title('細項失分原因排名\n(由高到低)', fontsize=14, fontweight='bold')
ax2.set_xlim(0, lose_counts_sorted['次數'].max() * 1.35)
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# 圖例 (大類別顏色說明)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E74C3C', label='主動失誤'),
    Patch(facecolor='#3498DB', label='被對手得分'),
    Patch(facecolor='#F39C12', label='被迫失誤'),
]
ax2.legend(handles=legend_elements, loc='lower right', fontsize=10, 
           framealpha=0.9, title='失分類別', title_fontsize=10)

# ────────────────────────────────────────────────────────
# 圖3: 各場次失分原因大類別堆疊長條圖
# ────────────────────────────────────────────────────────
ax3 = axes[2]

# 各場次的失分大類別統計
match_major = chou_lose_df.groupby(['match_id', 'major_category']).size().unstack(fill_value=0)

# 確保所有大類別都有欄位
all_major_cats = ['主動失誤', '被對手得分', '被迫失誤']
for cat in all_major_cats:
    if cat not in match_major.columns:
        match_major[cat] = 0
match_major = match_major[all_major_cats]

x = np.arange(len(match_major.index))
bar_width = 0.55

colors_stack = ['#E74C3C', '#3498DB', '#F39C12']
bottoms = np.zeros(len(match_major))

for cat, color in zip(all_major_cats, colors_stack):
    values = match_major[cat].values
    bars_stack = ax3.bar(x, values, bottom=bottoms, label=cat, 
                          color=color, edgecolor='white', linewidth=0.8,
                          width=bar_width)
    # 標示各段數值 (只有>0才標)
    for i, (val, bot) in enumerate(zip(values, bottoms)):
        if val > 0:
            ax3.text(x[i], bot + val/2, str(int(val)), 
                    ha='center', va='center', fontsize=10, 
                    fontweight='bold', color='white')
    bottoms += values

ax3.set_xticks(x)
ax3.set_xticklabels([f'場次\n{int(mid)}' for mid in match_major.index], fontsize=11)
ax3.set_ylabel('失分次數', fontsize=12)
ax3.set_title('各場次失分原因大類別分佈', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10, framealpha=0.9, title='失分類別', title_fontsize=10)
ax3.grid(axis='y', alpha=0.3, linestyle='--')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# 在每個堆疊長條頂端標示總數
totals = match_major.sum(axis=1).values
for i, total in enumerate(totals):
    ax3.text(x[i], total + 0.3, f'{int(total)}', 
             ha='center', va='bottom', fontsize=11, fontweight='bold', color='#2C3E50')

plt.tight_layout()

print("\n✅ 圖表已生成")
print("\n📝 分析說明:")
print("  - 主動失誤：周天成自己打出 → 出界/掛網/未過網/落點判斷失誤/犯規")
print("  - 被對手得分：對手打出致勝球 → 落地致勝/落點判斷失誤")
print("  - 被迫失誤：被對手逼迫下接球失誤")


16. 給我周天成最常見的失分原因

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_cols = ['match_id', 'set', 'rally']

rally_last = (
    df.sort_values(group_cols + ['ball_round'])
      .groupby(group_cols, as_index=False)
      .last()
)

# 周天成自己打最後一拍且失分，才是可歸因到周天成 lose_reason 的失分
ctc_lost_last_shot = rally_last[
    (rally_last['player'] == PLAYER) &
    (rally_last['getpoint_player'] == OPP) &
    (rally_last['lose_reason'].notna())
].copy()

lose_counts = (
    ctc_lost_last_shot['lose_reason']
    .value_counts()
    .reset_index()
)
lose_counts.columns = ['失分原因', '次數']

print(lose_counts)

17. 繪製周天成所有輸球原因為'掛網'時的球員站位熱區圖？

In [ ]:
# 1. 數據篩選：周天成且輸球原因為"掛網"
filtered_df = df[(df['player'] == 'CHOU Tien Chen') & (df['lose_reason'] == '掛網')]

# 2. 確認數據存在
if len(filtered_df) > 0:
    # 3. 繪製熱圖
    plt.figure(figsize=(10, 8))
    sns.kdeplot(data=filtered_df, x='player_location_x', y='player_location_y', cmap="Reds", fill=True, thresh=0, cbar=True)
    
    # 圖表標籤
    plt.title('周天成在輸球掛網時的站位熱區圖')
    plt.xlabel('站位X座標')
    plt.ylabel('站位Y座標')
    plt.xlim(-1, 1)  # 設定合理的x範圍
    plt.ylim(-0.5, 1)  # 設定合理的y範圍
    plt.axhline(0, color='black', linewidth=1, linestyle='--')  # 添加網線
    plt.axvline(0, color='black', linewidth=1, linestyle='--')  # 添加中線
    
    plt.tight_layout()
    fig = plt.gcf()
else:
    print("沒有符合條件的數據，無法繪製熱圖。")

# 輸出圖形物件
fig

18. 分析周天成在開局（自己0-4 分）、中局（自己5-17 分）和關鍵分（自己18 分以上）三個比分階段中，他自己移動距離的差異。

In [ ]:
import numpy as np
import pandas as pd

# 只保留周天成作為擊球者的資料
chou_df = df[df['player'] == 'CHOU Tien Chen'].copy()

# 計算移動距離
chou_df['move_distance'] = np.sqrt(
    chou_df['player_move_x'] ** 2 + chou_df['player_move_y'] ** 2
)

# 依照周天成自己的分數分階段
def assign_stage(score):
    if score < 5:
        return '開局 (0-4分)'
    elif score < 18:
        return '中局 (5-17分)'
    else:
        return '關鍵分 (18分以上)'

chou_df['stage'] = chou_df['CHOU Tien Chen_score'].apply(assign_stage)

stage_order = ['開局 (0-4分)', '中局 (5-17分)', '關鍵分 (18分以上)']
chou_df['stage'] = pd.Categorical(chou_df['stage'], categories=stage_order, ordered=True)

# 只輸出各階段平均移動距離
stage_mean = chou_df.groupby('stage', observed=True)['move_distance'].mean().reindex(stage_order)

print("各得分階段的平均移動距離：")
for stage, value in stage_mean.items():
    print(f"{stage}: {value:.4f}")

19. 當周天成打擊出界時，劃出站點的散佈圖?

In [ ]:
# 確認數據集是否有數據
if len(df) > 0:
    # 篩選周天成打擊且失分原因為 "出界" 的記錄
    out_of_bound_df = df[(df['player'] == 'CHOU Tien Chen') & (df['lose_reason'] == '出界')]
    
    # 繪製站位散佈圖
    if len(out_of_bound_df) > 0:
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.scatterplot(data=out_of_bound_df, x='player_location_x', y='player_location_y', alpha=0.6, s=50, color='red', ax=ax)
        ax.set_title('周天成出界時的站位分佈', fontsize=16)
        ax.set_xlabel('站位 X 坐標', fontsize=12)
        ax.set_ylabel('站位 Y 坐標', fontsize=12)
        ax.grid(True)

        plt.tight_layout()
        plt.show()
    else:
        print('沒有關於周天成的出界記錄。')

20. 周天成打擊出界失誤，該球的落點分布

In [ ]:
# 確保數據量大於0
if len(df) > 0:
    # 篩選周天成的出界失誤數據
    chou_out_df = df[(df['player'] == 'CHOU Tien Chen') & (df['lose_reason'] == '出界')]

    # 檢查篩選後的數據量
    if len(chou_out_df) > 0:
        print(f"周天成出界失誤的總數: {len(chou_out_df)}")

        # 繪製落點分布圖
        plt.figure(figsize=(8, 6))
        plt.scatter(chou_out_df['landing_x'], chou_out_df['landing_y'], c='red', alpha=0.6, label='落點')
        plt.xlabel('落點 X 座標')
        plt.ylabel('落點 Y 座標')
        plt.title('周天成出界失誤球的落點分布')
        plt.axhline(0, color='black', linewidth=0.8)
        plt.axvline(0, color='black', linewidth=0.8)
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend()
        plt.tight_layout()

        # 確保落點分布圖準備好
        fig = plt.gcf()
    else:
        print("沒有周天成的出界失誤數據。")
else:
    print("數據集為空。")

21. 在第一場次的比賽中，隨著局數的推進，周天成的「殺球」使用頻率和得分率是否出現了顯著的下滑

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 篩選第一場比賽中周天成的擊球資料
match_df = df[(df['match_id'] == 1) & (df['player'] == 'CHOU Tien Chen')].copy()

if len(match_df) == 0:
    print("找不到第一場比賽中周天成的資料。")
else:
    # 2. 定義主動得分與殺球相關指標
    match_df['is_active_win'] = match_df['player'] == match_df['getpoint_player']
    match_df['is_smash'] = match_df['type'] == '殺球'
    match_df['is_smash_win'] = match_df['is_smash'] & match_df['is_active_win']

    # 3. 依局數統計殺球使用頻率與得分率
    set_stats = match_df.groupby('set').agg(
        total_shots=('type', 'count'),
        smash_shots=('is_smash', 'sum'),
        smash_wins=('is_smash_win', 'sum')
    ).reset_index()

    set_stats['smash_frequency'] = set_stats['smash_shots'] / set_stats['total_shots']
    set_stats['smash_win_rate'] = set_stats['smash_wins'] / set_stats['smash_shots']

    print("第一場比賽各局的殺球使用頻率與得分率：")
    print(set_stats[['set', 'total_shots', 'smash_shots', 'smash_wins', 'smash_frequency', 'smash_win_rate']].round(4))

    # 4. 視覺化
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(set_stats['set'], set_stats['smash_frequency'], marker='o', linewidth=2)
    axes[0].set_title('周天成各局殺球使用頻率（第一場比賽）')
    axes[0].set_xlabel('局數')
    axes[0].set_ylabel('殺球使用頻率')
    axes[0].set_xticks(set_stats['set'])
    axes[0].grid(alpha=0.3)

    axes[1].plot(set_stats['set'], set_stats['smash_win_rate'], marker='o', linewidth=2, color='orange')
    axes[1].set_title('周天成各局殺球得分率（第一場比賽）')
    axes[1].set_xlabel('局數')
    axes[1].set_ylabel('殺球得分率')
    axes[1].set_xticks(set_stats['set'])
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

22. 分析周天成在不同比分階段(開局:0-4分,中局:5-17分,關鍵分:18分以上)中，使用不同球種的頻率變化趨勢

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 只保留周天成的擊球資料
chou_df = df[df['player'] == 'CHOU Tien Chen'].copy()

# 2. 依周天成比分分階段
def assign_stage(score):
    if score < 5:
        return '開局 (0-4分)'
    elif score < 18:
        return '中局 (5-17分)'
    else:
        return '關鍵分 (18分以上)'

stage_order = ['開局 (0-4分)', '中局 (5-17分)', '關鍵分 (18分以上)']
chou_df['score_stage'] = chou_df['CHOU Tien Chen_score'].apply(assign_stage)
chou_df['score_stage'] = pd.Categorical(chou_df['score_stage'], categories=stage_order, ordered=True)

# 3. 統計各比分階段中不同球種的使用次數
shot_counts = chou_df.groupby(['score_stage', 'type'], observed=True).size().reset_index(name='count')

# 4. 轉成各階段內的使用頻率
stage_totals = chou_df.groupby('score_stage', observed=True).size().reset_index(name='total_shots')
shot_counts = shot_counts.merge(stage_totals, on='score_stage', how='left')
shot_counts['usage_rate'] = shot_counts['count'] / shot_counts['total_shots']

# 5. 轉為方便閱讀的表格
usage_table = shot_counts.pivot(index='type', columns='score_stage', values='usage_rate').fillna(0)
usage_table = usage_table.reindex(columns=stage_order)

print("周天成在不同比分階段的各球種使用頻率：")
print(usage_table.round(4))

# 6. 視覺化趨勢
fig, ax = plt.subplots(figsize=(12, 7))

for shot_type in usage_table.index:
    ax.plot(
        usage_table.columns,
        usage_table.loc[shot_type],
        marker='o',
        linewidth=2,
        label=shot_type
    )

ax.set_title('周天成在不同比分階段的球種使用頻率變化')
ax.set_xlabel('比分階段')
ax.set_ylabel('使用頻率')
ax.legend(title='球種', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

22. 分析周天成在不同比分階段(開局:0-4分,中局:5-17分,關鍵分:18分以上)中，使用不同球種的頻率變化趨勢

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'

ctc_df = df[df['player'] == PLAYER].copy()

def score_phase(score):
    if score <= 4:
        return '開局(0-4)'
    elif score <= 17:
        return '中局(5-17)'
    else:
        return '關鍵分(18+)'

ctc_df['score_phase'] = ctc_df['CHOU Tien Chen_score'].apply(score_phase)

summary = (
    ctc_df.groupby(['score_phase', 'type'])
    .size()
    .reset_index(name='count')
)

summary['phase_total'] = summary.groupby('score_phase')['count'].transform('sum')
summary['frequency'] = summary['count'] / summary['phase_total']

print(summary.sort_values(['score_phase', 'count'], ascending=[True, False]))

23. 在關鍵分(18分以上)時，周天成採用的擊球策略與其他階段有何不同？

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
ATTACK_TYPES = ['殺球', '推撲球', '平球']
DEFENSE_TYPES = ['接殺防守', '挑球', '長球']
PLAYER_SCORE_COL = 'CHOU Tien Chen_score'

chou_df = df[df['player'] == PLAYER].copy()
chou_df['score_stage'] = chou_df[PLAYER_SCORE_COL].apply(lambda s: '關鍵分(18+)' if s >= 18 else '非關鍵分')
chou_df['is_direct_score'] = chou_df['getpoint_player'].eq(PLAYER)
chou_df['is_error'] = chou_df['lose_reason'].notna() & chou_df['getpoint_player'].eq(OPPONENT)
chou_df['style'] = chou_df['type'].apply(
    lambda t: '攻擊型' if t in ATTACK_TYPES else ('防守/控球型' if t in DEFENSE_TYPES else '其他')
)

shot_dist = (
    chou_df.groupby(['score_stage', 'type'])
    .size()
    .reset_index(name='次數')
)
shot_dist['比例'] = shot_dist['次數'] / shot_dist.groupby('score_stage')['次數'].transform('sum')
style_summary = chou_df.groupby('score_stage').agg(
    擊球數=('type', 'size'),
    攻擊型比例=('style', lambda s: (s == '攻擊型').mean()),
    防守控球型比例=('style', lambda s: (s == '防守/控球型').mean()),
    直接得分率=('is_direct_score', 'mean'),
    失誤率=('is_error', 'mean'),
).reset_index()

print('關鍵分 vs 非關鍵分：周天成球種分布')
print(shot_dist.sort_values(['score_stage', '次數'], ascending=[True, False]))
print('\n策略摘要：')
print(style_summary)

fig, ax = plt.subplots(figsize=(10, 5))
pivot = shot_dist.pivot(index='type', columns='score_stage', values='比例').fillna(0)
pivot.plot(kind='bar', ax=ax)
ax.set_title('周天成關鍵分 vs 非關鍵分球種比例')
ax.set_xlabel('球種')
ax.set_ylabel('比例')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

23. 在關鍵分(18分以上)時，周天成採用的擊球策略與其他階段有何不同？

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'

chou_df = df[df['player'] == PLAYER].copy()

# 較寬鬆定義：18分以上視為關鍵分
chou_df['stage'] = chou_df.apply(
    lambda row: '關鍵分階段'
    if (row['CHOU Tien Chen_score'] >= 18 or row['Kento MOMOTA_score'] >= 18)
    else '其他階段',
    axis=1
)

type_counts = chou_df.groupby(['stage', 'type']).size().unstack(fill_value=0)
type_ratios = type_counts.div(type_counts.sum(axis=1), axis=0)

print("周天成在關鍵分階段與其他階段的球種分布：")
print(type_counts)

print("\n周天成在關鍵分階段與其他階段的球種比例：")
print(type_ratios)

24. 周天成在長回合相較於短回合較常失誤的球種?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

rally_info = df.groupby(GROUP_COLS, as_index=False).agg(rally_length=('ball_round', 'max'))
rally_info['rally_length_group'] = pd.cut(
    rally_info['rally_length'],
    bins=[0, 4, 10, float('inf')],
    labels=['短回合(<=4)', '中回合(5-10)', '長回合(>=11)'],
    right=True
)

chou_df = pd.merge(df[df['player'] == PLAYER].copy(), rally_info, on=GROUP_COLS, how='left')
chou_df['is_error'] = chou_df['lose_reason'].notna() & chou_df['getpoint_player'].eq(OPPONENT)
target = chou_df[chou_df['rally_length_group'].isin(['短回合(<=4)', '長回合(>=11)'])].copy()

summary = (
    target.groupby(['type', 'rally_length_group'], observed=True)
    .agg(擊球數=('type', 'size'), 失誤數=('is_error', 'sum'))
    .reset_index()
)
summary['失誤率'] = summary['失誤數'] / summary['擊球數']

pivot_rate = summary.pivot(index='type', columns='rally_length_group', values='失誤率').fillna(0)
pivot_count = summary.pivot(index='type', columns='rally_length_group', values='擊球數').fillna(0)
for col in ['短回合(<=4)', '長回合(>=11)']:
    if col not in pivot_rate.columns:
        pivot_rate[col] = 0
    if col not in pivot_count.columns:
        pivot_count[col] = 0
comparison = pd.DataFrame({
    '短回合失誤率': pivot_rate['短回合(<=4)'],
    '長回合失誤率': pivot_rate['長回合(>=11)'],
    '長回合-短回合': pivot_rate['長回合(>=11)'] - pivot_rate['短回合(<=4)'],
    '短回合擊球數': pivot_count['短回合(<=4)'],
    '長回合擊球數': pivot_count['長回合(>=11)'],
}).sort_values('長回合-短回合', ascending=False)

print('周天成長回合相較短回合，各球種失誤率差異：')
print(comparison)
print('\n長回合失誤率上升最多的球種：')
print(comparison[comparison['長回合-短回合'] > 0].head(5))

fig, ax = plt.subplots(figsize=(10, 5))
comparison['長回合-短回合'].sort_values().plot(kind='barh', ax=ax, color='#72B7B2')
ax.axvline(0, color='black', linewidth=1)
ax.set_title('長回合 vs 短回合：周天成各球種失誤率差異')
ax.set_xlabel('長回合失誤率 - 短回合失誤率')
ax.set_ylabel('球種')
plt.tight_layout()

25. 當對手靠球場兩側時，周天成較常讓球落點到哪些區域？繪製熱區圖。

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 定義球場兩側區域
left_side_areas = [1, 5, 9, 13, 17, 21]
right_side_areas = [4, 8, 12, 16, 20, 24]
side_areas = left_side_areas + right_side_areas

# 2. 篩選周天成擊球，且對手站在球場兩側的資料
target_df = df[
    (df['player'] == 'CHOU Tien Chen') &
    (df['opponent_location_area'].isin(side_areas))
].copy()

print(f"符合條件的擊球筆數: {len(target_df)}")

# 3. 若資料存在，畫周天成的落點熱區圖
if len(target_df) > 0:
    plt.figure(figsize=(8, 10))

    sns.kdeplot(
        data=target_df,
        x='landing_x',
        y='landing_y',
        fill=True,
        cmap='YlOrRd',
        levels=20,
        thresh=0.05
    )

    plt.title('當對手靠球場兩側時，周天成的落點熱區圖')
    plt.xlabel('landing_x')
    plt.ylabel('landing_y')
    plt.tight_layout()
    plt.show()

    # 4. 額外輸出落點區域分布，方便後續洞察
    landing_area_counts = target_df['landing_area'].value_counts().reset_index()
    landing_area_counts.columns = ['landing_area', 'count']

    print("\n最常見的落點區域：")
    print(landing_area_counts.head(10))
else:
    print("沒有符合條件的資料。")

26. 分析周天成在不同局（第 1 局、第 2 局、第 3 局）殺球比重變化。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'

# 1. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 2. 統計各局總擊球數與殺球數
set_summary = chou_df.groupby('set').agg(
    總擊球數=('type', 'count'),
    殺球次數=('type', lambda x: (x == '殺球').sum())
).reset_index()

# 3. 計算殺球比重
set_summary['殺球比重'] = set_summary['殺球次數'] / set_summary['總擊球數']

print("周天成在不同局的殺球比重變化：")
print(set_summary)

# 4. 視覺化
plt.figure(figsize=(8, 5))
bars = plt.bar(set_summary['set'].astype(str), set_summary['殺球比重'], color=['#4F81BD', '#F39C34', '#D9534F'])

for bar, value in zip(bars, set_summary['殺球比重']):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{value:.2%}',
        ha='center',
        va='bottom'
    )

plt.title('周天成在不同局的殺球比重')
plt.xlabel('局數')
plt.ylabel('殺球比重')
plt.tight_layout()
plt.show()

27. 分析周天成在「邊線附近擊球」的失誤率是否偏高。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'

# 1. 定義邊線附近區域
# 依 court_place 的 4 欄結構，最左欄 A 與最右欄 D 視為邊線附近
sideline_areas = [1, 4, 5, 8, 9, 12, 13, 16, 17, 20, 21, 24]

# 2. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 3. 定義是否在邊線附近擊球
chou_df['near_sideline'] = chou_df['hit_area'].isin(sideline_areas)

# 4. 定義失誤
# 周天成這一拍有 lose_reason，且最後由對手得分
chou_df['is_error'] = (
    chou_df['lose_reason'].notna() &
    (chou_df['getpoint_player'] != PLAYER)
)

# 5. 統計邊線附近 vs 非邊線附近的失誤率
summary = chou_df.groupby('near_sideline').agg(
    總擊球數=('type', 'count'),
    失誤次數=('is_error', 'sum')
).reset_index()

summary['失誤率'] = summary['失誤次數'] / summary['總擊球數']
summary['擊球位置'] = summary['near_sideline'].map({
    True: '邊線附近',
    False: '非邊線附近'
})

print("周天成在邊線附近擊球的失誤率分析：")
print(summary[['擊球位置', '總擊球數', '失誤次數', '失誤率']])

# 6. 額外列出邊線附近擊球時的失誤原因分布
sideline_errors = chou_df[
    chou_df['near_sideline'] &
    chou_df['is_error']
].copy()

lose_reason_counts = sideline_errors['lose_reason'].value_counts().reset_index()
lose_reason_counts.columns = ['失誤原因', '次數']

print("\n邊線附近擊球時的失誤原因分布：")
print(lose_reason_counts)

28. 統計周天成在各局的平均每回合拍數，並比較第1局、第2局、第3局差異。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 計算每個回合的總拍數
rally_shot_counts = (
    df.groupby(GROUP_COLS)['ball_round']
    .max()
    .reset_index(name='shot_count')
)

# 2. 統計各局平均每回合總拍數
set_summary = rally_shot_counts.groupby('set').agg(
    平均拍數=('shot_count', 'mean'),
    中位數拍數=('shot_count', 'median'),
    回合數=('shot_count', 'count')
).reset_index()

print("各局平均每回合總拍數：")
print(set_summary)

# 3. 視覺化
plt.figure(figsize=(8, 5))
plt.bar(
    set_summary['set'].astype(str),
    set_summary['平均拍數'],
    color=['#4E79A7', '#F28E2B', '#E15759']
)

plt.title('各局平均每回合總拍數')
plt.xlabel('局數')
plt.ylabel('平均拍數')
plt.tight_layout()
plt.show()

28. 統計周天成在各局的平均每回合拍數，並比較第1局、第2局、第3局差異。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 2. 計算周天成在每個回合的個人出手數
rally_shot_counts = (
    chou_df.groupby(GROUP_COLS)
    .size()
    .reset_index(name='shot_count')
)

# 3. 統計各局平均每回合個人出手數
set_summary = rally_shot_counts.groupby('set').agg(
    平均出手數=('shot_count', 'mean'),
    中位數出手數=('shot_count', 'median'),
    回合數=('shot_count', 'count')
).reset_index()

print("周天成在各局平均每回合個人出手數：")
print(set_summary)

# 4. 視覺化
plt.figure(figsize=(8, 5))
plt.bar(
    set_summary['set'].astype(str),
    set_summary['平均出手數'],
    color=['#4E79A7', '#F28E2B', '#E15759']
)

plt.title('周天成在各局平均每回合個人出手數')
plt.xlabel('局數')
plt.ylabel('平均出手數')
plt.tight_layout()
plt.show()

29. 比較周天成在前場、中場、後場擊球時的掛網失誤率。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'

# 依 court_place.txt 的定義
front_areas = list(range(17, 25))  # 17~24
mid_areas = list(range(5, 17))     # 5~16
back_areas = list(range(1, 5))     # 1~4

def classify_court_zone(area):
    if pd.isna(area):
        return None
    area = int(area)
    if area in front_areas:
        return '前場'
    if area in mid_areas:
        return '中場'
    if area in back_areas:
        return '後場'
    return None

# 1. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 2. 依 player_location_area 分成前中後場
chou_df['court_zone'] = chou_df['player_location_area'].apply(classify_court_zone)
chou_df = chou_df[chou_df['court_zone'].notna()].copy()

# 3. 定義掛網失誤
chou_df['is_net_error'] = chou_df['lose_reason'] == '掛網'

# 4. 統計各場區掛網失誤率
summary = chou_df.groupby('court_zone').agg(
    總擊球數=('court_zone', 'count'),
    掛網失誤次數=('is_net_error', 'sum')
).reset_index()

summary['掛網失誤率'] = summary['掛網失誤次數'] / summary['總擊球數']

print("周天成在前中後場站位擊球時的掛網失誤率：")
print(summary)

# 5. 視覺化
zone_order = ['前場', '中場', '後場']
summary['court_zone'] = pd.Categorical(summary['court_zone'], categories=zone_order, ordered=True)
summary = summary.sort_values('court_zone')

plt.figure(figsize=(8, 5))
plt.bar(
    summary['court_zone'],
    summary['掛網失誤率'],
    color=['#59A14F', '#EDC948', '#E15759']
)

plt.title('周天成在不同站位區域擊球時的掛網失誤率')
plt.xlabel('站位區域')
plt.ylabel('掛網失誤率')
plt.tight_layout()
plt.show()

30. 周天成在比賽後段（第 2、3 局）移動距離與失誤率的關聯分析。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'

chou_df = df[(df['player'] == PLAYER) & (df['set'].isin([2, 3]))].copy()
chou_df['move_distance'] = np.sqrt(chou_df['player_move_x'] ** 2 + chou_df['player_move_y'] ** 2)
chou_df['is_error'] = chou_df['lose_reason'].notna() & chou_df['getpoint_player'].eq(OPPONENT)

set_summary = chou_df.groupby(['match_id', 'set']).agg(
    擊球數=('type', 'size'),
    平均移動距離=('move_distance', 'mean'),
    失誤數=('is_error', 'sum'),
    失誤率=('is_error', 'mean')
).reset_index()

corr = set_summary[['平均移動距離', '失誤率']].corr().iloc[0, 1] if len(set_summary) >= 2 else np.nan

print('周天成在第2、3局的移動距離與失誤率：')
print(set_summary)
print(f'平均移動距離與失誤率的相關係數: {corr:.3f}' if pd.notna(corr) else '資料不足，無法計算相關係數。')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(set_summary['平均移動距離'], set_summary['失誤率'], color='#E45756')
for _, row in set_summary.iterrows():
    ax.annotate(f"M{int(row['match_id'])}-S{int(row['set'])}", (row['平均移動距離'], row['失誤率']))
ax.set_title('第2、3局：移動距離與失誤率關聯')
ax.set_xlabel('平均移動距離')
ax.set_ylabel('失誤率')
plt.tight_layout()

31. 分析周天成使用殺球，對手都用什麼球種反擊，繪製圓餅圖

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()

# 對手回擊球種必須取同一 rally 的下一拍 type；不可使用 opponent_type。
ordered_df['next_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(-1)
ordered_df['next_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(-1)

opponent_replies = ordered_df[
    ordered_df['player'].eq(PLAYER) &
    ordered_df['type'].eq('殺球') &
    ordered_df['next_player'].ne(PLAYER) &
    ordered_df['next_type'].notna()
].copy()

reply_counts = opponent_replies['next_type'].value_counts()

print(f'周天成殺球後有對手下一拍回擊的次數: {len(opponent_replies)}')
print('對手回擊球種次數:')
print(reply_counts)

if reply_counts.empty:
    print('沒有周天成殺球後對手回擊的資料。')
else:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(
        reply_counts.values,
        labels=reply_counts.index,
        autopct='%1.1f%%',
        startangle=90,
        counterclock=False
    )
    ax.set_title('周天成殺球後對手回擊球種分布')
    ax.axis('equal')
    plt.tight_layout()

31. 分析周天成使用殺球，對手都用什麼球種反擊，繪製圓餅圖

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 用 player_type 作為等價欄位
type_map = {
    1: '發短球',
    2: '發長球',
    3: '長球',
    4: '殺球',
    5: '切球',
    6: '挑球',
    7: '平球',
    8: '網前球',
    9: '推撲球',
    10: '接殺防守',
    11: '接不到'
}

df_next = df.copy()
df_next['next_player_type'] = df_next.groupby(GROUP_COLS)['player_type'].shift(-1)

chou_smash_df = df_next[
    (df_next['player'] == PLAYER) &
    (df_next['type'] == '殺球')
].copy()

chou_smash_df['next_type_name'] = chou_smash_df['next_player_type'].map(type_map)

response_counts = chou_smash_df['next_type_name'].value_counts().reset_index()
response_counts.columns = ['對手反擊球種', '次數']
response_counts['比例'] = response_counts['次數'] / response_counts['次數'].sum()

print("周天成使用殺球後，對手反擊球種分布：")
print(response_counts)

32. 在10拍以上的回合中，周天成的得分率是多少？

In [ ]:
# ==========================================
# 分析：10拍以上回合中，周天成的得分率
# ==========================================

# 驗證資料量
print(f"總資料筆數: {len(df)}")
assert len(df) > 0, "DataFrame 為空！"

# Step 1: 計算每個回合的總擊球數（該回合最大 ball_round）
# 回合唯一識別：match_id + set + rally
rally_stats = df.groupby(['match_id', 'set', 'rally']).agg(
    total_hits=('ball_round', 'max'),          # 回合總擊球數
    getpoint_player=('getpoint_player', 'last') # 得分球員（最後一球紀錄）
).reset_index()

print(f"\n總回合數: {len(rally_stats)}")
print(f"\n各回合總擊球數分布（前幾名）:")
print(rally_stats['total_hits'].value_counts().sort_index().head(20))

# Step 2: 確認 getpoint_player 的填充狀況（每回合應只有一個得分）
print(f"\ngetpoint_player 空值回合數: {rally_stats['getpoint_player'].isna().sum()}")
print(f"getpoint_player 非空回合數: {rally_stats['getpoint_player'].notna().sum()}")

# Step 3: 篩選回合長度 >= 10 的回合
long_rallies = rally_stats[rally_stats['total_hits'] >= 10].copy()
print(f"\n10拍以上的回合數: {len(long_rallies)}")
print(f"10拍以上回合的 getpoint_player 分布:")
print(long_rallies['getpoint_player'].value_counts(dropna=False))

# Step 4: 計算周天成的得分次數
chou_wins = (long_rallies['getpoint_player'] == 'CHOU Tien Chen').sum()
total_long_rallies = len(long_rallies)
chou_win_rate = chou_wins / total_long_rallies * 100

print(f"\n========================================")
print(f"10拍以上回合總數: {total_long_rallies}")
print(f"周天成得分次數: {chou_wins}")
print(f"周天成得分率: {chou_wins}/{total_long_rallies} = {chou_win_rate:.2f}%")
print(f"========================================")

# 補充：對手（桃田賢斗）的得分次數
momota_wins = (long_rallies['getpoint_player'] == 'Kento MOMOTA').sum()
momota_win_rate = momota_wins / total_long_rallies * 100
print(f"桃田賢斗得分次數: {momota_wins}")
print(f"桃田賢斗得分率: {momota_wins}/{total_long_rallies} = {momota_win_rate:.2f}%")
print(f"未得分（無得分記錄）回合數: {long_rallies['getpoint_player'].isna().sum()}")

# ==========================================
# 視覺化
# ==========================================
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('10拍以上長回合分析', fontsize=16, fontweight='bold', y=1.01)

# --- 圖1: 10拍以上回合中雙方得分率圓餅圖 ---
ax1 = axes[0]

labels = ['周天成\n(CHOU Tien Chen)', '桃田賢斗\n(Kento MOMOTA)']
values = [chou_wins, momota_wins]
colors = ['#2196F3', '#FF5722']

# 若有無得分回合，加入"無記錄"
no_point = long_rallies['getpoint_player'].isna().sum()
if no_point > 0:
    labels.append('無得分記錄')
    values.append(no_point)
    colors.append('#9E9E9E')

wedges, texts, autotexts = ax1.pie(
    values,
    labels=labels,
    autopct=lambda pct: f'{pct:.1f}%\n({int(round(pct/100*total_long_rallies))}回合)',
    colors=colors,
    startangle=90,
    pctdistance=0.75,
    textprops={'fontsize': 10}
)
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight('bold')

ax1.set_title(f'10拍以上回合得分分布\n（共 {total_long_rallies} 個回合）', fontsize=12, fontweight='bold')

# --- 圖2: 回合長度分布（所有回合 vs 10拍以上標記）---
ax2 = axes[1]

all_hit_counts = rally_stats['total_hits'].value_counts().sort_index()
colors_bar = ['#FF5722' if x >= 10 else '#90CAF9' for x in all_hit_counts.index]

bars = ax2.bar(all_hit_counts.index, all_hit_counts.values, color=colors_bar, edgecolor='white', linewidth=0.5)

# 標記10拍分界線
ax2.axvline(x=9.5, color='red', linestyle='--', linewidth=1.5, label='10拍門檻')

ax2.set_xlabel('回合擊球數（拍）', fontsize=11)
ax2.set_ylabel('回合數', fontsize=11)
ax2.set_title('所有回合的擊球數分布\n（紅色 = 10拍以上）', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)

# 加入文字標籤（值較大時才顯示）
for bar in bars:
    height = bar.get_height()
    if height >= 3:
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.3,
                 f'{int(height)}', ha='center', va='bottom', fontsize=8)

ax2.set_xticks(all_hit_counts.index)
ax2.tick_params(axis='x', rotation=45)

# 加入統計摘要文字
ax2.text(0.97, 0.97,
         f'10拍以上回合：{total_long_rallies} 個\n佔總回合 {total_long_rallies/len(rally_stats)*100:.1f}%',
         transform=ax2.transAxes,
         fontsize=9, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
print(f"\n圖表已生成完畢")


33. 給我所有局的最終得分數

In [ ]:
# ============================================================
# 列出所有場次中每一局的最終得分
# ============================================================

# 驗證數據量
print(f"總資料筆數: {len(df)}")
print(f"包含的場次 (match_id): {sorted(df['match_id'].unique())}")
print(f"包含的局數 (set): {sorted(df['set'].unique())}")
print()

# df 已按照 (match_id, set, rally, ball_round) 排序
# 對每個 (match_id, set) 取最後一筆資料，即為該局最後一球
final_scores = (
    df.groupby(['match_id', 'set'], sort=True)
    .last()
    .reset_index()
)[['match_id', 'set', 'Kento MOMOTA_score', 'CHOU Tien Chen_score', 'rally']]

# 重命名欄位方便閱讀
final_scores = final_scores.rename(columns={
    'match_id': 'match_id',
    'set': 'set',
    'Kento MOMOTA_score': 'momota_score',
    'CHOU Tien Chen_score': 'chou_score',
    'rally': 'total_rallies'
})

# 轉換為整數方便閱讀
final_scores['momota_score'] = final_scores['momota_score'].astype(int)
final_scores['chou_score'] = final_scores['chou_score'].astype(int)
final_scores['set'] = final_scores['set'].astype(int)
final_scores['match_id'] = final_scores['match_id'].astype(int)
final_scores['total_rallies'] = final_scores['total_rallies'].astype(int)

# 判斷局勝者
def determine_set_winner(row):
    m = row['momota_score']
    c = row['chou_score']
    if m > c:
        return 'Kento MOMOTA'
    elif c > m:
        return 'CHOU Tien Chen'
    else:
        return '平手'

final_scores['winner'] = final_scores.apply(determine_set_winner, axis=1)

# ============================================================
# 印出詳細結果
# ============================================================
print("=" * 70)
print("各場次、各局最終得分結果")
print("=" * 70)

for _, row in final_scores.iterrows():
    print(
        f"場次 {row['match_id']} | 第 {row['set']} 局 | "
        f"Kento MOMOTA {row['momota_score']} - {row['chou_score']} CHOU Tien Chen | "
        f"局勝: {row['winner']} | 共 {row['total_rallies']} 回合"
    )

34. 分析周天成前一拍打什麼球種最容易造成對手未過網？

In [ ]:
# 確保df非空
if len(df) > 0:
    # 1. 找出周天成作為擊球球員的數據
   #chou_df = df[df['player'] == 'CHOU Tien Chen']

    # 2. 使用 shift 方法獲取下一拍的數據，並且下一拍的 'lose_reason' 是 '未過網'
    df['next_lose_reason'] = df['lose_reason'].shift(-1)
    df['next_player'] = df['player'].shift(-1)
    chou_df = df[df['player'] == 'CHOU Tien Chen']
    # 3. 篩選對手的失誤原因是「未過網」
    unforced_errors_df = chou_df[(chou_df['next_lose_reason'] == '未過網') & (chou_df['next_player'] != 'CHOU Tien Chen')]

    # 4. 統計前一拍周天成使用的球種
    shot_count = unforced_errors_df['type'].value_counts()

    # 打印數據
    print("周天成前一拍使用的球種，造成對手未過網的次數：")
    print(shot_count)

    # 5. 繪製圖表
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(shot_count.index, shot_count.values, color='b')
    ax.set_xlabel('球種')
    ax.set_ylabel('未過網次數')
    ax.set_title('周天成前一拍不同球種造成對手未過網的次數')
    plt.xticks(rotation=45)
    plt.tight_layout()

    # 打印fig物件
    print(fig)
else:
    print("數據集為空，無法進行分析。")

34. 分析周天成前一拍打什麼球種最容易造成對手未過網？

In [ ]:
# 確保數據集有資料
if len(df) > 0:
    # 過濾出對手因"未過網"而失分的數據
    relevant_df = df[df['lose_reason'] == '未過網']
    
    # 確保該數據集中有資料
    if len(relevant_df) > 0:
        # 使用 shift(1) 找到周天成在前一拍所打出的球種
        relevant_df['previous_player_type'] = df['type'].shift(1)
        
        # 過濾出前一拍的擊球者為周天成的情況
        previous_stroke_df = relevant_df[df['player'].shift(1) == 'CHOU Tien Chen']
        
        # 統計前一拍周天成打出各球種的次數
        shot_type_counts = previous_stroke_df['previous_player_type'].value_counts()
        
        # 找出次數最多的球種
        most_common_shot_type = shot_type_counts.idxmax()
        most_common_shot_count = shot_type_counts.max()
        
        # 打印結果
        print(f"周天成在造成對手未過網的前一拍，最常使用的球種是：{most_common_shot_type}，共出現 {most_common_shot_count} 次。")
    else:
        print("數據集中沒有對手因'未過網'而失分的記錄。")
else:
    print("數據集中沒有可用資料。")


35. 當對手被周天成調動到「前場」時，對手最常使用什麼球種？

In [ ]:
# =============================================
# 分析：當對手被周天成調動到前場時，對手的回擊球種統計
# =============================================

# 前場 Zone 定義（參見 Court Grid Definitions）
FRONT_COURT_ZONES = [17, 18, 19, 20, 21, 22, 23, 24]

# Step 1: 確認資料量
print(f"總資料行數: {len(df)}")
print(f"球員列表: {df['player'].unique()}")
print()

# Step 2: 找出周天成打球的所有 rows
chou_mask = df['player'] == 'CHOU Tien Chen'
print(f"周天成打球總次數: {chou_mask.sum()}")

# Step 3: 在周天成打球時，對手站位在前場的情況
# opponent_location_area 是當前 player 的對手站位
# 此時 player='CHOU', opponent='MOMOTA'，所以 opponent_location_area = MOMOTA 的站位
chou_opponent_front_mask = (
    chou_mask &
    (df['opponent_location_area'].isin(FRONT_COURT_ZONES))
)
print(f"周天成打球且對手站位在前場的次數: {chou_opponent_front_mask.sum()}")
print()

# Step 4: 使用 shift(-1) 取「下一球」（即對手的回擊）
# 在 rally 層級內 shift，避免跨 rally 錯誤
df_sorted = df.sort_values(['match_id', 'set', 'rally', 'ball_round'])

# 取下一球的 type 與 player（用於驗證）
next_type = df_sorted.groupby(['match_id', 'set', 'rally'])['type'].shift(-1)
next_player = df_sorted.groupby(['match_id', 'set', 'rally'])['player'].shift(-1)

# Step 5: 建立分析用 DataFrame
analysis_df = df_sorted[chou_opponent_front_mask].copy()
analysis_df['next_type'] = next_type[chou_opponent_front_mask]
analysis_df['next_player'] = next_player[chou_opponent_front_mask]

print(f"加入 next_type 後的資料量: {len(analysis_df)}")

# Step 6: 驗證 - 下一球的 player 應為 MOMOTA（對手）
print("\n下一球的 player 分布（應為 MOMOTA）：")
print(analysis_df['next_player'].value_counts(dropna=False))

# Step 7: 移除 next_type 為 NaN 的（表示是該 rally 的最後一球，無下一球）
analysis_df_valid = analysis_df.dropna(subset=['next_type'])
print(f"\n有效回擊資料量（排除 rally 最後一球）: {len(analysis_df_valid)}")

# Step 8: 統計對手回擊球種
shot_counts = analysis_df_valid['next_type'].value_counts().reset_index()
shot_counts.columns = ['球種', '次數']
shot_counts['佔比(%)'] = (shot_counts['次數'] / shot_counts['次數'].sum() * 100).round(1)

print("\n=== 對手（Kento MOMOTA）被調動至前場後的回擊球種統計（由高至低）===")
print(shot_counts.to_string(index=False))

# =============================================
# 視覺化：只保留圓餅圖
# =============================================

# 合併小比例（< 3%）為「其他」
threshold = 3.0
major = shot_counts[shot_counts['佔比(%)'] >= threshold].copy()
minor = shot_counts[shot_counts['佔比(%)'] < threshold]

if len(minor) > 0:
    other_row = pd.DataFrame({
        '球種': ['其他'],
        '次數': [minor['次數'].sum()],
        '佔比(%)': [minor['佔比(%)'].sum().round(1)]
    })
    pie_data = pd.concat([major, other_row], ignore_index=True)
else:
    pie_data = major

fig, ax = plt.subplots(figsize=(8, 6))

wedges, texts, autotexts = ax.pie(
    pie_data['次數'],
    labels=pie_data['球種'],
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)

for text in texts:
    text.set_fontsize(11)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')

ax.set_title('對手被調動至前場後的回擊球種佔比', fontsize=13, fontweight='bold')
plt.tight_layout()

36. 幫我比較周天成（CHOU Tien Chen）他在短回合（<=4拍）、中回合（5-10拍）、長回合（>=11拍）中的勝率

In [ ]:
# 驗證數據是否存在
if len(df) > 0:
    # 分類回合長度
    def categorize_rally_length(ball_round):
        if ball_round <= 4:
            return '短回合'
        elif 5 <= ball_round <= 10:
            return '中回合'
        else:
            return '長回合'
    
    # 增加一個新的分類欄位
    df['rally_length_category'] = df['ball_round'].apply(categorize_rally_length)
    
    # 確保以每個rally的最後一拍作為分組結果
    rally_end_df = df.groupby(['match_id', 'set', 'rally']).last().reset_index()
    
    # 篩選出周天成勝出的回合
    chou_win_df = rally_end_df[rally_end_df['getpoint_player'] == 'CHOU Tien Chen']

    # 計算每個回合長度類別的總回合數和周天成贏的回合數
    total_rallies = rally_end_df.groupby('rally_length_category').size()
    chou_wins = chou_win_df.groupby('rally_length_category').size()
    
    # 計算周天成在每個回合長度類別的勝率
    chou_win_rate = (chou_wins / total_rallies).fillna(0)  # 填充 NaN 為 0

    # 打印計算出來的勝率
    print("總回合數:")
    print(total_rallies)
    print("\n周天成贏得的回合數:")
    print(chou_wins)
    print("\n周天成的勝率:")
    print(chou_win_rate)
    
    # 可視化
    fig, ax = plt.subplots(figsize=(8, 6))
    chou_win_rate.plot(kind='bar', color=['lightblue', 'lightgreen', 'salmon'], ax=ax)

    ax.set_title('周天成在不同回合長度中的勝率')
    ax.set_xlabel('回合長度分類')
    ax.set_ylabel('勝率')
    plt.xticks(rotation=0)
    plt.tight_layout()
    
else:
    print("數據集中沒有數據。")

37. 分析周天成使用殺球造成失誤，請繪製失誤原因的圓餅圖

In [ ]:
# 確保 df 存在且大於零
if len(df) > 0:
    # 篩選出周天成的殺球失誤資料
    chou_smash_errors = df[(df['player'] == 'CHOU Tien Chen') & 
                           (df['type'] == '殺球') &
                           (df['lose_reason'].notna())]

    # 統計不同失誤原因的次數
    error_reasons_count = chou_smash_errors['lose_reason'].value_counts()

    # 繪製圓餅圖
    fig, ax = plt.subplots()
    ax.pie(error_reasons_count, labels=error_reasons_count.index, autopct='%1.1f%%', startangle=90, counterclock=False)
    ax.set_title('周天成在比賽中使用殺球的失誤原因分布', fontsize=14)
    
    plt.tight_layout()  # 確保不重疊
    plt.show()

    # 打印詳細結算信息
    print("周天成使用殺球造成失誤的原因分布：")
    print(error_reasons_count)
else:
    print("DataFrame is empty. Ensure data is loaded correctly.")

37. 分析周天成使用殺球造成失誤，請繪製失誤原因的圓餅圖

In [ ]:
# Step 1: Filter relevant data
filtered_df = df[(df['player'] == 'CHOU Tien Chen') & 
                 (df['type'] == '殺球')]

# Step 2: Ensure it's a losing point for CHOU Tien Chen
error_df = filtered_df[filtered_df['getpoint_player'] == 'Kento MOMOTA']

# Step 3: Extract the losing reason
error_reasons = error_df['lose_reason'].dropna()

# Step 4: Count frequencies of each losing reason
reason_counts = error_reasons.value_counts()

# Step 5: Plotting
fig, ax = plt.subplots()
reason_counts.plot.pie(ax=ax, autopct='%1.1f%%', startangle=90, labels=reason_counts.index, counterclock=False)
ax.set_ylabel('')  # Remove the default ylabel
ax.set_title('周天成殺球失誤原因分佈')
plt.tight_layout()

# Print the counts to get a sense of the data
print("周天成殺球失誤原因及次數：")
print(reason_counts)

38. 在周天成的所有失分中，有多少百分比是因為對手的殺球導致的？

In [ ]:
# ============================================================
# 分析周天成的失分球：對手殺球導致失分
# ============================================================
import numpy as np

# 驗證數據量
print(f"總資料筆數: {len(df)}")
assert len(df) > 0, "DataFrame 為空！"

target_player = 'CHOU Tien Chen'
opponent_player = 'Kento MOMOTA'

# ────────────────────────────────────────────────────────────
# 1. 計算周天成的「總失分」
#    定義：getpoint_player 為對手（桃田賢斗）的球
# ────────────────────────────────────────────────────────────
chou_lose_df = df[df['getpoint_player'] == opponent_player].copy()
total_lose = len(chou_lose_df)
print(f"\n周天成 總失分數: {total_lose} 分")

# ────────────────────────────────────────────────────────────
# 2. 計算「對手殺球直接導致周天成失分」
#    定義：
#      - 該球類型為殺球 (type == '殺球')
#      - 打球者為對手 (player == 'Kento MOMOTA')  <- 排除周天成自己打殺球後失誤
#      - 得分者為對手 (getpoint_player == 'Kento MOMOTA')
# ────────────────────────────────────────────────────────────
smash_lose_df = df[
    (df['getpoint_player'] == opponent_player) &
    (df['type'] == '殺球') &
    (df['player'] == opponent_player)  # 確認是對手打的殺球
].copy()
smash_lose_count = len(smash_lose_df)
print(f"對手殺球直接導致周天成失分數: {smash_lose_count} 次")

# ────────────────────────────────────────────────────────────
# 3. 分類：對手主動得分 vs 周天成自身失誤
# ────────────────────────────────────────────────────────────
active_win_df = chou_lose_df[chou_lose_df['player'] == opponent_player]   # 對手主動得分
chou_error_df = chou_lose_df[chou_lose_df['player'] == target_player]     # 周天成失誤

print(f"\n  對手主動得分（直接打死）: {len(active_win_df)} 次")
print(f"  周天成自身失誤（出界/掛網等）: {len(chou_error_df)} 次")
print(f"  合計: {len(active_win_df) + len(chou_error_df)} 次（應等於 {total_lose}）")

# 各球種分布（對手主動得分）
active_by_type = active_win_df['type'].value_counts()
print(f"\n對手主動得分 - 各球種分布:\n{active_by_type}")

# 周天成失誤球種
error_by_type = chou_error_df['type'].value_counts()
print(f"\n周天成自身失誤 - 各球種分布:\n{error_by_type}")

# 失分原因
lose_reason_dist = chou_lose_df['lose_reason'].value_counts(dropna=False)
print(f"\n失分原因分布:\n{lose_reason_dist}")

# ────────────────────────────────────────────────────────────
# 4. 計算殺球失分佔比
# ────────────────────────────────────────────────────────────
smash_pct = (smash_lose_count / total_lose * 100) if total_lose > 0 else 0
print(f"\n{'='*50}")
print(f"殺球導致周天成失分: {smash_lose_count} 次 / 總失分 {total_lose} 次")
print(f"佔比: {smash_pct:.1f}%")
print(f"{'='*50}")

# ────────────────────────────────────────────────────────────
# 5. 視覺化
# ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("周天成失分分析", fontsize=16, fontweight='bold')

# --- 圖1: 對手主動得分球種分布（長條圖）---
ax1 = axes[0]

type_counts = active_win_df['type'].value_counts().copy()

# 合併小比例為「其他」(少於總失分3%的類別)
threshold = total_lose * 0.03
other_mask = type_counts < threshold
if other_mask.sum() > 1:
    other_count = int(type_counts[other_mask].sum())
    type_counts = type_counts[~other_mask]
    type_counts['其他'] = other_count

colors_bar = plt.cm.Set2(np.linspace(0, 0.8, len(type_counts)))
bars = ax1.bar(type_counts.index, type_counts.values, color=colors_bar,
               edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, type_counts.values):
    pct = val / total_lose * 100
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{val}次\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

ax1.set_title("對手主動得分各球種分布\n（佔周天成總失分比例）", fontsize=12, fontweight='bold')
ax1.set_xlabel("球種", fontsize=11)
ax1.set_ylabel("次數", fontsize=11)
ax1.set_ylim(0, type_counts.max() * 1.3)
ax1.tick_params(axis='x', labelsize=9)

# --- 圖2: 殺球失分 vs 其他失分 圓餅圖 ---
ax2 = axes[1]

smash_n = smash_lose_count
other_active_n = len(active_win_df) - smash_n
error_n = len(chou_error_df)

labels = ['對手殺球得分', '對手其他球種得分', '周天成自身失誤']
values = [smash_n, other_active_n, error_n]
colors_pie = ['#E74C3C', '#3498DB', '#95A5A6']
explode = (0.08, 0, 0)  # 突出殺球區塊

wedges, texts, autotexts = ax2.pie(
    values,
    labels=None,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors_pie,
    explode=explode,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)

for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')

legend_labels = [f'{l}（{v}次）' for l, v in zip(labels, values)]
ax2.legend(
    wedges, legend_labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.18),
    fontsize=10,
    frameon=True
)

ax2.set_title(
    f"周天成總失分組成分析\n（總失分 {total_lose} 次）",
    fontsize=12, fontweight='bold'
)

plt.tight_layout()
print("\n圖表已生成。")


38. 在周天成的所有失分中，有多少百分比是因為對手的殺球導致的？

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'

# getpoint_player 只會出現在回合最後一拍
final_loss_df = df[
    (df['getpoint_player'] == OPPONENT) &
    (df['player'] == OPPONENT)
].copy()

smash_loss_df = final_loss_df[final_loss_df['type'] == '殺球'].copy()

total_lost_rallies = final_loss_df['rally_id'].nunique()
smash_lost_rallies = smash_loss_df['rally_id'].nunique()

ratio = smash_lost_rallies / total_lost_rallies if total_lost_rallies > 0 else 0

print(f"周天成失分回合數: {total_lost_rallies}")
print(f"其中最後一拍因對手殺球失分的回合數: {smash_lost_rallies}")
print(f"比例: {ratio:.4f}")

39. 當對手使用「殺球」攻擊周天成時，周天成回球的失誤率是多少？

In [ ]:
# 確保數據集不為空
if len(df) > 0:
    # 設定當前球員與對手
    player_name = 'CHOU Tien Chen'
    opponent_name = 'Kento MOMOTA'
    
    # 過濾出周天成對於對手殺球的回球情況
    df['previous_type'] = df.groupby(['match_id', 'set', 'rally'])['type'].shift(1)
    
    # 過濾出對手殺球情況
    opponent_smash_df = df[(df['player'] == player_name) & 
                           (df['previous_type'] == '殺球') &
                           (df['opponent'] == opponent_name)]
    
    # 總的回球次數
    total_replies = len(opponent_smash_df)
    
    # 周天成回球失誤的情況 (對手得分且是周天成失誤)
    error_conditions = opponent_smash_df['getpoint_player'] == opponent_name
    errors_df = opponent_smash_df[error_conditions & opponent_smash_df['lose_reason'].notna()]
    
    # 計算失誤率
    error_count = len(errors_df)
    if total_replies > 0:
        error_rate = (error_count / total_replies) * 100
    else:
        error_rate = 0.0
    
    # 打印結果
    print(f"當對手使用「殺球」攻擊周天成時，周天成回球的失誤率為: {error_rate:.2f}%")
else:
    print("數據集為空，無法計算失誤率。")

39. 當對手使用「殺球」攻擊周天成時，周天成回球的失誤率是多少？

In [ ]:
# 確保資料集非空
if len(df) > 0:
    # 1. 篩選出 Kento MOMOTA 使用「殺球」
    smashes = df[(df['type'] == '殺球') & (df['player'] == 'Kento MOMOTA')]
    
    # 2. 獲取下一拍，也就是周天成的回球
    smashes_next = smashes.copy()
    smashes_next['next_player'] = df['player'].shift(-1)
    smashes_next['next_getpoint_player'] = df['getpoint_player'].shift(-1)
    smashes_next['next_rally'] = df['rally'].shift(-1)

    # 3. 篩選周天成是否回球且在同一回合
    smashes_next_chou = smashes_next[(smashes_next['next_player'] == 'CHOU Tien Chen') & 
                                     (smashes_next['rally'] == smashes_next['next_rally'])]
    
    # 4. 從中查找失誤: 如果 getpoint_player 在下一拍是 Kento MOMOTA，說明周天成失誤導致對方得分
    chou_errors = smashes_next_chou[smashes_next_chou['next_getpoint_player'] == 'Kento MOMOTA']

    # 5. 計算失誤率
    total_attempts = len(smashes_next_chou)
    total_errors = len(chou_errors)
    error_rate = total_errors / total_attempts if total_attempts > 0 else 0

    # 打印結果
    print("周天成在對手使用殺球後的回球次數:", total_attempts)
    print("周天成在對手使用殺球後的回球失誤次數:", total_errors)
    print("周天成在對手使用殺球後的回球失誤率:", error_rate)

40. 分析周天成在開局（他0-4 分）、中局（他5-17 分）和關鍵分（他18 分以上）三個比分階段中，每回合與對手來回的球數的差異

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 計算每個 rally 的來回球數與周天成該回合開始時的比分
rally_info = df.groupby(['match_id', 'set', 'rally']).agg(
    rally_length=('ball_round', 'max'),
    chou_score_start=('CHOU Tien Chen_score', 'first')
).reset_index()

# 2. 依周天成比分分階段
def assign_stage(score):
    if score < 5:
        return '開局 (0-4分)'
    elif score < 18:
        return '中局 (5-17分)'
    else:
        return '關鍵分 (18分以上)'

stage_order = ['開局 (0-4分)', '中局 (5-17分)', '關鍵分 (18分以上)']
rally_info['stage'] = rally_info['chou_score_start'].apply(assign_stage)
rally_info['stage'] = pd.Categorical(rally_info['stage'], categories=stage_order, ordered=True)

# 3. 各階段統計
summary = rally_info.groupby('stage', observed=True)['rally_length'].agg(
    回合數='count',
    平均來回球數='mean',
    中位數='median'
).reindex(stage_order).round(2)

print("各比分階段的來回球數摘要：")
print(summary)

# 4. 必要視覺化：平均來回球數長條圖
avg_rally_length = rally_info.groupby('stage', observed=True)['rally_length'].mean().reindex(stage_order)

plt.figure(figsize=(8, 5))
bars = plt.bar(avg_rally_length.index, avg_rally_length.values, color=['#4F81BD', '#F39C34', '#D9534F'])

for bar, value in zip(bars, avg_rally_length.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{value:.2f}', ha='center', va='bottom')

plt.title('周天成不同比分階段的平均來回球數')
plt.xlabel('比分階段')
plt.ylabel('平均來回球數')
plt.tight_layout()
plt.show()

41. 周天成使用殺球的前一拍，對手都用什麼球種，畫出圓餅圖

In [ ]:
# 擷取殺球前一拍的對手球種
# 過濾周天成的殺球數據
filter_smash = df[(df['player'] == 'CHOU Tien Chen') & (df['type'] == '殺球')]

# 獲得殺球前一拍的對手球種
filter_smash['opponent_previous_type'] = df.groupby(['match_id', 'set', 'rally'])['player_type'].shift(1)

# 確保擷取有效數據
valid_shots = filter_smash.dropna(subset=['opponent_previous_type'])

# 計算對手使用球種的分布
opponent_shot_distribution = valid_shots['opponent_previous_type'].value_counts()

# 球種名稱對照
shot_types = {
    1.0: '發短球', 
    2.0: '發長球', 
    3.0: '長球', 
    4.0: '殺球', 
    5.0: '切球', 
    6.0: '挑球', 
    7.0: '平球', 
    8.0: '網前球', 
    9.0: '推撲球', 
    10.0: '接殺防守', 
    11.0: '接不到'
}

# 將數字轉換為名稱
opponent_shot_distribution.index = opponent_shot_distribution.index.map(shot_types)

# 合併小於 5% 的類別為「其他 (Others)」
threshold = 0.05 * opponent_shot_distribution.sum()
others = opponent_shot_distribution[opponent_shot_distribution < threshold].sum()
opponent_shot_distribution = opponent_shot_distribution[opponent_shot_distribution >= threshold]
opponent_shot_distribution['其他'] = others

# 繪製圓餅圖
fig, ax = plt.subplots()
ax.pie(opponent_shot_distribution, labels=opponent_shot_distribution.index, autopct='%1.1f%%')
ax.set_title('周天成殺球前一拍對手球種分布')

plt.tight_layout()

41. 周天成使用殺球的前一拍，對手都用什麼球種，畫出圓餅圖

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'

type_map = {
    1: '發短球',
    2: '發長球',
    3: '長球',
    4: '殺球',
    5: '切球',
    6: '挑球',
    7: '平球',
    8: '網前球',
    9: '推撲球',
    10: '接殺防守',
    11: '接不到'
}

chou_smash_df = df[
    (df['player'] == PLAYER) &
    (df['type'] == '殺球')
].copy()

# opponent_type 依欄位定義即為對手前一球球種
chou_smash_df['opponent_type_name'] = chou_smash_df['opponent_type'].map(type_map)

counts = chou_smash_df['opponent_type_name'].value_counts()

print("周天成使用殺球前，對手前一拍球種分布：")
print(counts)

plt.figure(figsize=(8, 8))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90)
plt.title('周天成使用殺球前，對手前一拍球種分布')
plt.tight_layout()
plt.show()

42. 分析周天成在開局（自己0-4 分）、中局（自己5-17 分）和關鍵分（自己18 分以上）三個比分階中，他的「殺球」使用比例以及得分率，是否有顯著變化？

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 只看周天成自己的擊球資料
chou_df = df[df['player'] == 'CHOU Tien Chen'].copy()

# 2. 依周天成自己的比分分階段
def assign_stage(score):
    if score < 5:
        return '開局 (0-4分)'
    elif score < 18:
        return '中局 (5-17分)'
    else:
        return '關鍵分 (18分以上)'

chou_df['stage'] = chou_df['CHOU Tien Chen_score'].apply(assign_stage)
stage_order = ['開局 (0-4分)', '中局 (5-17分)', '關鍵分 (18分以上)']
chou_df['stage'] = pd.Categorical(chou_df['stage'], categories=stage_order, ordered=True)

# 3. 定義殺球、主動得分、殺球主動得分
chou_df['is_smash'] = chou_df['type'] == '殺球'
chou_df['is_active_win'] = chou_df['player'] == chou_df['getpoint_player']
chou_df['is_smash_win'] = chou_df['is_smash'] & chou_df['is_active_win']

# 4. 各階段統計
stage_stats = chou_df.groupby('stage', observed=True).agg(
    total_shots=('type', 'count'),
    smash_shots=('is_smash', 'sum'),
    smash_wins=('is_smash_win', 'sum')
).reindex(stage_order)

stage_stats['smash_usage_rate'] = stage_stats['smash_shots'] / stage_stats['total_shots']
stage_stats['smash_win_rate'] = stage_stats['smash_wins'] / stage_stats['smash_shots']

print("周天成在不同比分階段的殺球使用比例與得分率：")
print(stage_stats[['total_shots', 'smash_shots', 'smash_wins', 'smash_usage_rate', 'smash_win_rate']].round(4))

# 5. 視覺化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(stage_stats.index.astype(str), stage_stats['smash_usage_rate'], color=['#4F81BD', '#F39C34', '#D9534F'])
axes[0].set_title('不同比分階段的殺球使用比例')
axes[0].set_xlabel('比分階段')
axes[0].set_ylabel('殺球使用比例')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(stage_stats.index.astype(str), stage_stats['smash_win_rate'], color=['#4F81BD', '#F39C34', '#D9534F'])
axes[1].set_title('不同比分階段的殺球得分率')
axes[1].set_xlabel('比分階段')
axes[1].set_ylabel('殺球得分率')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

42. 分析周天成在開局（自己0-4 分）、中局（自己5-17 分）和關鍵分（自己18 分以上）三個比分階中，他的「殺球」使用比例以及得分率，是否有顯著變化？

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
ctc_df = df[df['player'] == PLAYER].copy()

def score_phase(score):
    if score <= 4:
        return '開局(0-4)'
    elif score <= 17:
        return '中局(5-17)'
    else:
        return '關鍵分(18+)'

ctc_df['score_phase'] = ctc_df['CHOU Tien Chen_score'].apply(score_phase)
ctc_df['is_smash'] = ctc_df['type'] == '殺球'
ctc_df['is_smash_active_win'] = (
    (ctc_df['type'] == '殺球') &
    (ctc_df['player'] == PLAYER) &
    (ctc_df['getpoint_player'] == PLAYER)
)
ctc_df['is_active_win_any'] = (
    (ctc_df['player'] == PLAYER) &
    (ctc_df['getpoint_player'] == PLAYER)
)

summary = (
    ctc_df.groupby('score_phase')
    .agg(
        total_shots=('type', 'count'),
        smash_shots=('is_smash', 'sum'),
        total_active_wins=('is_active_win_any', 'sum'),
        smash_active_wins=('is_smash_active_win', 'sum')
    )
    .reset_index()
)

summary['smash_usage_rate'] = summary['smash_shots'] / summary['total_shots']
summary['smash_active_win_rate'] = summary['smash_active_wins'] / summary['total_active_wins']

print(summary)

43. 計算周天成在落後 3 分以上時，平均每個回合的擊球次數，並與比分膠著時的回合平均擊球次數進行比較。

In [ ]:
# 1. 先整理每個 rally 的基本資訊
rally_info = df.groupby(['match_id', 'set', 'rally']).agg(
    rally_length=('ball_round', 'max'),
    chou_score=('CHOU Tien Chen_score', 'first'),
    momota_score=('Kento MOMOTA_score', 'first')
).reset_index()

# 2. 定義比分狀態
# 落後 3 分以上：周天成比分落後對手至少 3 分
# 比分膠著：雙方分差在 1 分以內
rally_info['score_diff'] = rally_info['chou_score'] - rally_info['momota_score']

def classify_score_state(diff):
    if diff <= -3:
        return '落後3分以上'
    elif abs(diff) <= 1:
        return '比分膠著'
    else:
        return '其他'

rally_info['score_state'] = rally_info['score_diff'].apply(classify_score_state)

# 3. 只保留題目要求的兩種情況
target_rallies = rally_info[rally_info['score_state'].isin(['落後3分以上', '比分膠著'])].copy()

# 4. 計算各情況的平均每回合擊球次數
summary = target_rallies.groupby('score_state', observed=True).agg(
    回合數=('rally_length', 'count'),
    平均擊球次數=('rally_length', 'mean'),
    中位數=('rally_length', 'median')
).reindex(['落後3分以上', '比分膠著']).round(2)

print("周天成在不同比分狀態下的每回合平均擊球次數：")
print(summary)

# 5. 視覺化比較
avg_lengths = summary['平均擊球次數']

plt.figure(figsize=(8, 5))
bars = plt.bar(avg_lengths.index, avg_lengths.values, color=['#D9534F', '#4F81BD'])

for bar, value in zip(bars, avg_lengths.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        f'{value:.2f}',
        ha='center',
        va='bottom'
    )

plt.title('周天成在不同比分狀態下的平均每回合擊球次數')
plt.xlabel('比分狀態')
plt.ylabel('平均擊球次數')
plt.tight_layout()
plt.show()

44. 幫我統整周天成（CHOU Tien Chen）的得分戰術組合(例如:最常見的是，他切球，對手挑球，他殺球得分)，並列出前五名

In [ ]:
import pandas as pd

# 假設 `df` 已經被正確地載入，不需要重新讀取數據
# 設定正確的字體 (這步驟省略)

# 確保數據完整性
if len(df) == 0:
    print("數據集為空，無法進行分析")
else:
    # 選取周天成得分的情況
    chou_win_points = df[(df['getpoint_player'] == 'CHOU Tien Chen') & (df['player'] == 'CHOU Tien Chen')]

    # 計算並獲取得分前的兩次球拍類型
    chou_sorted = df.sort_values(['match_id', 'set', 'rally', 'ball_round'])

    # Shift to get previous shot types
    chou_sorted['type_prev_1'] = chou_sorted.groupby(['match_id', 'set', 'rally'])['type'].shift(1)
    chou_sorted['type_prev_2'] = chou_sorted.groupby(['match_id', 'set', 'rally'])['type'].shift(2)

    # 篩選出周天成擊球並且得分的記錄
    chou_points = chou_sorted[(chou_sorted['getpoint_player'] == 'CHOU Tien Chen') & (chou_sorted['player'] == 'CHOU Tien Chen')]
    
    # 統計戰術組合
    chou_points['tactical_combination'] = chou_points['type_prev_2'] + ' -> ' + chou_points['type_prev_1'] + ' -> ' + chou_points['type']
    tactic_counts = chou_points['tactical_combination'].value_counts().head(5)
    
    # 打印最常見的前五名戰術組合
    print("周天成比賽中最常見的前五名得分戰術組合:")
    print(tactic_counts)

45. 統整周天成每個場次中各局的表現(例如，球種變化、殺球頻與失誤率、失誤原因等等)

In [ ]:
# 檢查數據長度
if len(df) > 0:
    # 篩選出周天成的數據
    chou_df = df[df['player'] == 'CHOU Tien Chen']

    # 按照 `match_id` 和 `set` 分組
    grouped_df = chou_df.groupby(['match_id', 'set'])

    # 計算每一局各球種的使用次數
    type_count = grouped_df['type'].value_counts().unstack(fill_value=0)

    # 計算殺球頻率
    kills_count = grouped_df.apply(lambda x: (x['type'] == '殺球').sum())
    total_shots = grouped_df.size()
    kill_rate = (kills_count / total_shots) * 100  # 百分比

    # 計算失誤率和失誤原因
    lose_reasons = grouped_df['lose_reason'].value_counts().unstack(fill_value=0)
    faults_count = grouped_df.apply(lambda x: x['lose_reason'].notna().sum())
    fault_rate = (faults_count / total_shots) * 100  # 百分比

    # 打印統計結果
    print("周天成各場比賽中的數據統計：")
    print("--------------------------------------------------")
    print("\n每局的球種使用次數：")
    print(type_count)
    print("\n每局的殺球頻率 (%)：")
    print(kill_rate)
    print("\n每局的失誤率 (%)：")
    print(fault_rate)
    print("\n每局的失誤原因：")
    print(lose_reasons)

    # 可視化
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))
    
    # 繪制各球種使用次數
    type_count.plot(kind='bar', stacked=True, ax=axes[0, 0])
    axes[0, 0].set_title('周天成各局的球種使用次數')
    axes[0, 0].set_xlabel('比賽局 (match_id, set)')
    axes[0, 0].set_ylabel('次數')
    
    # 繪制殺球頻率
    kill_rate.plot(kind='bar', color='coral', ax=axes[0, 1])
    axes[0, 1].set_title('周天成各局的殺球頻率 (%)')
    axes[0, 1].set_xlabel('比賽局 (match_id, set)')
    axes[0, 1].set_ylabel('頻率 (%)')
    
    # 繪制失誤率
    fault_rate.plot(kind='bar', color='skyblue', ax=axes[1, 0])
    axes[1, 0].set_title('周天成各局的失誤率 (%)')
    axes[1, 0].set_xlabel('比賽局 (match_id, set)')
    axes[1, 0].set_ylabel('頻率 (%)')
    
    # 繪制失誤原因
    lose_reasons.plot(kind='bar', stacked=True, ax=axes[1, 1])
    axes[1, 1].set_title('周天成各局的失誤原因')
    axes[1, 1].set_xlabel('比賽局 (match_id, set)')
    axes[1, 1].set_ylabel('次數')
    
    plt.tight_layout()
else:
    print('數據不足，無法進一步分析。')

45. 統整周天成每個場次中各局的表現(例如，球種變化、殺球頻與失誤率、失誤原因等等)

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_keys = ['match_id', 'set']

ctc_df = df[df['player'] == PLAYER].copy()
ctc_df['is_last_shot'] = (
    ctc_df['ball_round'] ==
    ctc_df.groupby(['match_id', 'set', 'rally'])['ball_round'].transform('max')
)

ctc_df['is_active_win'] = (
    ctc_df['is_last_shot'] &
    (ctc_df['getpoint_player'] == PLAYER)
)

ctc_df['is_lost_point'] = (
    ctc_df['is_last_shot'] &
    (ctc_df['getpoint_player'] == OPP)
)

ctc_df['is_smash'] = ctc_df['type'] == '殺球'

summary = (
    ctc_df.groupby(group_keys)
    .agg(
        total_shots=('type', 'count'),
        unique_shot_types=('type', 'nunique'),
        smash_count=('is_smash', 'sum'),
        active_wins=('is_active_win', 'sum'),
        lost_points=('is_lost_point', 'sum')
    )
    .reset_index()
)

summary['smash_usage_rate'] = summary['smash_count'] / summary['total_shots']
print(summary)

error_reason = (
    ctc_df[ctc_df['is_lost_point'] & ctc_df['lose_reason'].notna()]
    .groupby(group_keys + ['lose_reason'])
    .size()
    .reset_index(name='count')
)

print(error_reason)

46. 哪些球種，在比賽後半段失誤率顯著上升？

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

analysis_df = df[df['player'] == PLAYER].copy()
set_max_rally = df.groupby(['match_id', 'set'])['rally'].transform('max')
analysis_df['set_max_rally'] = set_max_rally.loc[analysis_df.index]
analysis_df['match_phase'] = analysis_df.apply(
    lambda row: '後半段' if row['rally'] > row['set_max_rally'] / 2 else '前半段',
    axis=1
)

# 周天成自己失誤：此拍 player 是周天成、lose_reason 有值，且回合得分者為對手。
analysis_df['is_error'] = analysis_df['lose_reason'].notna() & analysis_df['getpoint_player'].eq(OPPONENT)

summary = (
    analysis_df.groupby(['type', 'match_phase'])
    .agg(擊球數=('type', 'size'), 失誤數=('is_error', 'sum'))
    .reset_index()
)
summary['失誤率'] = summary['失誤數'] / summary['擊球數']

pivot = summary.pivot(index='type', columns='match_phase', values='失誤率').fillna(0)
count_pivot = summary.pivot(index='type', columns='match_phase', values='擊球數').fillna(0)
for col in ['前半段', '後半段']:
    if col not in pivot.columns:
        pivot[col] = 0
    if col not in count_pivot.columns:
        count_pivot[col] = 0

comparison = pd.DataFrame({
    '前半段失誤率': pivot['前半段'],
    '後半段失誤率': pivot['後半段'],
    '後半段-前半段': pivot['後半段'] - pivot['前半段'],
    '前半段擊球數': count_pivot['前半段'],
    '後半段擊球數': count_pivot['後半段'],
}).sort_values('後半段-前半段', ascending=False)

print('周天成各球種在比賽前半段 vs 後半段的失誤率差異：')
print(comparison)
print('\n後半段失誤率上升最多的球種：')
print(comparison[comparison['後半段-前半段'] > 0].head(5))

fig, ax = plt.subplots(figsize=(10, 5))
comparison['後半段-前半段'].sort_values().plot(kind='barh', ax=ax, color='#F58518')
ax.axvline(0, color='black', linewidth=1)
ax.set_title('周天成各球種後半段失誤率變化')
ax.set_xlabel('後半段失誤率 - 前半段失誤率')
ax.set_ylabel('球種')
plt.tight_layout()

46. 哪些球種，在比賽後半段失誤率顯著上升？

In [ ]:
import pandas as pd

# 1. 確認資料非空
if len(df) == 0:
    print("DataFrame is empty. No analysis can be conducted.")
else:
    # 2. 定義前半段 / 後半段
    #    這裡採較寬鬆定義：任一方達 10 分後視為後半段
    df_analysis = df.copy()
    df_analysis['match_half'] = df_analysis.apply(
        lambda row: '後半段'
        if (row['CHOU Tien Chen_score'] >= 10 or row['Kento MOMOTA_score'] >= 10)
        else '前半段',
        axis=1
    )

    # 3. 定義失誤
    #    只要 lose_reason 有值，就視為該拍為失誤
    df_analysis['is_error'] = df_analysis['lose_reason'].notna()

    # 4. 統計各球種在前後半段的總次數與失誤次數
    summary = df_analysis.groupby(['type', 'match_half']).agg(
        total_shots=('type', 'count'),
        error_count=('is_error', 'sum')
    ).reset_index()

    summary['error_rate'] = summary['error_count'] / summary['total_shots']

    print("各球種在前後半段的失誤率：")
    print(summary)

    # 5. 轉成前後半段對照表
    pivot_table = summary.pivot(index='type', columns='match_half', values='error_rate').fillna(0)

    if '前半段' not in pivot_table.columns:
        pivot_table['前半段'] = 0
    if '後半段' not in pivot_table.columns:
        pivot_table['後半段'] = 0

    pivot_table['increase'] = pivot_table['後半段'] - pivot_table['前半段']
    pivot_table = pivot_table.sort_values('increase', ascending=False)

    print("\n後半段失誤率上升較多的球種：")
    print(pivot_table)

47. 比較周天成與他的對手的球種使用頻率與其得分率。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 比較兩位球員各球種使用頻率與直接得分率。
analysis_df = df.copy()
analysis_df['is_direct_score'] = analysis_df['player'].eq(analysis_df['getpoint_player'])
player_type_stats = (
    analysis_df.groupby(['player', 'type'])
    .agg(使用次數=('type', 'size'), 直接得分次數=('is_direct_score', 'sum'))
    .reset_index()
)
player_type_stats['使用比例'] = player_type_stats['使用次數'] / player_type_stats.groupby('player')['使用次數'].transform('sum')
player_type_stats['直接得分率'] = player_type_stats['直接得分次數'] / player_type_stats['使用次數']

print('周天成與對手的球種使用頻率與直接得分率：')
print(player_type_stats.sort_values(['player', '使用次數'], ascending=[True, False]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
usage_pivot = player_type_stats.pivot(index='type', columns='player', values='使用比例').fillna(0)
score_pivot = player_type_stats.pivot(index='type', columns='player', values='直接得分率').fillna(0)
usage_pivot.plot(kind='bar', ax=axes[0])
axes[0].set_title('球種使用比例')
axes[0].set_xlabel('球種')
axes[0].set_ylabel('使用比例')
score_pivot.plot(kind='bar', ax=axes[1])
axes[1].set_title('各球種直接得分率')
axes[1].set_xlabel('球種')
axes[1].set_ylabel('直接得分率')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

47. 比較周天成與他的對手的球種使用頻率與其得分率。

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

players = ['CHOU Tien Chen', 'Kento MOMOTA']
work = df[df['player'].isin(players) & df['type'].notna()].copy()

work['is_active_win'] = work['player'] == work['getpoint_player']

summary = (
    work.groupby(['player', 'type'])
    .agg(
        usage_count=('type', 'size'),
        active_win_count=('is_active_win', 'sum')
    )
    .reset_index()
)

summary['usage_total'] = summary.groupby('player')['usage_count'].transform('sum')
summary['usage_frequency'] = summary['usage_count'] / summary['usage_total']
summary['active_win_rate'] = summary['active_win_count'] / summary['usage_count']

print(summary.sort_values(['player', 'usage_count'], ascending=[True, False]))

48. 比較周天成在各局自己先得分與對手先得分情況下的勝率差別。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']
SET_COLS = ['match_id', 'set']

rally_result = (
    df.sort_values(GROUP_COLS + ['ball_round'])
    .groupby(GROUP_COLS, as_index=False)
    .last()[GROUP_COLS + ['getpoint_player']]
)
valid_rallies = rally_result[rally_result['getpoint_player'].notna()].copy()

first_scorer = valid_rallies.groupby(SET_COLS, as_index=False).first()[SET_COLS + ['getpoint_player']]
first_scorer = first_scorer.rename(columns={'getpoint_player': 'first_point_player'})
set_winner = valid_rallies.groupby(SET_COLS, as_index=False).last()[SET_COLS + ['getpoint_player']]
set_winner = set_winner.rename(columns={'getpoint_player': 'set_winner'})

set_summary = pd.merge(first_scorer, set_winner, on=SET_COLS, how='inner')
set_summary['first_point_group'] = set_summary['first_point_player'].apply(lambda p: '周天成先得分' if p == PLAYER else '對手先得分')
set_summary['chou_won_set'] = set_summary['set_winner'].eq(PLAYER)

result = set_summary.groupby('first_point_group').agg(
    局數=('set', 'size'),
    周天成勝局數=('chou_won_set', 'sum'),
    周天成勝率=('chou_won_set', 'mean')
).reset_index()

print('每局先得分者與該局勝者：')
print(set_summary)
print('\n周天成在自己先得分 vs 對手先得分時的勝率：')
print(result)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(result['first_point_group'], result['周天成勝率'], color='#54A24B')
ax.set_title('各局先得分情況下的周天成勝率')
ax.set_xlabel('先得分情況')
ax.set_ylabel('周天成勝率')
ax.set_ylim(0, 1)
plt.tight_layout()

49. 周天成的發球種類是否影響他在該回合的得分率

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']
SERVE_TYPES = ['發短球', '發長球']

ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
rally_result = ordered_df.groupby(GROUP_COLS, as_index=False).last()[GROUP_COLS + ['getpoint_player']]
rally_result = rally_result.rename(columns={'getpoint_player': 'rally_winner'})

serve_df = ordered_df[
    (ordered_df['player'] == PLAYER) &
    (ordered_df['ball_round'] == 1) &
    (ordered_df['type'].isin(SERVE_TYPES))
].copy()
serve_df = pd.merge(serve_df, rally_result, on=GROUP_COLS, how='left')
serve_df['chou_won_rally'] = serve_df['rally_winner'].eq(PLAYER)

serve_summary = serve_df.groupby('type').agg(
    發球次數=('type', 'size'),
    得分回合數=('chou_won_rally', 'sum'),
    得分率=('chou_won_rally', 'mean')
).reset_index()

print('周天成不同發球種類對回合得分率的影響：')
print(serve_summary)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(serve_summary['type'], serve_summary['得分率'], color='#9D755D')
ax.set_title('周天成發球種類與回合得分率')
ax.set_xlabel('發球種類')
ax.set_ylabel('得分率')
ax.set_ylim(0, 1)
plt.tight_layout()

50. 分析周天成接發球策略：長球、網前球、平球的比例與得分影響。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']
RECEIVE_TYPES = ['長球', '網前球', '平球']

ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
rally_result = ordered_df.groupby(GROUP_COLS, as_index=False).last()[GROUP_COLS + ['getpoint_player']]
rally_result = rally_result.rename(columns={'getpoint_player': 'rally_winner'})

# 接發球是每個 rally 的第 2 拍；只分析周天成接發球時採用的長球/網前球/平球。
receive_df = ordered_df[
    (ordered_df['player'] == PLAYER) &
    (ordered_df['ball_round'] == 2) &
    (ordered_df['type'].isin(RECEIVE_TYPES))
].copy()
receive_df = pd.merge(receive_df, rally_result, on=GROUP_COLS, how='left')
receive_df['chou_won_rally'] = receive_df['rally_winner'].eq(PLAYER)

receive_summary = receive_df.groupby('type').agg(
    使用次數=('type', 'size'),
    比例=('type', lambda s: len(s) / len(receive_df) if len(receive_df) else 0),
    得分回合數=('chou_won_rally', 'sum'),
    得分率=('chou_won_rally', 'mean')
).reset_index()

print('周天成接發球策略與得分影響：')
print(receive_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(receive_summary['type'], receive_summary['比例'], color='#4C78A8')
axes[0].set_title('接發球策略比例')
axes[0].set_xlabel('接發球球種')
axes[0].set_ylabel('比例')
axes[1].bar(receive_summary['type'], receive_summary['得分率'], color='#54A24B')
axes[1].set_title('接發球策略得分率')
axes[1].set_xlabel('接發球球種')
axes[1].set_ylabel('得分率')
axes[1].set_ylim(0, 1)
plt.tight_layout()

50. 分析周天成接發球策略：長球、網前球、平球的比例與得分影響。

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

group_cols = ['match_id', 'set', 'rally']
PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
TARGET_TYPES = ['長球', '網前球', '平球']

# 每回合第一拍：判斷發球者
rally_first = (
    df.sort_values(group_cols + ['ball_round'])
      .groupby(group_cols, as_index=False)
      .first()[group_cols + ['player']]
      .rename(columns={'player': 'server_player'})
)

# 只看對手發球的回合
opp_serve_rallies = rally_first[rally_first['server_player'] == OPP][group_cols]

# 周天成接發球：第 2 拍且為周天成
receive_df = (
    df[(df['ball_round'] == 2) & (df['player'] == PLAYER)]
    .merge(opp_serve_rallies, on=group_cols, how='inner')
    .copy()
)

receive_df = receive_df[receive_df['type'].isin(TARGET_TYPES)].copy()

# 每回合最終得分者
rally_last = (
    df.sort_values(group_cols + ['ball_round'])
      .groupby(group_cols, as_index=False)
      .last()[group_cols + ['getpoint_player']]
)

receive_df = receive_df.merge(rally_last, on=group_cols, how='left')
receive_df['rally_win'] = receive_df['getpoint_player'] == PLAYER

# 使用比例
shot_counts = receive_df['type'].value_counts().reindex(TARGET_TYPES, fill_value=0)
shot_ratio = (shot_counts / shot_counts.sum()).rename('ratio')

# 回合勝率
win_rate = (
    receive_df.groupby('type')['rally_win']
    .mean()
    .reindex(TARGET_TYPES)
    .rename('rally_win_rate')
)

result = pd.concat([shot_counts.rename('count'), shot_ratio, win_rate], axis=1)
print(result)

51. 分析對手殺球到前中後排後，周天成分別會怎麼去反擊？

In [ ]:
# 1. 定義前中後場區域
front_court = [17, 18, 19, 20, 21, 22, 23, 24]
mid_court = [9, 10, 11, 12, 13, 14, 15, 16]
back_court = [1, 2, 3, 4, 5, 6, 7, 8]

# 2. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(['match_id', 'set', 'rally'])['player'].shift(-1)
df_next['next_type'] = df_next.groupby(['match_id', 'set', 'rally'])['type'].shift(-1)

# 3. 找出對手（Kento MOMOTA）使用殺球的資料
opponent_smash_df = df_next[
    (df_next['player'] == 'Kento MOMOTA') &
    (df_next['type'] == '殺球')
].copy()

# 4. 只保留下一拍確實是周天成回擊的情況
opponent_smash_df = opponent_smash_df[
    opponent_smash_df['next_player'] == 'CHOU Tien Chen'
].copy()

# 5. 根據對手殺球的落點區域分成前中後場
def classify_court_area(area):
    if area in front_court:
        return '前場'
    elif area in mid_court:
        return '中場'
    elif area in back_court:
        return '後場'
    else:
        return '其他'

opponent_smash_df['smash_landing_zone'] = opponent_smash_df['landing_area'].apply(classify_court_area)

# 6. 統計周天成在不同區域面對殺球時的回擊球種
response_summary = (
    opponent_smash_df
    .groupby(['smash_landing_zone', 'next_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['smash_landing_zone', 'count'], ascending=[True, False])
)

print("對手殺球到前中後場後，周天成的回擊球種統計：")
print(response_summary)

# 7. 額外列出各區域最常見的反擊球種
top_responses = response_summary.groupby('smash_landing_zone').head(3)

print("\n各區域最常見的反擊方式（前 3 名）：")
print(top_responses)

51. 分析對手殺球到前中後排後，周天成分別會怎麼去反擊？

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_cols = ['match_id', 'set', 'rally']

work = df.sort_values(group_cols + ['ball_round']).copy()
work['next_player'] = work.groupby(group_cols)['player'].shift(-1)
work['next_type'] = work.groupby(group_cols)['type'].shift(-1)

def classify_zone(area):
    if pd.isna(area):
        return '其他'
    area = int(area)
    if 17 <= area <= 24:
        return '前排'
    elif 5 <= area <= 16:
        return '中排'
    elif 1 <= area <= 4:
        return '後排'
    return '其他'

opp_smash = work[
    (work['player'] == OPP) &
    (work['type'] == '殺球') &
    (work['next_player'] == PLAYER)
].copy()

opp_smash['smash_zone'] = opp_smash['landing_area'].apply(classify_zone)
opp_smash = opp_smash[opp_smash['smash_zone'].isin(['前排', '中排', '後排'])].copy()

summary = (
    opp_smash.groupby(['smash_zone', 'next_type'])
    .size()
    .reset_index(name='count')
)

summary['zone_total'] = summary.groupby('smash_zone')['count'].transform('sum')
summary['ratio'] = summary['count'] / summary['zone_total']

print(summary.sort_values(['smash_zone', 'count'], ascending=[True, False]))

52. 當對手失誤掛網時，分析前一拍周天成出球球種與落點特徵。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立前一拍資訊
df_prev = df.copy()
df_prev['prev_player'] = df_prev.groupby(GROUP_COLS)['player'].shift(1)
df_prev['prev_type'] = df_prev.groupby(GROUP_COLS)['type'].shift(1)
df_prev['prev_landing_area'] = df_prev.groupby(GROUP_COLS)['landing_area'].shift(1)
df_prev['prev_landing_x'] = df_prev.groupby(GROUP_COLS)['landing_x'].shift(1)
df_prev['prev_landing_y'] = df_prev.groupby(GROUP_COLS)['landing_y'].shift(1)

# 2. 找出對手失誤掛網的情況
opponent_net_error_df = df_prev[
    (df_prev['player'] == OPPONENT) &
    (df_prev['lose_reason'] == '掛網')
].copy()

# 3. 只保留前一拍是周天成出球的情況
target_df = opponent_net_error_df[
    opponent_net_error_df['prev_player'] == PLAYER
].copy()

print(f"對手失誤掛網且前一拍為周天成出球的次數: {len(target_df)}")

# 4. 統計前一拍周天成的球種
prev_type_counts = target_df['prev_type'].value_counts().reset_index()
prev_type_counts.columns = ['前一拍球種', '次數']

print("\n前一拍周天成出球球種分布：")
print(prev_type_counts)

# 5. 統計前一拍周天成的落點區域
prev_landing_area_counts = target_df['prev_landing_area'].value_counts().reset_index()
prev_landing_area_counts.columns = ['前一拍落點區域', '次數']

print("\n前一拍周天成落點區域分布：")
print(prev_landing_area_counts)

# 6. 輸出前一拍落點座標的摘要特徵
landing_summary = target_df[['prev_landing_x', 'prev_landing_y']].describe()

print("\n前一拍周天成落點座標特徵摘要：")
print(landing_summary)

53. 統計周天成所有「得分帶有連續性」的回合（連 2 分以上）的常見戰術組合。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 先整理每個 rally 的最終得分者
rally_result = (
    df.groupby(GROUP_COLS)
    .last()
    .reset_index()[GROUP_COLS + ['getpoint_player']]
)

# 2. 依比賽流程排序後，判斷周天成是否處於連續得分中
rally_result = rally_result.sort_values(GROUP_COLS).reset_index(drop=True)
rally_result['chou_scored'] = rally_result['getpoint_player'] == PLAYER

# 3. 建立連續段落編號
rally_result['streak_group'] = (
    rally_result['chou_scored'] != rally_result['chou_scored'].shift(1)
).cumsum()

# 4. 找出「周天成連續得分 2 分以上」的 rally
streak_summary = (
    rally_result.groupby('streak_group')
    .agg(
        chou_scored=('chou_scored', 'first'),
        streak_len=('chou_scored', 'size')
    )
    .reset_index()
)

valid_streak_ids = streak_summary[
    (streak_summary['chou_scored']) &
    (streak_summary['streak_len'] >= 2)
]['streak_group']

target_rallies = rally_result[
    rally_result['streak_group'].isin(valid_streak_ids)
][GROUP_COLS].copy()

print(f"周天成連續得分 2 分以上的回合數: {len(target_rallies)}")

# 5. 在完整資料上建立前兩拍資訊
df_seq = df.copy()
df_seq['prev_type_1'] = df_seq.groupby(GROUP_COLS)['type'].shift(1)
df_seq['prev_type_2'] = df_seq.groupby(GROUP_COLS)['type'].shift(2)
df_seq['prev_player_1'] = df_seq.groupby(GROUP_COLS)['player'].shift(1)
df_seq['prev_player_2'] = df_seq.groupby(GROUP_COLS)['player'].shift(2)

# 6. 抓出這些回合中，周天成最後得分球的三拍組合
target_shots = pd.merge(df_seq, target_rallies, on=GROUP_COLS, how='inner')

combo_df = target_shots[
    (target_shots['player'] == PLAYER) &
    (target_shots['getpoint_player'] == PLAYER) &
    target_shots['prev_type_1'].notna() &
    target_shots['prev_type_2'].notna()
].copy()

combo_df = combo_df[
    (combo_df['prev_player_2'] == PLAYER) &
    (combo_df['prev_player_1'] != PLAYER)
].copy()

combo_df['strategy_combo'] = (
    combo_df['prev_type_2'] + ' -> ' +
    combo_df['prev_type_1'] + ' -> ' +
    combo_df['type']
)

combo_counts = combo_df['strategy_combo'].value_counts().reset_index()
combo_counts.columns = ['戰術組合', '次數']

print("\n周天成在連續得分回合中的常見戰術組合：")
print(combo_counts.head(10))

54. 統計周天成所有「一拍得分」的比例與地點分布。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 先整理每個 rally 的基本資訊
rally_summary = (
    df.groupby(GROUP_COLS)
    .agg(
        rally_length=('ball_round', 'max'),
        getpoint_player=('getpoint_player', 'last'),
        landing_area=('landing_area', 'first'),
        landing_x=('landing_x', 'first'),
        landing_y=('landing_y', 'first')
    )
    .reset_index()
)

# 2. 定義「一拍得分」
#    - 該回合只有 1 拍
#    - 最後得分者是周天成
one_shot_score_df = rally_summary[
    (rally_summary['rally_length'] == 1) &
    (rally_summary['getpoint_player'] == PLAYER)
].copy()

# 3. 計算比例
total_scoring_rallies = (rally_summary['getpoint_player'] == PLAYER).sum()
one_shot_score_count = len(one_shot_score_df)
one_shot_score_rate = (
    one_shot_score_count / total_scoring_rallies
    if total_scoring_rallies > 0 else 0
)

print(f"周天成總得分回合數: {total_scoring_rallies}")
print(f"一拍得分回合數: {one_shot_score_count}")
print(f"一拍得分比例: {one_shot_score_rate:.4f}")

# 4. 地點分布：先看區域代碼分布
landing_area_counts = one_shot_score_df['landing_area'].value_counts().reset_index()
landing_area_counts.columns = ['landing_area', 'count']

print("\n一拍得分的落點區域分布：")
print(landing_area_counts)

# 5. 額外輸出落點座標摘要
landing_coordinate_summary = one_shot_score_df[['landing_x', 'landing_y']].describe()

print("\n一拍得分的落點座標摘要：")
print(landing_coordinate_summary)

54. 統計周天成所有「一拍得分」的比例與地點分布。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 找出周天成一拍得分
one_shot_df = df[
    (df['ball_round'] == 1) &
    (df['getpoint_player'] == PLAYER)
].copy()

# 2. 計算比例（以所有 rally 為分母）
total_rallies = df.groupby(GROUP_COLS).ngroups
one_shot_rallies = one_shot_df.groupby(GROUP_COLS).ngroups
one_shot_ratio = one_shot_rallies / total_rallies if total_rallies > 0 else 0

print(f"周天成一拍得分回合數: {one_shot_rallies}")
print(f"總回合數: {total_rallies}")
print(f"一拍得分比例: {one_shot_ratio:.4f}")

# 3. 地點分布
landing_area_counts = one_shot_df['landing_area'].value_counts().reset_index()
landing_area_counts.columns = ['landing_area', 'count']

print("\n一拍得分的落點分布：")
print(landing_area_counts)

55. 當周天成打出反手回擊時，對手下一拍攻擊成功率是多少？

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)
df_next['next_getpoint_player'] = df_next.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 2. 只看周天成的反手回擊
chou_backhand_df = df_next[
    (df_next['player'] == PLAYER) &
    (df_next['backhand'] == 1)
].copy()

# 3. 找出對手所有下一拍
opponent_next_shot_df = chou_backhand_df[
    chou_backhand_df['next_player'] == OPPONENT
].copy()

# 4. 定義對手下一拍成功
#    採較嚴格定義：對手下一拍直接得分
opponent_next_shot_df['shot_success'] = (
    opponent_next_shot_df['next_getpoint_player'] == OPPONENT
)

# 5. 計算整體成功率
total_next_shots = len(opponent_next_shot_df)
success_count = opponent_next_shot_df['shot_success'].sum()
success_rate = success_count / total_next_shots if total_next_shots > 0 else 0

print(f"周天成反手回擊總次數: {len(chou_backhand_df)}")
print(f"其中對手有下一拍回擊次數: {total_next_shots}")
print(f"對手下一拍直接得分次數: {success_count}")
print(f"對手下一拍成功率: {success_rate:.4f}")

# 6. 分球種統計
shot_summary = opponent_next_shot_df.groupby('next_type').agg(
    使用次數=('next_type', 'count'),
    成功次數=('shot_success', 'sum')
).reset_index()

shot_summary['成功率'] = shot_summary['成功次數'] / shot_summary['使用次數']

print("\n對手不同下一拍球種的成功率：")
print(shot_summary)

56. 當周天成在「局末追分」情況下是否更依賴殺球？（使用率 vs 分數差）

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'

# 1. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 2. 計算分數差（周天成分數 - 對手分數）
chou_df['score_diff'] = chou_df['CHOU Tien Chen_score'] - chou_df['Kento MOMOTA_score']

# 3. 定義局末追分情況
#    - 局末：任一方分數 >= 18
#    - 追分：周天成落後（score_diff < 0）
endgame_chasing_df = chou_df[
    (
        (chou_df['CHOU Tien Chen_score'] >= 18) |
        (chou_df['Kento MOMOTA_score'] >= 18)
    ) &
    (chou_df['score_diff'] < 0)
].copy()

# 4. 統計不同落後分差下的總擊球數與殺球次數
summary = endgame_chasing_df.groupby('score_diff').agg(
    總擊球數=('type', 'count'),
    殺球次數=('type', lambda x: (x == '殺球').sum())
).reset_index()

summary['殺球使用率'] = summary['殺球次數'] / summary['總擊球數']
summary = summary.sort_values('score_diff')

print("周天成在局末追分情況下，不同落後分差的殺球使用率：")
print(summary)

# 5. 視覺化
plt.figure(figsize=(8, 5))
plt.plot(summary['score_diff'], summary['殺球使用率'], marker='o', linewidth=2)

for _, row in summary.iterrows():
    plt.text(
        row['score_diff'],
        row['殺球使用率'] + 0.005,
        f"{row['殺球使用率']:.2%}",
        ha='center',
        va='bottom'
    )

plt.title('周天成在局末追分時的殺球使用率 vs 分數差')
plt.xlabel('分數差（周天成 - 對手）')
plt.ylabel('殺球使用率')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

57. 分析周天成發短球後，對手下一拍最常使用的球種分布。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)

# 2. 找出周天成的發短球
short_serve_df = df_next[
    (df_next['player'] == PLAYER) &
    (df_next['type'] == '發短球') &
    df_next['next_player'].notna()
].copy()

# 3. 統計發短球後，對手下一拍球種分布
next_type_counts = short_serve_df['next_type'].value_counts()

print("周天成發短球後，對手下一拍球種分布：")
print(next_type_counts)

# 4. 視覺化
if not next_type_counts.empty:
    plt.figure(figsize=(8, 8))
    plt.pie(
        next_type_counts,
        labels=next_type_counts.index,
        autopct='%1.1f%%',
        startangle=90
    )
    plt.title('周天成發短球後，對手下一拍球種分布')
    plt.tight_layout()
    plt.show()

58. 統計周天成在對手殺球之後，自己最常採用的前三種回擊球種。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)

# 2. 找出對手的殺球，且下一拍確實是周天成回擊
opponent_smash_df = df_next[
    (df_next['player'] == OPPONENT) &
    (df_next['type'] == '殺球') &
    (df_next['next_player'] == PLAYER)
].copy()

# 3. 統計周天成回擊球種分布
response_counts = opponent_smash_df['next_type'].value_counts().reset_index()
response_counts.columns = ['周天成回擊球種', '次數']
response_counts['比例'] = response_counts['次數'] / response_counts['次數'].sum()

print("周天成在對手殺球之後最常採用的前三種回擊球種：")
print(response_counts.head(3))

58. 統計周天成在對手殺球之後，自己最常採用的前三種回擊球種。

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_cols = ['match_id', 'set', 'rally']

work = df.sort_values(group_cols + ['ball_round']).copy()
work['prev_player'] = work.groupby(group_cols)['player'].shift(1)
work['prev_type'] = work.groupby(group_cols)['type'].shift(1)

reply_df = work[
    (work['player'] == PLAYER) &
    (work['prev_player'] == OPP) &
    (work['prev_type'] == '殺球')
].copy()

shot_counts = reply_df['type'].value_counts().head(3)
print(shot_counts)

59. 比較周天成在長回合中，最後一拍前一拍常見的對手球種分布。

In [ ]:
import pandas as pd

OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立前一拍資訊
df_prev = df.copy()
df_prev['prev_player'] = df_prev.groupby(GROUP_COLS)['player'].shift(1)
df_prev['prev_type'] = df_prev.groupby(GROUP_COLS)['type'].shift(1)

# 2. 計算每個回合拍數，定義長回合
rally_lengths = (
    df.groupby(GROUP_COLS)['ball_round']
    .max()
    .reset_index(name='rally_length')
)

# 3. 找出回合最後一拍
final_shots = df_prev[df_prev['getpoint_player'].notna()].copy()
final_shots = final_shots.merge(rally_lengths, on=GROUP_COLS, how='left')

# 4. 只保留長回合，且最後一拍前一拍是對手出手的情況
long_rally_final = final_shots[
    (final_shots['rally_length'] >= 11) &
    (final_shots['prev_player'] == OPPONENT)
].copy()

# 5. 統計最後一拍前一拍常見的對手球種
prev_type_counts = long_rally_final['prev_type'].value_counts().reset_index()
prev_type_counts.columns = ['對手前一拍球種', '次數']
prev_type_counts['比例'] = prev_type_counts['次數'] / prev_type_counts['次數'].sum()

print("長回合中，最後一拍前一拍常見的對手球種分布：")
print(prev_type_counts)

60. 分析周天成在自己失分回合中，最後一拍前一拍最常使用的球種。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立前一拍資訊
df_prev = df.copy()
df_prev['prev_player'] = df_prev.groupby(GROUP_COLS)['player'].shift(1)
df_prev['prev_type'] = df_prev.groupby(GROUP_COLS)['type'].shift(1)

# 2. 找出回合最後一拍
final_shots = df_prev[df_prev['getpoint_player'].notna()].copy()

# 3. 只保留周天成失分的回合，且最後一拍前一拍是周天成出手
target_df = final_shots[
    (final_shots['getpoint_player'] != PLAYER) &
    (final_shots['prev_player'] == PLAYER)
].copy()

# 4. 統計最後一拍前一拍，周天成最常使用的球種
prev_type_counts = target_df['prev_type'].value_counts().reset_index()
prev_type_counts.columns = ['周天成前一拍球種', '次數']
prev_type_counts['比例'] = prev_type_counts['次數'] / prev_type_counts['次數'].sum()

print("在周天成失分回合中，最後一拍前一拍常見的周天成球種分布：")
print(prev_type_counts)

60. 分析周天成在自己失分回合中，最後一拍前一拍最常使用的球種。

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_cols = ['match_id', 'set', 'rally']

work = df.sort_values(group_cols + ['ball_round']).copy()

# 每回合最後一拍
rally_last = (
    work.groupby(group_cols, as_index=False)
    .tail(1)
)

# 周天成失分回合
lost_rallies = rally_last[rally_last['getpoint_player'] == OPP][group_cols + ['ball_round']].copy()
lost_rallies = lost_rallies.rename(columns={'ball_round': 'last_ball_round'})

merged = work.merge(lost_rallies, on=group_cols, how='inner')
prev_shots = merged[merged['ball_round'] == merged['last_ball_round'] - 1].copy()

# 題目主體要鎖周天成
prev_shots_ctc = prev_shots[prev_shots['player'] == PLAYER].copy()

shot_counts = prev_shots_ctc['type'].value_counts().reset_index()
shot_counts.columns = ['球種', '次數']

print(shot_counts)

61. 在被對手四角拉吊得回合下，周天成勝率是多少？四角拉吊定義:(同一個球員在場區[1, 4, 21, 24]內跑動，擊球點從其中一角跑到另一角)

In [ ]:
import pandas as pd

# 定義四角區域
corner_areas = [1, 4, 21, 24]

# 只看周天成自己的擊球資料
chou_df = df[df['player'] == 'CHOU Tien Chen'].copy()

# 在同一個 rally 中，找周天成前一次自己的擊球點
chou_df['prev_hit_area'] = chou_df.groupby(['match_id', 'set', 'rally'])['hit_area'].shift(1)

# 判斷是否出現從一個角落移動到另一個角落擊球
chou_df['corner_to_corner_move'] = (
    chou_df['hit_area'].isin(corner_areas) &
    chou_df['prev_hit_area'].isin(corner_areas) &
    (chou_df['hit_area'] != chou_df['prev_hit_area'])
)

# 找出有出現四角拉吊特徵的 rally
corner_move_rallies = chou_df.groupby(['match_id', 'set', 'rally'])['corner_to_corner_move'].any().reset_index()
corner_move_rallies = corner_move_rallies[corner_move_rallies['corner_to_corner_move']]

# 取得每個 rally 的最終得分者
rally_result = df.groupby(['match_id', 'set', 'rally']).agg(
    getpoint_player=('getpoint_player', 'last')
).reset_index()

# 合併並計算勝率
target_rallies = pd.merge(
    corner_move_rallies[['match_id', 'set', 'rally']],
    rally_result,
    on=['match_id', 'set', 'rally'],
    how='left'
)

total_rallies = len(target_rallies)
chou_win_rallies = (target_rallies['getpoint_player'] == 'CHOU Tien Chen').sum()
win_rate = chou_win_rallies / total_rallies if total_rallies > 0 else 0

print("被對手四角拉吊的回合數:", total_rallies)
print("其中周天成獲勝的回合數:", chou_win_rallies)
print(f"周天成在被對手四角拉吊回合下的勝率: {win_rate:.2%}")

62. 在被對手四角拉吊得回合下，周天成的失誤原因，繪製圓餅圖，四角拉吊定義:(同一個球員在場區[1, 4, 21, 24]內跑動，擊球點從其中一角跑到另一角)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 定義四角區域
corner_areas = [1, 4, 21, 24]

# 1. 只看周天成自己的擊球資料
chou_df = df[df['player'] == 'CHOU Tien Chen'].copy()

# 2. 在同一個 rally 中，找周天成前一次自己的擊球點
chou_df['prev_hit_area'] = chou_df.groupby(['match_id', 'set', 'rally'])['hit_area'].shift(1)

# 3. 判斷是否出現從一個角落移動到另一個角落擊球
chou_df['corner_to_corner_move'] = (
    chou_df['hit_area'].isin(corner_areas) &
    chou_df['prev_hit_area'].isin(corner_areas) &
    (chou_df['hit_area'] != chou_df['prev_hit_area'])
)

# 4. 找出有出現四角拉吊特徵的 rally
corner_move_rallies = chou_df.groupby(['match_id', 'set', 'rally'])['corner_to_corner_move'].any().reset_index()
corner_move_rallies = corner_move_rallies[corner_move_rallies['corner_to_corner_move']]

# 5. 取出這些 rally 中，周天成自己的失誤資料
target_errors = pd.merge(
    chou_df,
    corner_move_rallies[['match_id', 'set', 'rally']],
    on=['match_id', 'set', 'rally'],
    how='inner'
)

target_errors = target_errors[
    (target_errors['getpoint_player'] != 'CHOU Tien Chen') &
    (target_errors['lose_reason'].notna())
]

# 6. 統計失誤原因
lose_reason_counts = target_errors['lose_reason'].value_counts()

print("被對手四角拉吊的回合中，周天成失誤原因分布：")
print(lose_reason_counts)

# 7. 繪製圓餅圖
if len(lose_reason_counts) > 0:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.pie(
        lose_reason_counts,
        labels=lose_reason_counts.index,
        autopct='%1.1f%%',
        startangle=90
    )
    ax.set_title('周天成在被對手四角拉吊回合中的失誤原因分布')
    plt.tight_layout()
    plt.show()
else:
    print("沒有符合條件的失誤資料。")

63. 哪些球種組合是周天成最常使用的，反而讓對手主動攻擊？(可能是周天成連續打長球，讓對手抓時機殺球)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
ATTACK_TYPES = ['殺球', '推撲球', '平球']

# 在同一個 rally 內取下一拍，避免跨 rally 誤配。
ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
ordered_df['next_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(-1)
ordered_df['next_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(-1)

# 定義「周天成的球種 -> 對手下一拍主動攻擊球種」為讓對手主動攻擊的組合。
attack_after_chou = ordered_df[
    (ordered_df['player'] == PLAYER) &
    (ordered_df['next_player'] == OPPONENT) &
    (ordered_df['next_type'].isin(ATTACK_TYPES))
].copy()

attack_after_chou['combo'] = attack_after_chou['type'] + ' -> ' + attack_after_chou['next_type']
combo_counts = attack_after_chou['combo'].value_counts().reset_index()
combo_counts.columns = ['周天成球種 -> 對手攻擊球種', '次數']
combo_counts['比例'] = combo_counts['次數'] / combo_counts['次數'].sum()

print('周天成出球後，對手下一拍主動攻擊的常見球種組合：')
print(combo_counts.head(10))

# 另看周天成是否連續使用同類球種後被攻擊，例如連續長球後被殺球。
ordered_df['prev2_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(2)
ordered_df['prev2_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(2)
repeated_setup = ordered_df[
    (ordered_df['player'] == PLAYER) &
    (ordered_df['prev2_player'] == PLAYER) &
    (ordered_df['prev2_type'] == ordered_df['type']) &
    (ordered_df['next_player'] == OPPONENT) &
    (ordered_df['next_type'].isin(ATTACK_TYPES))
].copy()
repeated_setup['combo'] = repeated_setup['prev2_type'] + ' -> ' + repeated_setup['type'] + ' -> ' + repeated_setup['next_type']
print('\n周天成連續同球種後，對手主動攻擊的組合：')
print(repeated_setup['combo'].value_counts().head(10))

if not combo_counts.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df = combo_counts.head(10).sort_values('次數')
    ax.barh(plot_df['周天成球種 -> 對手攻擊球種'], plot_df['次數'], color='#E45756')
    ax.set_title('周天成出球後對手主動攻擊的常見組合')
    ax.set_xlabel('次數')
    ax.set_ylabel('球種組合')
    plt.tight_layout()

64. 在被對手四角拉吊時，周天成在前後場分別常用什麼球種回擊？四角拉吊定義:(同一個球員在場區[1, 4, 21, 24]內跑動，擊球點從其中一角跑到另一角)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']
CORNER_AREAS = [1, 4, 21, 24]

# 依 court_place.txt：後場 1~4，前場 17~24
FRONT_COURT = list(range(17, 25))
BACK_COURT = list(range(1, 5))

# 1. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 2. 在同一 rally 內，找周天成前一次自己的擊球點
chou_df['prev_chou_hit_area'] = (
    chou_df.groupby(GROUP_COLS)['hit_area']
    .shift(1)
)

# 3. 定義「被對手四角拉吊」：
#    周天成連續兩次自己的擊球點都在四角，且從一角移到另一角
corner_response_df = chou_df[
    chou_df['hit_area'].isin(CORNER_AREAS) &
    chou_df['prev_chou_hit_area'].notna() &
    chou_df['prev_chou_hit_area'].isin(CORNER_AREAS) &
    (chou_df['hit_area'] != chou_df['prev_chou_hit_area'])
].copy()

# 4. 將周天成當下的擊球點分成前場 / 後場
def classify_court_zone(area):
    if pd.isna(area):
        return None
    area = int(area)
    if area in FRONT_COURT:
        return '前場'
    if area in BACK_COURT:
        return '後場'
    return None

corner_response_df['court_zone'] = corner_response_df['hit_area'].apply(classify_court_zone)
corner_response_df = corner_response_df[corner_response_df['court_zone'].notna()].copy()

# 5. 統計前場 / 後場常用回擊球種
summary = (
    corner_response_df.groupby(['court_zone', 'type'])
    .size()
    .reset_index(name='count')
)

summary['ratio'] = (
    summary.groupby('court_zone')['count']
    .transform(lambda x: x / x.sum())
)

print("周天成在被對手四角拉吊時，前後場常用回擊球種：")
print(summary.sort_values(['court_zone', 'count'], ascending=[True, False]))

# 6. 視覺化
pivot_counts = summary.pivot(index='type', columns='court_zone', values='count').fillna(0)

if not pivot_counts.empty:
    ax = pivot_counts.plot(
        kind='bar',
        figsize=(10, 6)
    )
    ax.set_title('被對手四角拉吊時，周天成前/後場回擊球種分布')
    ax.set_xlabel('球種')
    ax.set_ylabel('次數')
    plt.tight_layout()
    plt.show()

65. 在周天成連續失分3分(含)以上時，他的打擊策略選擇是否變得激進或過於保守？

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONSERVATIVE_TYPES = ['長球', '挑球', '接殺防守', '網前球']

# 先在 rally 層級計算周天成在每個 rally 前的連續失分數。
rally_result = (
    df.sort_values(GROUP_COLS + ['ball_round'])
    .groupby(GROUP_COLS, as_index=False)
    .last()[GROUP_COLS + ['getpoint_player']]
)
rally_result['chou_scored'] = rally_result['getpoint_player'].eq(PLAYER)

streaks = []
loss_streak = 0
prev_key = None
for row in rally_result.itertuples(index=False):
    key = (row.match_id, row.set)
    if key != prev_key:
        loss_streak = 0
        prev_key = key
    streaks.append(loss_streak)
    loss_streak = 0 if row.chou_scored else loss_streak + 1
rally_result['pre_rally_loss_streak'] = streaks
rally_result['state'] = rally_result['pre_rally_loss_streak'].apply(lambda x: '連續失分3分以上後' if x >= 3 else '其他')

chou_shots = pd.merge(df[df['player'] == PLAYER].copy(), rally_result[GROUP_COLS + ['state', 'pre_rally_loss_streak']], on=GROUP_COLS, how='left')
chou_shots['style'] = chou_shots['type'].apply(
    lambda t: '激進' if t in ATTACK_TYPES else ('保守/控球' if t in CONSERVATIVE_TYPES else '其他')
)

summary = (
    chou_shots.groupby(['state', 'style'])
    .size()
    .reset_index(name='次數')
)
summary['比例'] = summary['次數'] / summary.groupby('state')['次數'].transform('sum')
shot_summary = (
    chou_shots.groupby(['state', 'type'])
    .size()
    .reset_index(name='次數')
)
shot_summary['比例'] = shot_summary['次數'] / shot_summary.groupby('state')['次數'].transform('sum')

print('周天成連續失分3分以上後的策略類型比例：')
print(summary)
print('\n球種比例：')
print(shot_summary.sort_values(['state', '次數'], ascending=[True, False]))

fig, ax = plt.subplots(figsize=(8, 5))
pivot = summary.pivot(index='style', columns='state', values='比例').fillna(0)
pivot.plot(kind='bar', ax=ax)
ax.set_title('連續失分3分以上後：周天成策略傾向')
ax.set_xlabel('策略類型')
ax.set_ylabel('比例')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()

66. 分析周天成在「領先 3 分以上」時的擊球策略是否更保守？請統計球種比例。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONSERVATIVE_TYPES = ['長球', '挑球', '接殺防守', '網前球']

chou_df = df[df['player'] == PLAYER].copy()
chou_df['score_diff'] = chou_df['CHOU Tien Chen_score'] - chou_df['Kento MOMOTA_score']
chou_df['score_state'] = chou_df['score_diff'].apply(lambda d: '領先3分以上' if d >= 3 else '非領先3分以上')
chou_df['style'] = chou_df['type'].apply(
    lambda t: '激進' if t in ATTACK_TYPES else ('保守/控球' if t in CONSERVATIVE_TYPES else '其他')
)

shot_summary = chou_df.groupby(['score_state', 'type']).size().reset_index(name='次數')
shot_summary['比例'] = shot_summary['次數'] / shot_summary.groupby('score_state')['次數'].transform('sum')
style_summary = chou_df.groupby(['score_state', 'style']).size().reset_index(name='次數')
style_summary['比例'] = style_summary['次數'] / style_summary.groupby('score_state')['次數'].transform('sum')

print('周天成領先3分以上時的球種比例：')
print(shot_summary.sort_values(['score_state', '次數'], ascending=[True, False]))
print('\n激進 vs 保守/控球比例：')
print(style_summary)

fig, ax = plt.subplots(figsize=(10, 5))
pivot = shot_summary.pivot(index='type', columns='score_state', values='比例').fillna(0)
pivot.plot(kind='bar', ax=ax)
ax.set_title('領先3分以上 vs 其他比分：周天成球種比例')
ax.set_xlabel('球種')
ax.set_ylabel('比例')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

66. 分析周天成在「領先 3 分以上」時的擊球策略是否更保守？請統計球種比例。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'

# 1. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 2. 定義是否領先 3 分以上
chou_df['score_diff'] = chou_df['CHOU Tien Chen_score'] - chou_df['Kento MOMOTA_score']
chou_df['state'] = chou_df['score_diff'].apply(
    lambda x: '領先3分以上' if x >= 3 else '其他情況'
)

# 3. 統計兩種情況下的球種分布
type_counts = chou_df.groupby(['state', 'type']).size().unstack(fill_value=0)
type_percentage = type_counts.div(type_counts.sum(axis=1), axis=0)

print("周天成在不同比分狀態下的球種分布：")
print(type_counts)

print("\n周天成在不同比分狀態下的球種比例：")
print(type_percentage)

67. 當分數差距逐漸縮小時（領先轉膠著），周天成殺球成功率是否下降？

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 以每個 rally 開始時的比分差判斷分數狀態。
rally_start = (
    df.sort_values(GROUP_COLS + ['ball_round'])
    .groupby(GROUP_COLS, as_index=False)
    .first()[GROUP_COLS + ['CHOU Tien Chen_score', 'Kento MOMOTA_score']]
)
rally_start['score_diff'] = rally_start['CHOU Tien Chen_score'] - rally_start['Kento MOMOTA_score']
rally_start = rally_start.sort_values(GROUP_COLS).copy()
rally_start['prev_score_diff'] = rally_start.groupby(['match_id', 'set'])['score_diff'].shift(1)

# 領先轉膠著：前一回合開始時領先 >=3，目前差距縮小到 0~2。
rally_start['score_state'] = '其他'
rally_start.loc[(rally_start['prev_score_diff'] >= 3) & (rally_start['score_diff'].between(0, 2)), 'score_state'] = '領先轉膠著'
rally_start.loc[(rally_start['score_diff'] >= 3) & (rally_start['score_state'] == '其他'), 'score_state'] = '穩定領先3分以上'

chou_smashes = df[(df['player'] == PLAYER) & (df['type'] == '殺球')].copy()
chou_smashes = pd.merge(chou_smashes, rally_start[GROUP_COLS + ['score_state', 'score_diff', 'prev_score_diff']], on=GROUP_COLS, how='left')
chou_smashes['is_direct_score'] = chou_smashes['getpoint_player'].eq(PLAYER)

summary = chou_smashes.groupby('score_state').agg(
    殺球次數=('type', 'size'),
    殺球直接得分數=('is_direct_score', 'sum'),
    殺球成功率=('is_direct_score', 'mean')
).reset_index()

print('比分差距縮小時，周天成殺球成功率比較：')
print(summary)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(summary['score_state'], summary['殺球成功率'], color='#B279A2')
ax.set_title('領先轉膠著時周天成殺球成功率')
ax.set_xlabel('比分狀態')
ax.set_ylabel('殺球成功率')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()

67. 當分數差距逐漸縮小時（領先轉膠著），周天成殺球成功率是否下降？

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'

ctc_df = df[df['player'] == PLAYER].copy()
ctc_df['score_diff'] = ctc_df[f'{PLAYER}_score'] - ctc_df[f'{OPP}_score']

# 分數狀態：用較明確口徑避免誤判
def score_state(x):
    if x >= 3:
        return '領先3分以上'
    elif -2 <= x <= 2:
        return '膠著'
    else:
        return '其他'

ctc_df['score_state'] = ctc_df['score_diff'].apply(score_state)

smash_df = ctc_df[ctc_df['type'] == '殺球'].copy()
smash_df['is_smash_active_win'] = (
    (smash_df['player'] == PLAYER) &
    (smash_df['getpoint_player'] == PLAYER)
)

summary = (
    smash_df[smash_df['score_state'].isin(['領先3分以上', '膠著'])]
    .groupby('score_state')
    .agg(
        smash_count=('type', 'size'),
        smash_active_win_count=('is_smash_active_win', 'sum')
    )
    .reset_index()
)

summary['smash_success_rate'] = (
    summary['smash_active_win_count'] / summary['smash_count']
)

print(summary)

68. 分析周天成在對手回球質量降低情況下（落點過中、回球過淺）的得分方式統計。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

FRONT_OR_SHALLOW = list(range(13, 25))
CENTER_AREAS = [6, 7, 10, 11, 14, 15, 18, 19, 22, 23]
POOR_RETURN_AREAS = sorted(set(FRONT_OR_SHALLOW + CENTER_AREAS))

ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
ordered_df['next_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(-1)
ordered_df['next_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(-1)
ordered_df['next_getpoint_player'] = ordered_df.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 對手回球質量降低：對手擊球的 landing_area 過淺或偏中，且下一拍由周天成處理。
poor_returns = ordered_df[
    (ordered_df['player'] == OPPONENT) &
    (ordered_df['landing_area'].isin(POOR_RETURN_AREAS)) &
    (ordered_df['next_player'] == PLAYER)
].copy()

chou_scoring_after_poor_return = poor_returns[poor_returns['next_getpoint_player'] == PLAYER].copy()
score_method_counts = chou_scoring_after_poor_return['next_type'].value_counts().reset_index()
score_method_counts.columns = ['周天成得分球種', '次數']
score_method_counts['比例'] = score_method_counts['次數'] / score_method_counts['次數'].sum()

print(f'對手低品質回球後，周天成下一拍處理次數: {len(poor_returns)}')
print(f'其中周天成下一拍直接得分次數: {len(chou_scoring_after_poor_return)}')
print('周天成得分方式統計：')
print(score_method_counts)

if not score_method_counts.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(score_method_counts['周天成得分球種'], score_method_counts['次數'], color='#FF9DA6')
    ax.set_title('對手低品質回球後，周天成得分方式')
    ax.set_xlabel('得分球種')
    ax.set_ylabel('次數')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

69. 當周天成接到「後場高遠球」時，他採取直線 vs 斜線的比例比較。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
BACK_COURT = [1, 2, 3, 4]
CLEAR_TYPES = ['長球', '發長球']
column_map = {
    1: 'A', 2: 'B', 3: 'C', 4: 'D',
    5: 'A', 6: 'B', 7: 'C', 8: 'D',
    9: 'A', 10: 'B', 11: 'C', 12: 'D',
    13: 'A', 14: 'B', 15: 'C', 16: 'D',
    17: 'A', 18: 'B', 19: 'C', 20: 'D',
    21: 'A', 22: 'B', 23: 'C', 24: 'D',
}

ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
ordered_df['next_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(-1)
ordered_df['next_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(-1)
ordered_df['next_landing_area'] = ordered_df.groupby(GROUP_COLS)['landing_area'].shift(-1)

# 周天成接到後場高遠球：對手打長球/發長球，落到後場，下一拍由周天成擊球。
received_clears = ordered_df[
    (ordered_df['player'] == OPPONENT) &
    (ordered_df['type'].isin(CLEAR_TYPES)) &
    (ordered_df['landing_area'].isin(BACK_COURT)) &
    (ordered_df['next_player'] == PLAYER)
].copy()

received_clears['incoming_column'] = received_clears['landing_area'].map(column_map)
received_clears['response_landing_column'] = received_clears['next_landing_area'].map(column_map)
received_clears = received_clears[
    received_clears['incoming_column'].notna() & received_clears['response_landing_column'].notna()
].copy()
received_clears['direction'] = received_clears.apply(
    lambda row: '直線' if row['incoming_column'] == row['response_landing_column'] else '斜線',
    axis=1
)

direction_counts = received_clears['direction'].value_counts().reset_index()
direction_counts.columns = ['方向', '次數']
direction_counts['比例'] = direction_counts['次數'] / direction_counts['次數'].sum()

print('周天成接到後場高遠球後，回球直線 vs 斜線比例：')
print(direction_counts)

fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(direction_counts['次數'], labels=direction_counts['方向'], autopct='%1.1f%%', startangle=90)
ax.set_title('接後場高遠球後：直線 vs 斜線')
ax.axis('equal')
plt.tight_layout()

70. 當周天成連續得分 3 分以上時，他的球種選擇是否變得更激進？

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONSERVATIVE_TYPES = ['長球', '挑球', '接殺防守', '網前球']

rally_result = (
    df.sort_values(GROUP_COLS + ['ball_round'])
    .groupby(GROUP_COLS, as_index=False)
    .last()[GROUP_COLS + ['getpoint_player']]
)
rally_result['chou_scored'] = rally_result['getpoint_player'].eq(PLAYER)

streaks = []
score_streak = 0
prev_key = None
for row in rally_result.itertuples(index=False):
    key = (row.match_id, row.set)
    if key != prev_key:
        score_streak = 0
        prev_key = key
    streaks.append(score_streak)
    score_streak = score_streak + 1 if row.chou_scored else 0
rally_result['pre_rally_score_streak'] = streaks
rally_result['state'] = rally_result['pre_rally_score_streak'].apply(lambda x: '連續得分3分以上後' if x >= 3 else '其他')

chou_shots = pd.merge(df[df['player'] == PLAYER].copy(), rally_result[GROUP_COLS + ['state', 'pre_rally_score_streak']], on=GROUP_COLS, how='left')
chou_shots['style'] = chou_shots['type'].apply(
    lambda t: '激進' if t in ATTACK_TYPES else ('保守/控球' if t in CONSERVATIVE_TYPES else '其他')
)

style_summary = chou_shots.groupby(['state', 'style']).size().reset_index(name='次數')
style_summary['比例'] = style_summary['次數'] / style_summary.groupby('state')['次數'].transform('sum')
shot_summary = chou_shots.groupby(['state', 'type']).size().reset_index(name='次數')
shot_summary['比例'] = shot_summary['次數'] / shot_summary.groupby('state')['次數'].transform('sum')

print('周天成連續得分3分以上後，球種策略是否更激進：')
print(style_summary)
print('\n球種比例：')
print(shot_summary.sort_values(['state', '次數'], ascending=[True, False]))

fig, ax = plt.subplots(figsize=(8, 5))
pivot = style_summary.pivot(index='style', columns='state', values='比例').fillna(0)
pivot.plot(kind='bar', ax=ax)
ax.set_title('連續得分3分以上後：周天成策略傾向')
ax.set_xlabel('策略類型')
ax.set_ylabel('比例')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()

71. 分析周天成在自己領先時與落後時，網前球的使用比例差異。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
PLAYER_SCORE_COL = 'CHOU Tien Chen_score'
OPPONENT_SCORE_COL = 'Kento MOMOTA_score'

# 1. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 2. 定義比分狀態
chou_df['score_state'] = chou_df.apply(
    lambda row: '領先' if row[PLAYER_SCORE_COL] > row[OPPONENT_SCORE_COL] else '落後或平手',
    axis=1
)

# 3. 定義網前球
chou_df['is_net_shot'] = chou_df['type'] == '網前球'

# 4. 統計不同比分狀態下的網前球使用比例
summary = chou_df.groupby('score_state').agg(
    總擊球數=('type', 'count'),
    網前球次數=('is_net_shot', 'sum')
).reset_index()

summary['網前球使用比例'] = summary['網前球次數'] / summary['總擊球數']

print("周天成在領先與落後/平手時的網前球使用比例：")
print(summary)

# 5. 視覺化
plt.figure(figsize=(8, 5))
plt.bar(
    summary['score_state'],
    summary['網前球使用比例'],
    color=['#4E79A7', '#E15759']
)

plt.title('周天成在不同比分狀態下的網前球使用比例')
plt.xlabel('比分狀態')
plt.ylabel('網前球使用比例')
plt.tight_layout()
plt.show()

72. 在網前互相推撥情況下，周天成的勝率是多少？

In [ ]:
import pandas as pd

# 1. 定義網前區域與網前推撥球種
front_court_areas = [17, 18, 19, 20, 21, 22, 23, 24]
net_exchange_types = ['推撲球', '網前球']

# 2. 在完整資料上建立前一拍資訊
df_prev = df.copy()
df_prev['prev_player'] = df_prev.groupby(['match_id', 'set', 'rally'])['player'].shift(1)
df_prev['prev_type'] = df_prev.groupby(['match_id', 'set', 'rally'])['type'].shift(1)
df_prev['prev_hit_area'] = df_prev.groupby(['match_id', 'set', 'rally'])['hit_area'].shift(1)

# 3. 找出「網前互相推撥」的 shot
net_exchange_df = df_prev[
    df_prev['prev_player'].notna() &
    (df_prev['hit_area'].isin(front_court_areas)) &
    (df_prev['prev_hit_area'].isin(front_court_areas)) &
    (df_prev['type'].isin(net_exchange_types)) &
    (df_prev['prev_type'].isin(net_exchange_types)) &
    (df_prev['player'] != df_prev['prev_player'])
].copy()

# 4. 找出有出現這種情況的 rally
target_rallies = net_exchange_df[['match_id', 'set', 'rally']].drop_duplicates()

# 5. 取得每個 rally 的最終得分者
rally_result = df.groupby(['match_id', 'set', 'rally']).agg(
    getpoint_player=('getpoint_player', 'last')
).reset_index()

target_result = pd.merge(
    target_rallies,
    rally_result,
    on=['match_id', 'set', 'rally'],
    how='left'
)

# 6. 計算周天成勝率
total_rallies = len(target_result)
chou_win_rallies = (target_result['getpoint_player'] == 'CHOU Tien Chen').sum()
win_rate = chou_win_rallies / total_rallies if total_rallies > 0 else 0

print(f"符合網前互相推撥情況的回合數: {total_rallies}")
print(f"其中周天成獲勝的回合數: {chou_win_rallies}")
print(f"周天成在網前互相推撥情況下的勝率: {win_rate:.2%}")

73. 周天成對直線 vs 斜線殺球的對手回擊效果（得失分比較）。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 定義每個區域所屬的欄位（A, B, C, D）
column_map = {
    1: 'A',  2: 'B',  3: 'C',  4: 'D',
    5: 'A',  6: 'B',  7: 'C',  8: 'D',
    9: 'A', 10: 'B', 11: 'C', 12: 'D',
    13: 'A', 14: 'B', 15: 'C', 16: 'D',
    17: 'A', 18: 'B', 19: 'C', 20: 'D',
    21: 'A', 22: 'B', 23: 'C', 24: 'D'
}

# 2. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)

# 3. 找出對手的殺球
opponent_smashes = df_next[
    (df_next['player'] == OPPONENT) &
    (df_next['type'] == '殺球')
].copy()

# 4. 判斷對手殺球是直線還是斜線
opponent_smashes['hit_column'] = opponent_smashes['hit_area'].map(column_map)
opponent_smashes['landing_column'] = opponent_smashes['landing_area'].map(column_map)

opponent_smashes = opponent_smashes[
    opponent_smashes['hit_column'].notna() &
    opponent_smashes['landing_column'].notna()
].copy()

opponent_smashes['smash_direction'] = opponent_smashes.apply(
    lambda row: '直線' if row['hit_column'] == row['landing_column'] else '斜線',
    axis=1
)

# 5. 只保留下一拍確實是周天成回擊的情況
target_df = opponent_smashes[
    opponent_smashes['next_player'] == PLAYER
].copy()

# 6. 取得每個 rally 的最終得分者
rally_result = (
    df.groupby(GROUP_COLS)
    .agg(rally_winner=('getpoint_player', 'last'))
    .reset_index()
)

target_df = pd.merge(
    target_df,
    rally_result,
    on=GROUP_COLS,
    how='left'
)

# 7. 用 rally 最終得分作為周天成回擊效果
target_df['return_result'] = target_df['rally_winner'].apply(
    lambda x: '得分' if x == PLAYER else '失分'
)

summary = (
    target_df.groupby(['smash_direction', 'return_result'])
    .size()
    .reset_index(name='count')
)

pivot_summary = summary.pivot(index='smash_direction', columns='return_result', values='count').fillna(0)

if '得分' not in pivot_summary.columns:
    pivot_summary['得分'] = 0
if '失分' not in pivot_summary.columns:
    pivot_summary['失分'] = 0

pivot_summary['總次數'] = pivot_summary['得分'] + pivot_summary['失分']
pivot_summary['得分率'] = pivot_summary['得分'] / pivot_summary['總次數']

print("面對對手直線 vs 斜線殺球時，周天成回擊效果比較：")
print(pivot_summary)

74. 分析周天成與對手在高壓球（平推、平抽）交換中誰先失誤的統計。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 依欄位定義，將「高壓球（平推、平抽）」對應為資料中的球種名稱
fast_exchange_types = ['推撲球', '平球']

# 1. 在完整資料上建立前一拍資訊
df_prev = df.copy()
df_prev['prev_player'] = df_prev.groupby(GROUP_COLS)['player'].shift(1)
df_prev['prev_type'] = df_prev.groupby(GROUP_COLS)['type'].shift(1)

# 2. 找出高壓球交換的 shot
fast_exchange_df = df_prev[
    df_prev['prev_player'].notna() &
    (df_prev['player'] != df_prev['prev_player']) &
    (df_prev['type'].isin(fast_exchange_types)) &
    (df_prev['prev_type'].isin(fast_exchange_types))
].copy()

# 3. 找出有發生高壓球交換的 rally
target_rallies = fast_exchange_df[GROUP_COLS].drop_duplicates()

# 4. 取得每個 rally 的最後一拍資訊
rally_last_shot = (
    df.groupby(GROUP_COLS)
    .last()
    .reset_index()
)

target_result = pd.merge(
    target_rallies,
    rally_last_shot[['match_id', 'set', 'rally', 'player', 'getpoint_player', 'lose_reason']],
    on=GROUP_COLS,
    how='left'
)

# 5. 判斷誰先失誤
target_result['error_player'] = target_result.apply(
    lambda row: row['player']
    if pd.notna(row['lose_reason']) and row['getpoint_player'] != row['player']
    else None,
    axis=1
)

error_summary = target_result['error_player'].value_counts().reset_index()
error_summary.columns = ['先失誤球員', '次數']

print("高壓球交換中誰先失誤的統計：")
print(error_summary)

# 6. 額外整理成周天成 vs 對手
chou_errors = (target_result['error_player'] == PLAYER).sum()
opponent_errors = (target_result['error_player'] == OPPONENT).sum()
valid_error_rallies = target_result['error_player'].notna().sum()

print(f"\n有明確失誤紀錄的高壓球交換回合數: {valid_error_rallies}")
print(f"周天成先失誤次數: {chou_errors}")
print(f"對手先失誤次數: {opponent_errors}")

75. 當周天成被迫後仰擊球時，他的失誤率 vs 得分率比較。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'

# 1. 篩選周天成「後仰擊球」資料
# 這裡以 aroundhead == 1 作為後仰擊球的代理定義
chou_aroundhead_df = df[
    (df['player'] == PLAYER) &
    (df['aroundhead'] == 1)
].copy()

# 2. 定義失誤與得分
# 失誤：這一拍由周天成擊球，且有 lose_reason，最後不是周天成得分
chou_aroundhead_df['is_error'] = (
    chou_aroundhead_df['lose_reason'].notna() &
    (chou_aroundhead_df['getpoint_player'] != PLAYER)
)

# 得分：周天成這一拍直接成為該回合得分者
chou_aroundhead_df['is_score'] = (
    chou_aroundhead_df['getpoint_player'] == PLAYER
)

# 3. 統計次數與比例
total_shots = len(chou_aroundhead_df)
error_count = chou_aroundhead_df['is_error'].sum()
score_count = chou_aroundhead_df['is_score'].sum()

error_rate = error_count / total_shots if total_shots > 0 else 0
score_rate = score_count / total_shots if total_shots > 0 else 0

print(f"周天成後仰擊球總次數: {total_shots}")
print(f"失誤次數: {error_count}")
print(f"得分次數: {score_count}")
print(f"失誤率: {error_rate:.4f}")
print(f"得分率: {score_rate:.4f}")

76. 在被動情況下（跨步救球），下一拍攻擊性球種的成功機率。

In [ ]:
import pandas as pd
import numpy as np

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立周天成「下一次自己的擊球」資訊
# 單打資料通常為雙方交替擊球，因此用 shift(-2) 取得同一位球員的下一拍
df_seq = df.copy()
df_seq['next2_player'] = df_seq.groupby(GROUP_COLS)['player'].shift(-2)
df_seq['next2_type'] = df_seq.groupby(GROUP_COLS)['type'].shift(-2)
df_seq['next2_getpoint_player'] = df_seq.groupby(GROUP_COLS)['getpoint_player'].shift(-2)
df_seq['next2_opponent'] = df_seq.groupby(GROUP_COLS)['opponent'].shift(-2)

# 2. 只看周天成的擊球，並計算移動距離
chou_df = df_seq[df_seq['player'] == PLAYER].copy()
chou_df['move_distance'] = np.sqrt(
    chou_df['player_move_x'] ** 2 + chou_df['player_move_y'] ** 2
)

# 3. 用移動距離前 25% 作為「被動情況（跨步救球）」的代理定義
move_threshold = chou_df['move_distance'].quantile(0.75)

passive_df = chou_df[
    chou_df['move_distance'] >= move_threshold
].copy()

# 4. 定義攻擊性球種
# 依目前欄位定義，保守使用較明確的攻擊球種
attack_types = ['殺球', '推撲球']

# 5. 找出周天成在被動救球後，下一次自己的攻擊性擊球
next_attack_df = passive_df[
    (passive_df['next2_player'] == PLAYER) &
    (passive_df['next2_type'].isin(attack_types))
].copy()

# 6. 定義攻擊性球種的成功
# 依欄位定義：該球未直接導致自己失分 => getpoint_player != opponent
next_attack_df['is_success'] = (
    next_attack_df['next2_getpoint_player'] != OPPONENT
)

# 7. 統計整體成功機率
total_attacks = len(next_attack_df)
success_count = next_attack_df['is_success'].sum()
success_rate = success_count / total_attacks if total_attacks > 0 else 0

print(f"周天成被動跨步救球代理條件（移動距離前25%）的門檻值: {move_threshold:.4f}")
print(f"被動情況下，下一次自己的攻擊性擊球總次數: {total_attacks}")
print(f"成功次數: {success_count}")
print(f"成功機率: {success_rate:.4f}")

# 8. 分球種統計
attack_summary = next_attack_df.groupby('next2_type').agg(
    使用次數=('next2_type', 'count'),
    成功次數=('is_success', 'sum')
).reset_index()

attack_summary['成功機率'] = attack_summary['成功次數'] / attack_summary['使用次數']

print("\n各攻擊性球種的成功機率：")
print(attack_summary)

76. 在被動情況下（跨步救球），下一拍攻擊性球種的成功機率。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

if len(df) == 0:
    print("No data available.")
else:
    df_analysis = df.copy()

    # 1. 在完整資料上建立同一位球員下一次自己的擊球資訊
    # 單打通常雙方交替擊球，因此用 shift(-2) 近似「自己下一拍」
    df_analysis['next2_player'] = df_analysis.groupby(GROUP_COLS)['player'].shift(-2)
    df_analysis['next2_type'] = df_analysis.groupby(GROUP_COLS)['type'].shift(-2)
    df_analysis['next2_getpoint_player'] = df_analysis.groupby(GROUP_COLS)['getpoint_player'].shift(-2)

    # 2. 只看周天成的被動防守球（以接殺防守作為代理）
    passive_defense = df_analysis[
        (df_analysis['player'] == PLAYER) &
        (df_analysis['type'] == '接殺防守')
    ].copy()

    # 3. 定義攻擊性球種
    attack_types = ['殺球', '推撲球']

    # 4. 找出被動防守後，下一次自己的攻擊性出手
    aggressive_attacks = passive_defense[
        (passive_defense['next2_player'] == PLAYER) &
        (passive_defense['next2_type'].isin(attack_types))
    ].copy()

    # 5. 定義成功：該攻擊球未直接導致自己失分
    aggressive_attacks['is_success'] = (
        aggressive_attacks['next2_getpoint_player'] != OPPONENT
    )

    # 6. 計算成功率
    total_attacks = len(aggressive_attacks)
    success_count = aggressive_attacks['is_success'].sum()
    success_rate = success_count / total_attacks if total_attacks > 0 else 0

    print(f"周天成以接殺防守作為被動情況的總次數: {len(passive_defense)}")
    print(f"其中下一次自己的出手屬於攻擊性球種的次數: {total_attacks}")
    print(f"成功次數: {success_count}")
    print(f"成功機率: {success_rate:.4f}")

    # 7. 分球種統計
    attack_summary = aggressive_attacks.groupby('next2_type').agg(
        使用次數=('next2_type', 'count'),
        成功次數=('is_success', 'sum')
    ).reset_index()

    attack_summary['成功機率'] = attack_summary['成功次數'] / attack_summary['使用次數']

    print("\n各攻擊性球種的成功機率：")
    print(attack_summary)

77. 分析周天成對不同對手攻擊方式差異。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)
df_next['next_lose_reason'] = df_next.groupby(GROUP_COLS)['lose_reason'].shift(-1)
df_next['next_getpoint_player'] = df_next.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 2. 找出對手的進攻 / 出手
opponent_shots_df = df_next[
    (df_next['player'] == OPPONENT) &
    (df_next['next_player'] == PLAYER)
].copy()

# 3. 定義周天成的應對結果
def classify_response(row):
    # 對手這一拍就直接得分
    if row['getpoint_player'] == OPPONENT:
        return '失敗應對（對手直接得分）'
    # 周天成下一拍有明確失誤，且該回合由對手得分
    elif pd.notna(row['next_lose_reason']) and row['next_getpoint_player'] == OPPONENT:
        return '失敗應對（回擊失誤）'
    # 其他情況視為先成功應對
    else:
        return '成功應對'

opponent_shots_df['response_result'] = opponent_shots_df.apply(classify_response, axis=1)

# 4. 統計不同對手出手球種下，周天成的回擊球種
response_summary = (
    opponent_shots_df.groupby(['type', 'next_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['type', 'count'], ascending=[True, False])
)

response_summary.columns = ['對手出手球種', '周天成回擊球種', '次數']

print("不同對手出手球種下，周天成的回擊球種分布：")
print(response_summary)

# 5. 統計不同對手出手球種下，周天成的應對結果
result_summary = (
    opponent_shots_df.groupby(['type', 'response_result'])
    .size()
    .reset_index(name='count')
)

result_summary.columns = ['對手出手球種', '應對結果', '次數']

print("\n不同對手出手球種下，周天成的應對結果：")
print(result_summary)

78. 計算周天成「放短球後被撲殺」的比例，以及是否在比賽後段上升。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 定義放短球類型
short_shot_types = ['發短球', '網前球']

# 2. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)
df_next['next_getpoint_player'] = df_next.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 3. 找出周天成的放短球
short_shot_df = df_next[
    (df_next['player'] == PLAYER) &
    (df_next['type'].isin(short_shot_types))
].copy()

# 4. 定義「放短球後被撲殺」
short_shot_df['killed_after_short_shot'] = (
    (short_shot_df['next_player'] == OPPONENT) &
    (short_shot_df['next_type'] == '推撲球') &
    (short_shot_df['next_getpoint_player'] == OPPONENT)
)

# 5. 計算整體比例
total_short_shots = len(short_shot_df)
killed_count = short_shot_df['killed_after_short_shot'].sum()
killed_rate = killed_count / total_short_shots if total_short_shots > 0 else 0

print(f"周天成放短球總次數: {total_short_shots}")
print(f"放短球後被撲殺次數: {killed_count}")
print(f"放短球後被撲殺比例: {killed_rate:.4f}")

# 6. 依放短球類型分開統計
type_summary = short_shot_df.groupby('type').agg(
    放短球次數=('type', 'count'),
    被撲殺次數=('killed_after_short_shot', 'sum')
).reset_index()

type_summary['被撲殺比例'] = (
    type_summary['被撲殺次數'] / type_summary['放短球次數']
)

print("\n不同放短球類型的被撲殺情況：")
print(type_summary)

# 7. 判斷是否在比賽後段上升
max_rally_in_set = df.groupby(['match_id', 'set'])['rally'].transform('max')
df_next['max_rally_in_set'] = max_rally_in_set

short_shot_df = df_next[
    (df_next['player'] == PLAYER) &
    (df_next['type'].isin(short_shot_types))
].copy()

short_shot_df['killed_after_short_shot'] = (
    (short_shot_df['next_player'] == OPPONENT) &
    (short_shot_df['next_type'] == '推撲球') &
    (short_shot_df['next_getpoint_player'] == OPPONENT)
)

short_shot_df['match_phase'] = short_shot_df.apply(
    lambda row: '後半段' if row['rally'] > row['max_rally_in_set'] / 2 else '前半段',
    axis=1
)

phase_summary = short_shot_df.groupby('match_phase').agg(
    放短球次數=('type', 'count'),
    被撲殺次數=('killed_after_short_shot', 'sum')
).reset_index()

phase_summary['被撲殺比例'] = (
    phase_summary['被撲殺次數'] / phase_summary['放短球次數']
)

print("\n前後半段比較：")
print(phase_summary)

78. 計算周天成「放短球後被撲殺」的比例，以及是否在比賽後段上升。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 定義放短球類型
short_shot_types = ['發短球', '網前球']

# 2. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)
df_next['next_getpoint_player'] = df_next.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 3. 找出周天成的放短球
short_shot_df = df_next[
    (df_next['player'] == PLAYER) &
    (df_next['type'].isin(short_shot_types))
].copy()

# 4. 定義「放短球後被殺球直接得分」
short_shot_df['killed_after_short_shot'] = (
    (short_shot_df['next_player'] == OPPONENT) &
    (short_shot_df['next_type'] == '殺球') &
    (short_shot_df['next_getpoint_player'] == OPPONENT)
)

# 5. 計算整體比例
total_short_shots = len(short_shot_df)
killed_count = short_shot_df['killed_after_short_shot'].sum()
killed_rate = killed_count / total_short_shots if total_short_shots > 0 else 0

print(f"周天成放短球總次數: {total_short_shots}")
print(f"放短球後被殺球直接得分次數: {killed_count}")
print(f"放短球後被殺球直接得分比例: {killed_rate:.4f}")

# 6. 依放短球類型分開統計
type_summary = short_shot_df.groupby('type').agg(
    放短球次數=('type', 'count'),
    被殺球直接得分次數=('killed_after_short_shot', 'sum')
).reset_index()

type_summary['被殺球直接得分比例'] = (
    type_summary['被殺球直接得分次數'] / type_summary['放短球次數']
)

print("\n不同放短球類型的被殺球情況：")
print(type_summary)

# 7. 判斷是否在比賽後段上升
max_rally_in_set = df.groupby(['match_id', 'set'])['rally'].transform('max')
df_next['max_rally_in_set'] = max_rally_in_set

short_shot_df = df_next[
    (df_next['player'] == PLAYER) &
    (df_next['type'].isin(short_shot_types))
].copy()

short_shot_df['killed_after_short_shot'] = (
    (short_shot_df['next_player'] == OPPONENT) &
    (short_shot_df['next_type'] == '殺球') &
    (short_shot_df['next_getpoint_player'] == OPPONENT)
)

short_shot_df['match_phase'] = short_shot_df.apply(
    lambda row: '後半段' if row['rally'] > row['max_rally_in_set'] / 2 else '前半段',
    axis=1
)

phase_summary = short_shot_df.groupby('match_phase').agg(
    放短球次數=('type', 'count'),
    被殺球直接得分次數=('killed_after_short_shot', 'sum')
).reset_index()

phase_summary['被殺球直接得分比例'] = (
    phase_summary['被殺球直接得分次數'] / phase_summary['放短球次數']
)

print("\n前後半段比較：")
print(phase_summary)

79. 計算對手挑球過短時，周天成球種的選擇統計。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 定義「挑球過短」的落點區域
# 這裡採用前中場 + 前場作為過短的判定
short_clear_areas = list(range(13, 25))  # 13~24

# 2. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)

# 3. 找出對手的挑球
opponent_clear_df = df_next[
    (df_next['player'] == OPPONENT) &
    (df_next['type'] == '挑球')
].copy()

# 4. 判斷是否為挑球過短
short_clear_df = opponent_clear_df[
    opponent_clear_df['landing_area'].isin(short_clear_areas)
].copy()

# 5. 只保留下一拍確實是周天成回擊的情況
target_df = short_clear_df[
    short_clear_df['next_player'] == PLAYER
].copy()

# 6. 統計周天成下一拍球種選擇
shot_counts = target_df['next_type'].value_counts().reset_index()
shot_counts.columns = ['球種', '次數']
shot_counts['比例'] = shot_counts['次數'] / shot_counts['次數'].sum()

print(f"對手挑球過短的次數: {len(short_clear_df)}")
print(f"其中周天成有下一拍回擊的次數: {len(target_df)}")

print("\n對手挑球過短時，周天成球種的選擇統計：")
print(shot_counts)

80. 當周天成發現自己較容易掛網時，應該如何調整自己的擊球策略與站位，以減少掛網失誤的發生？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


chou_df = df[df['player'] == PLAYER].copy()
chou_df['hit_depth'] = chou_df['hit_area'].apply(depth_label)
chou_df['stand_depth'] = chou_df['player_location_area'].apply(depth_label)
chou_df['is_net_error'] = chou_df['lose_reason'].eq('掛網') & chou_df['getpoint_player'].eq(OPPONENT)

risk_by_type = chou_df.groupby('type').agg(
    擊球數=('type', 'size'),
    掛網次數=('is_net_error', 'sum'),
    掛網率=('is_net_error', 'mean')
).reset_index().sort_values('掛網率', ascending=False)

risk_by_hit_depth = chou_df.groupby(['hit_depth', 'type']).agg(
    擊球數=('type', 'size'),
    掛網次數=('is_net_error', 'sum'),
    掛網率=('is_net_error', 'mean')
).reset_index().sort_values('掛網率', ascending=False)

net_errors = chou_df[chou_df['is_net_error']].copy()
print('周天成掛網失誤的球種風險：')
print(risk_by_type)
print('\n依擊球區域與球種的掛網風險：')
print(risk_by_hit_depth.head(12))
print('\n實際掛網失誤分布（球種、擊球區域、站位區域）：')
print(net_errors[['type', 'hit_area', 'hit_depth', 'player_location_area', 'stand_depth']].value_counts().head(15))
print('\n建議：降低高掛網率球種在高風險擊球區域的使用，改用更高容錯的挑深、長球或較高弧線網前處理。')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(risk_by_type['type'], risk_by_type['掛網率'], color='#E45756')
ax.set_title('周天成各球種掛網率')
ax.set_xlabel('球種')
ax.set_ylabel('掛網率')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

81. 周天成在被對手調動到四個角落後，他在哪些角落的「回球品質」最容易下降？下降的原因?

In [ ]:
# 1. 定義四個角落區域
corner_areas = [1, 4, 21, 24]
corner_name_map = {
    1: '後場左角',
    4: '後場右角',
    21: '前場左角',
    24: '前場右角'
}

# 2. 找出周天成被調動到四個角落後的回球
chou_corner_returns = df[
    (df['player'] == 'CHOU Tien Chen') &
    (df['hit_area'].isin(corner_areas))
].copy()

# 3. 保守定義「回球品質下降」：
#    只看周天成在這一拍直接失誤、導致對手得分的情況
chou_corner_returns['quality_drop'] = (
    (chou_corner_returns['getpoint_player'] != 'CHOU Tien Chen') &
    (chou_corner_returns['lose_reason'].notna())
)

chou_corner_returns['corner_name'] = chou_corner_returns['hit_area'].map(corner_name_map)

# 4. 計算各角落的回球品質下降比例
corner_summary = chou_corner_returns.groupby(['hit_area', 'corner_name']).agg(
    total_returns=('quality_drop', 'size'),
    quality_drop_count=('quality_drop', 'sum')
).reset_index()

corner_summary['quality_drop_rate'] = (
    corner_summary['quality_drop_count'] / corner_summary['total_returns']
)

corner_summary = corner_summary.sort_values('quality_drop_rate', ascending=False)

print("周天成在四個角落的回球品質下降情況：")
print(corner_summary[['corner_name', 'total_returns', 'quality_drop_count', 'quality_drop_rate']].round(4))

# 5. 統計各角落最常見的失誤原因
drop_reason_counts = (
    chou_corner_returns[chou_corner_returns['quality_drop']]
    .groupby(['corner_name', 'lose_reason'])
    .size()
    .reset_index(name='count')
)

top_reasons = (
    drop_reason_counts.sort_values(['corner_name', 'count'], ascending=[True, False])
    .groupby('corner_name')
    .head(3)
)

print("\n各角落最常見的回球品質下降原因（前 3 名）：")
print(top_reasons)

82. 統計周天成成功逼迫對手挑球的頻率，以及最常造成的球種來源。

In [ ]:
# 1. 在完整資料上建立前一拍資訊
df_prev = df.copy()
df_prev['prev_player'] = df_prev.groupby(['match_id', 'set', 'rally'])['player'].shift(1)
df_prev['prev_type'] = df_prev.groupby(['match_id', 'set', 'rally'])['type'].shift(1)

# 2. 找出對手（Kento MOMOTA）回挑球的情況
opponent_clear_df = df_prev[
    (df_prev['player'] == 'Kento MOMOTA') &
    (df_prev['type'] == '挑球')
].copy()

# 3. 只保留前一拍是周天成打出的情況
forced_clear_df = opponent_clear_df[
    opponent_clear_df['prev_player'] == 'CHOU Tien Chen'
].copy()

# 4. 計算周天成總擊球次數
chou_total_shots = df[df['player'] == 'CHOU Tien Chen'].shape[0]

# 5. 計算成功逼迫對手挑球的次數與頻率
forced_clear_count = len(forced_clear_df)
forced_clear_rate = forced_clear_count / chou_total_shots if chou_total_shots > 0 else 0

print(f"周天成成功逼迫對手挑球的次數: {forced_clear_count}")
print(f"周天成總擊球次數: {chou_total_shots}")
print(f"成功逼迫對手挑球的頻率: {forced_clear_rate:.4f}")

# 6. 統計最常造成對手挑球的周天成球種來源
source_type_counts = forced_clear_df['prev_type'].value_counts().reset_index()
source_type_counts.columns = ['球種來源', '次數']
source_type_counts['比例'] = source_type_counts['次數'] / source_type_counts['次數'].sum()

print("\n最常造成對手挑球的球種來源：")
print(source_type_counts)

83. 周天成挑高球的落點分布，是否常造成對手容易得分？

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)
df_next['next_getpoint_player'] = df_next.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 2. 找出周天成的挑球
chou_lift_df = df_next[
    (df_next['player'] == PLAYER) &
    (df_next['type'] == '挑球')
].copy()

# 3. 定義「容易讓對手得分」
# 這裡採保守定義：對手在下一拍直接得分
chou_lift_df['opponent_scores_next_shot'] = (
    (chou_lift_df['next_player'] == OPPONENT) &
    (chou_lift_df['next_getpoint_player'] == OPPONENT)
)

# 4. 統計整體情況
total_lifts = len(chou_lift_df)
punished_count = chou_lift_df['opponent_scores_next_shot'].sum()
punished_rate = punished_count / total_lifts if total_lifts > 0 else 0

print(f"周天成挑球總次數: {total_lifts}")
print(f"對手下一拍直接得分次數: {punished_count}")
print(f"對手下一拍直接得分比例: {punished_rate:.4f}")

# 5. 依落點區域分析
landing_summary = chou_lift_df.groupby('landing_area').agg(
    挑球次數=('landing_area', 'count'),
    對手下一拍直接得分次數=('opponent_scores_next_shot', 'sum')
).reset_index()

landing_summary['對手下一拍直接得分比例'] = (
    landing_summary['對手下一拍直接得分次數'] / landing_summary['挑球次數']
)

landing_summary = landing_summary.sort_values(
    '對手下一拍直接得分比例',
    ascending=False
)

print("\n不同落點區域的挑球與對手下一拍得分情況：")
print(landing_summary)

# 6. 視覺化落點分布
plt.figure(figsize=(8, 10))
sns.kdeplot(
    data=chou_lift_df,
    x='landing_x',
    y='landing_y',
    fill=True,
    cmap='Blues',
    levels=20,
    thresh=0.05
)

plt.title('周天成挑球落點分布')
plt.xlabel('landing_x')
plt.ylabel('landing_y')
plt.tight_layout()
plt.show()

84. 當周天成採取多拍耐心拉吊策略時的平均勝率。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 定義拉吊型球種
rallying_types = ['長球', '切球', '挑球']

# 2. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 3. 計算每個 rally 的總拍數
rally_length = (
    df.groupby(GROUP_COLS)['ball_round']
    .max()
    .reset_index(name='rally_length')
)

# 4. 計算每個 rally 中，周天成拉吊型球種的使用比例
chou_rally_stats = (
    chou_df.groupby(GROUP_COLS)
    .agg(
        chou_total_shots=('type', 'count'),
        rallying_shots=('type', lambda x: x.isin(rallying_types).sum())
    )
    .reset_index()
)

chou_rally_stats['rallying_ratio'] = (
    chou_rally_stats['rallying_shots'] / chou_rally_stats['chou_total_shots']
)

# 5. 合併 rally 拍數
strategy_df = pd.merge(
    chou_rally_stats,
    rally_length,
    on=GROUP_COLS,
    how='left'
)

# 6. 放寬定義「多拍耐心拉吊策略」
#    - 總拍數 >= 8
#    - 拉吊型球種比例 >= 0.3
strategy_rallies = strategy_df[
    (strategy_df['rally_length'] >= 8) &
    (strategy_df['rallying_ratio'] >= 0.3)
].copy()

# 7. 取得每個 rally 的最終得分者
rally_result = (
    df.groupby(GROUP_COLS)
    .agg(getpoint_player=('getpoint_player', 'last'))
    .reset_index()
)

strategy_rallies = pd.merge(
    strategy_rallies,
    rally_result,
    on=GROUP_COLS,
    how='left'
)

# 8. 計算勝率
total_rallies = len(strategy_rallies)
win_count = (strategy_rallies['getpoint_player'] == PLAYER).sum()
win_rate = win_count / total_rallies if total_rallies > 0 else 0

print(f"符合多拍耐心拉吊策略的回合數: {total_rallies}")
print(f"其中周天成獲勝回合數: {win_count}")
print(f"平均勝率: {win_rate:.4f}")

85. 統計周天成經常「從防守轉攻擊得分」的常見回合模式。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 定義防守與攻擊球種
defensive_types = ['接殺防守']
attacking_types = ['殺球', '推撲球']

# 2. 在完整資料上建立後續兩拍資訊
df_seq = df.copy()
df_seq['next_player'] = df_seq.groupby(GROUP_COLS)['player'].shift(-1)
df_seq['next_type'] = df_seq.groupby(GROUP_COLS)['type'].shift(-1)
df_seq['next2_player'] = df_seq.groupby(GROUP_COLS)['player'].shift(-2)
df_seq['next2_type'] = df_seq.groupby(GROUP_COLS)['type'].shift(-2)

# 3. 取得每個 rally 的最終得分者
rally_result = (
    df.groupby(GROUP_COLS)
    .agg(rally_winner=('getpoint_player', 'last'))
    .reset_index()
)

df_seq = pd.merge(df_seq, rally_result, on=GROUP_COLS, how='left')

# 4. 找出「從防守轉攻擊得分」的模式
#    - 周天成這一拍是防守球種
#    - 兩拍後仍是周天成，且已轉成攻擊球種
#    - 該 rally 最後由周天成得分
pattern_df = df_seq[
    (df_seq['player'] == PLAYER) &
    (df_seq['type'].isin(defensive_types)) &
    (df_seq['next2_player'] == PLAYER) &
    (df_seq['next2_type'].isin(attacking_types)) &
    (df_seq['rally_winner'] == PLAYER)
].copy()

# 5. 建立三拍模式字串
pattern_df['pattern'] = (
    pattern_df['type'] + ' -> ' +
    pattern_df['next_type'].fillna('未知') + ' -> ' +
    pattern_df['next2_type']
)

# 6. 統計常見模式
pattern_counts = pattern_df['pattern'].value_counts().reset_index()
pattern_counts.columns = ['回合模式', '次數']
pattern_counts['比例'] = pattern_counts['次數'] / pattern_counts['次數'].sum()

print("周天成從防守轉攻擊得分的常見回合模式：")
print(pattern_counts)

85. 統計周天成經常「從防守轉攻擊得分」的常見回合模式。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 定義對手的壓迫型球種與周天成的攻擊型球種
opponent_pressure_types = ['殺球', '推撲球']
attacking_types = ['殺球', '推撲球']

# 2. 在完整資料上建立前一拍與後續兩拍資訊
df_seq = df.copy()
df_seq['prev_player'] = df_seq.groupby(GROUP_COLS)['player'].shift(1)
df_seq['prev_type'] = df_seq.groupby(GROUP_COLS)['type'].shift(1)
df_seq['next_player'] = df_seq.groupby(GROUP_COLS)['player'].shift(-1)
df_seq['next_type'] = df_seq.groupby(GROUP_COLS)['type'].shift(-1)
df_seq['next2_player'] = df_seq.groupby(GROUP_COLS)['player'].shift(-2)
df_seq['next2_type'] = df_seq.groupby(GROUP_COLS)['type'].shift(-2)

# 3. 取得每個 rally 的最終得分者
rally_result = (
    df.groupby(GROUP_COLS)
    .agg(rally_winner=('getpoint_player', 'last'))
    .reset_index()
)

df_seq = pd.merge(df_seq, rally_result, on=GROUP_COLS, how='left')

# 4. 定義「防守轉攻擊得分」
#    - 周天成這一拍之前，對手前一拍是壓迫型球種
#    - 兩拍後周天成轉成攻擊型球種
#    - 該 rally 最後由周天成得分
pattern_df = df_seq[
    (df_seq['player'] == PLAYER) &
    (df_seq['prev_player'] == OPPONENT) &
    (df_seq['prev_type'].isin(opponent_pressure_types)) &
    (df_seq['next2_player'] == PLAYER) &
    (df_seq['next2_type'].isin(attacking_types)) &
    (df_seq['rally_winner'] == PLAYER)
].copy()

# 5. 建立模式字串
pattern_df['pattern'] = (
    pattern_df['prev_type'] + ' -> ' +
    pattern_df['type'] + ' -> ' +
    pattern_df['next_type'].fillna('未知') + ' -> ' +
    pattern_df['next2_type']
)

# 6. 統計常見模式
pattern_counts = pattern_df['pattern'].value_counts().reset_index()
pattern_counts.columns = ['回合模式', '次數']
pattern_counts['比例'] = pattern_counts['次數'] / pattern_counts['次數'].sum()

print("周天成從防守轉攻擊得分的常見回合模式（以前一球對手球種定義防守）：")
print(pattern_counts)

85. 統計周天成經常「從防守轉攻擊得分」的常見回合模式。

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_cols = ['match_id', 'set', 'rally']

work = df.sort_values(group_cols + ['ball_round']).copy()
work['prev_player'] = work.groupby(group_cols)['player'].shift(1)
work['prev_type'] = work.groupby(group_cols)['type'].shift(1)

defensive_types = ['接殺防守', '挑球', '長球']
attacking_types = ['殺球', '推撲球', '平球', '網前球', '切球']

# 防守轉攻擊得分：前一拍為對手，周天成此拍為攻擊球，且此拍主動得分
pattern_df = work[
    (work['player'] == PLAYER) &
    (work['prev_player'] == OPP) &
    (work['prev_type'].isin(defensive_types)) &
    (work['type'].isin(attacking_types)) &
    (work['getpoint_player'] == PLAYER)
].copy()

pattern_counts = (
    pattern_df.groupby(['prev_type', 'type'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

print(pattern_counts)

86. 分析周天成在壓迫對手前場後，選擇吊後場 vs 搶網得分的效率比較。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'

# 1. 定義前場與後場落點區域
front_court_areas = list(range(17, 25))  # 17~24
back_court_areas = list(range(1, 5))     # 1~4

# 2. 篩選周天成擊球，且對手已被壓在前場的情況
target_df = df[
    (df['player'] == PLAYER) &
    (df['opponent_location_area'].isin(front_court_areas))
].copy()

# 3. 只保留落點在前場或後場的球
target_df = target_df[
    target_df['landing_area'].isin(front_court_areas + back_court_areas)
].copy()

# 4. 用落點對應原題中的兩種選擇
def classify_strategy(area):
    if area in back_court_areas:
        return '吊後場'
    elif area in front_court_areas:
        return '前場處理（搶網方向）'
    else:
        return None

target_df['strategy_type'] = target_df['landing_area'].apply(classify_strategy)

# 5. 定義「該拍直接得分」
target_df['direct_score'] = (
    target_df['getpoint_player'] == PLAYER
)

# 6. 統計兩種策略的使用次數、直接得分次數與直接得分率
summary = target_df.groupby('strategy_type').agg(
    使用次數=('strategy_type', 'count'),
    直接得分次數=('direct_score', 'sum')
).reset_index()

summary['直接得分率'] = summary['直接得分次數'] / summary['使用次數']

print("當周天成壓迫對手前場後，不同選擇的直接得分效率比較：")
print(summary)

86. 分析周天成在壓迫對手前場後，選擇吊後場 vs 搶網得分的效率比較。

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'

# 1. 定義對手在前場的區域
front_court_areas = list(range(17, 25))  # 17~24

# 2. 定義兩種策略對應的球種
backcourt_strategy_types = ['長球', '挑球']
frontcourt_strategy_types = ['網前球', '推撲球']

# 3. 篩選周天成擊球，且對手已被壓在前場的情況
target_df = df[
    (df['player'] == PLAYER) &
    (df['opponent_location_area'].isin(front_court_areas))
].copy()

# 4. 將球種對應到原題中的兩種選擇
def classify_strategy(shot_type):
    if shot_type in backcourt_strategy_types:
        return '吊後場'
    elif shot_type in frontcourt_strategy_types:
        return '前場處理（搶網方向）'
    else:
        return None

target_df['strategy_type'] = target_df['type'].apply(classify_strategy)

# 5. 只保留這兩類策略
target_df = target_df[target_df['strategy_type'].notna()].copy()

# 6. 定義「該拍直接得分」
target_df['direct_score'] = (
    target_df['getpoint_player'] == PLAYER
)

# 7. 統計兩種策略的使用次數、直接得分次數與直接得分率
summary = target_df.groupby('strategy_type').agg(
    使用次數=('strategy_type', 'count'),
    直接得分次數=('direct_score', 'sum')
).reset_index()

summary['直接得分率'] = summary['直接得分次數'] / summary['使用次數']

print("當周天成壓迫對手前場後，不同策略選擇的直接得分效率比較：")
print(summary)

87. 根據對手的殺球，周天成應該如何布置接殺站位與防守策略？

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)
df_next['next_player_location_area'] = df_next.groupby(GROUP_COLS)['player_location_area'].shift(-1)
df_next['next_lose_reason'] = df_next.groupby(GROUP_COLS)['lose_reason'].shift(-1)
df_next['next_getpoint_player'] = df_next.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 2. 找出對手的殺球
opponent_smash_df = df_next[
    (df_next['player'] == OPPONENT) &
    (df_next['type'] == '殺球')
].copy()

# 3. 只保留：
#    - 對手這拍直接結束回合
#    - 或下一拍確實是周天成回擊
target_df = opponent_smash_df[
    (opponent_smash_df['getpoint_player'] == OPPONENT) |
    (opponent_smash_df['next_player'] == PLAYER) |
    (opponent_smash_df['getpoint_player'] == PLAYER)
].copy()

# 4. 定義結果分類
def classify_result(row):
    # 對手這拍直接得分
    if row['getpoint_player'] == OPPONENT:
        return '對手直接得分'
    # 對手自己這拍失誤，周天成直接得分
    elif row['getpoint_player'] == PLAYER and pd.notna(row['lose_reason']):
        return '對手直接失誤'
    # 周天成下一拍回擊失誤
    elif pd.notna(row['next_lose_reason']) and row['next_getpoint_player'] == OPPONENT:
        return '回擊失誤'
    # 其他情況視為至少成功回擊
    else:
        return '回擊成功'

target_df['result_type'] = target_df.apply(classify_result, axis=1)

# 5. 整理周天成接殺站位
position_summary = (
    target_df[target_df['next_player'] == PLAYER]
    .groupby(['next_player_location_area', 'result_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['next_player_location_area', 'count'], ascending=[True, False])
)

position_summary.columns = ['周天成接殺站位區域', '結果類型', '次數']

print("不同接殺站位區域下的結果分布：")
print(position_summary)

# 6. 整理周天成回擊球種
shot_summary = (
    target_df[target_df['next_player'] == PLAYER]
    .groupby(['next_type', 'result_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['next_type', 'count'], ascending=[True, False])
)

shot_summary.columns = ['周天成回擊球種', '結果類型', '次數']

print("\n不同回擊球種下的結果分布：")
print(shot_summary)

87. 根據對手的殺球，周天成應該如何布置接殺站位與防守策略？

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_lose_reason'] = df_next.groupby(GROUP_COLS)['lose_reason'].shift(-1)
df_next['next_getpoint_player'] = df_next.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 2. 找出對手的殺球
opponent_smash_df = df_next[
    (df_next['player'] == OPPONENT) &
    (df_next['type'] == '殺球')
].copy()

# 3. 定義結果分類
def classify_result(row):
    if row['getpoint_player'] == OPPONENT:
        return '對手直接得分'
    elif row['getpoint_player'] == PLAYER and pd.notna(row['lose_reason']):
        return '對手直接失誤'
    elif pd.notna(row['next_lose_reason']) and row['next_getpoint_player'] == OPPONENT:
        return '回擊失誤'
    elif row['next_player'] == PLAYER:
        return '回擊成功'
    else:
        return '其他'

opponent_smash_df['result_type'] = opponent_smash_df.apply(classify_result, axis=1)

# 4. 統計不同殺球落點下的結果分布
landing_result_summary = (
    opponent_smash_df.groupby(['landing_area', 'result_type'])
    .size()
    .reset_index(name='count')
)

print("不同對手殺球落點下，周天成的接殺結果分布：")
print(landing_result_summary)

# 5. 另外單獨計算「回球成功率」
landing_success_summary = opponent_smash_df.groupby('landing_area').agg(
    殺球次數=('landing_area', 'count'),
    回擊成功次數=('result_type', lambda x: (x == '回擊成功').sum())
).reset_index()

landing_success_summary['回球成功率'] = (
    landing_success_summary['回擊成功次數'] / landing_success_summary['殺球次數']
)

print("\n不同對手殺球落點下，周天成的回球成功率：")
print(landing_success_summary)

88. 對手若習慣在第三拍主動攻擊，周天成應該如何設計接發策略，在前三拍就破壞對手節奏並轉守為攻？

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 只取前三拍
first_three_df = df[df['ball_round'].isin([1, 2, 3])].copy()

# 2. 找出對手第三拍
opponent_third_shot_df = first_three_df[
    (first_three_df['ball_round'] == 3) &
    (first_three_df['player'] == OPPONENT)
].copy()

# 3. 統計對手第三拍常用球種
third_shot_type_counts = opponent_third_shot_df['type'].value_counts().reset_index()
third_shot_type_counts.columns = ['第三拍球種', '次數']
third_shot_type_counts['比例'] = third_shot_type_counts['次數'] / third_shot_type_counts['次數'].sum()

print("對手第三拍常用球種統計：")
print(third_shot_type_counts)

# 4. 統計對手第三拍落點區域
third_shot_landing_counts = opponent_third_shot_df['landing_area'].value_counts().reset_index()
third_shot_landing_counts.columns = ['第三拍落點區域', '次數']
third_shot_landing_counts['比例'] = third_shot_landing_counts['次數'] / third_shot_landing_counts['次數'].sum()

print("\n對手第三拍落點區域統計：")
print(third_shot_landing_counts)

88. 對手若習慣在第三拍主動攻擊，周天成應該如何設計接發策略，在前三拍就破壞對手節奏並轉守為攻？

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 在完整資料上建立前一拍資訊
df_prev = df.copy()
df_prev['prev_player'] = df_prev.groupby(GROUP_COLS)['player'].shift(1)
df_prev['prev_type'] = df_prev.groupby(GROUP_COLS)['type'].shift(1)

# 2. 找出對手第三拍
opponent_third_shot_df = df_prev[
    (df_prev['ball_round'] == 3) &
    (df_prev['player'] == OPPONENT)
].copy()

# 3. 只保留前一拍是周天成接發的情況
target_df = opponent_third_shot_df[
    (opponent_third_shot_df['prev_player'] == PLAYER)
].copy()

# 4. 統計「周天成第二拍球種 -> 對手第三拍球種」
pattern_counts = (
    target_df.groupby(['prev_type', 'type'])
    .size()
    .reset_index(name='count')
    .sort_values(['prev_type', 'count'], ascending=[True, False])
)

pattern_counts.columns = ['周天成第二拍球種', '對手第三拍球種', '次數']

print("周天成第二拍球種與對手第三拍球種的關係：")
print(pattern_counts)

89. 若對手在回合中主動進攻，周天成可以如何利用「節奏」與「落點深度」讓對手無法進入舒服的進攻狀態？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
ordered_df['next_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(-1)
ordered_df['next_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(-1)
ordered_df['next_landing_area'] = ordered_df.groupby(GROUP_COLS)['landing_area'].shift(-1)
ordered_df['next_landing_x'] = ordered_df.groupby(GROUP_COLS)['landing_x'].shift(-1)
ordered_df['next_landing_y'] = ordered_df.groupby(GROUP_COLS)['landing_y'].shift(-1)

rally_winner = ordered_df.groupby(GROUP_COLS, as_index=False).last()[GROUP_COLS + ['getpoint_player']].rename(columns={'getpoint_player': 'rally_winner'})
opponent_attacks = ordered_df[
    (ordered_df['player'] == OPPONENT) &
    (ordered_df['type'].isin(ATTACK_TYPES)) &
    (ordered_df['next_player'] == PLAYER)
].copy()
opponent_attacks = pd.merge(opponent_attacks, rally_winner, on=GROUP_COLS, how='left')
opponent_attacks['chou_response_depth'] = opponent_attacks['next_landing_area'].apply(depth_label)
opponent_attacks['chou_won_rally'] = opponent_attacks['rally_winner'].eq(PLAYER)

response_summary = opponent_attacks.groupby(['next_type', 'chou_response_depth']).agg(
    回應次數=('next_type', 'size'),
    周天成得分回合數=('chou_won_rally', 'sum'),
    回合得分率=('chou_won_rally', 'mean')
).reset_index().sort_values(['回合得分率', '回應次數'], ascending=[False, False])

depth_summary = opponent_attacks.groupby('chou_response_depth').agg(
    回應次數=('next_type', 'size'),
    回合得分率=('chou_won_rally', 'mean')
).reset_index().sort_values('回合得分率', ascending=False)

print('對手主動進攻後，周天成用球種與落點深度打斷節奏的效果：')
print(response_summary)
print('\n依落點深度彙總：')
print(depth_summary)
print('\n建議：優先選擇回合得分率較高且樣本數足夠的回應方式，並用前後場深度變化讓對手無法連續進攻。')

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = response_summary.head(10).copy()
plot_df['策略'] = plot_df['next_type'] + '到' + plot_df['chou_response_depth']
ax.barh(plot_df['策略'][::-1], plot_df['回合得分率'][::-1], color='#4C78A8')
ax.set_title('對手進攻後，周天成回應策略得分率')
ax.set_xlabel('回合得分率')
plt.tight_layout()

89. 若對手在回合中主動進攻，周天成可以如何利用「節奏」與「落點深度」讓對手無法進入舒服的進攻狀態？

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_cols = ['match_id', 'set', 'rally']

work = df.sort_values(group_cols + ['ball_round']).copy()
work['next_player'] = work.groupby(group_cols)['player'].shift(-1)
work['next_type'] = work.groupby(group_cols)['type'].shift(-1)

aggressive_types = ['殺球', '平球', '推撲球']
front_zones = list(range(17, 25))
mid_zones = list(range(5, 17))
back_zones = list(range(1, 5))

def zone_cat(area):
    if pd.isna(area):
        return '未知'
    area = int(area)
    if area in front_zones:
        return '前場'
    if area in mid_zones:
        return '中場'
    if area in back_zones:
        return '後場'
    return '其他'

# 對手主動進攻前一拍，由周天成打出
ctc_before_opp_attack = work[
    (work['player'] == PLAYER) &
    (work['next_player'] == OPP) &
    (work['next_type'].isin(aggressive_types))
].copy()

ctc_before_opp_attack['landing_depth'] = ctc_before_opp_attack['landing_area'].apply(zone_cat)

summary = (
    ctc_before_opp_attack.groupby(['type', 'landing_depth'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

print(summary)

90. 在對手打網前球很精準時，周天成應如何改變網前球的球質與連接方式，減少對手打網前球的頻率？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
ordered_df['next_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(-1)
ordered_df['next_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(-1)
ordered_df['next_landing_area'] = ordered_df.groupby(GROUP_COLS)['landing_area'].shift(-1)
ordered_df['next_getpoint_player'] = ordered_df.groupby(GROUP_COLS)['getpoint_player'].shift(-1)
ordered_df['prev_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(1)
ordered_df['prev_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(1)

# A. 對手打網前球後，周天成如何回應。
opponent_net = ordered_df[
    (ordered_df['player'] == OPPONENT) &
    (ordered_df['type'] == '網前球') &
    (ordered_df['next_player'] == PLAYER)
].copy()
opponent_net['response_depth'] = opponent_net['next_landing_area'].apply(depth_label)
opponent_net['response_direct_score'] = opponent_net['next_getpoint_player'].eq(PLAYER)
response_summary = opponent_net.groupby(['next_type', 'response_depth']).agg(
    次數=('next_type', 'size'),
    下一拍直接得分率=('response_direct_score', 'mean')
).reset_index().sort_values(['下一拍直接得分率', '次數'], ascending=[False, False])

# B. 周天成哪些前一拍容易讓對手下一拍打網前球。
risk_df = ordered_df[
    (ordered_df['player'] == PLAYER) &
    (ordered_df['next_player'] == OPPONENT)
].copy()
risk_df['opponent_next_net'] = risk_df['next_type'].eq('網前球')
risk_summary = risk_df.groupby('type').agg(
    周天成出球次數=('type', 'size'),
    對手下一拍網前球次數=('opponent_next_net', 'sum'),
    對手下一拍網前球率=('opponent_next_net', 'mean')
).reset_index().sort_values('對手下一拍網前球率', ascending=False)

print('對手網前球後，周天成回應方式與直接得分率：')
print(response_summary)
print('\n周天成哪些球種較容易讓對手下一拍打網前球：')
print(risk_summary)
print('\n建議：減少高風險球種的單調銜接，優先用回應成效較好的推撲、挑深或快速轉後場，降低對手連續網前球機會。')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
response_summary.head(8).plot(x='next_type', y='下一拍直接得分率', kind='bar', ax=axes[0], legend=False, color='#54A24B')
axes[0].set_title('對手網前後：周天成回應得分率')
risk_summary.plot(x='type', y='對手下一拍網前球率', kind='bar', ax=axes[1], legend=False, color='#F58518')
axes[1].set_title('周天成出球後對手網前球率')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

91. 若對手的右後場出球不穩，周天成應如何調整策略，攻擊其弱側？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


RIGHT_BACK_AREAS = [4]
RIGHT_BACK_ADJACENT = [4, 8, 12, 16, 20, 24]
chou_df = df[df['player'] == PLAYER].copy()
chou_df['is_direct_score'] = chou_df['getpoint_player'].eq(PLAYER)

right_back_attack = chou_df[chou_df['landing_area'].isin(RIGHT_BACK_AREAS + RIGHT_BACK_ADJACENT)].copy()
right_back_attack['target_zone'] = right_back_attack['landing_area'].apply(
    lambda a: '右後場核心(4)' if a in RIGHT_BACK_AREAS else '右側延伸/相鄰區'
)
summary = right_back_attack.groupby(['target_zone', 'landing_area', 'type']).agg(
    使用次數=('type', 'size'),
    直接得分數=('is_direct_score', 'sum'),
    直接得分率=('is_direct_score', 'mean')
).reset_index().sort_values(['直接得分率', '使用次數'], ascending=[False, False])

type_summary = right_back_attack.groupby('type').agg(
    使用次數=('type', 'size'),
    直接得分率=('is_direct_score', 'mean')
).reset_index().sort_values('直接得分率', ascending=False)

print('攻擊對手右後場/右側深區的球種與效率：')
print(summary)
print('\n依球種彙總：')
print(type_summary)
print('\n建議：若右後場不穩，可用長球/切球/殺球落到右後場核心與右側延伸區，並用前後變化逼出短球或失誤。')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(type_summary['type'], type_summary['直接得分率'], color='#72B7B2')
ax.set_title('攻擊右後場/右側深區的直接得分率')
ax.set_xlabel('球種')
ax.set_ylabel('直接得分率')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

91. 若對手的右後場出球不穩，周天成應如何調整策略，攻擊其弱側？

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_cols = ['match_id', 'set', 'rally']

# 對手右後場：依目前場地口徑用 3,4
right_back_zones = [3, 4]

work = df.sort_values(group_cols + ['ball_round']).copy()
work['is_last_shot'] = (
    work.groupby(group_cols)['ball_round'].transform('max') == work['ball_round']
)

# 周天成把球打到對手右後場
attack_df = work[
    (work['player'] == PLAYER) &
    (work['landing_area'].isin(right_back_zones))
].copy()

attack_df['direct_active_win'] = (
    attack_df['is_last_shot'] &
    (attack_df['getpoint_player'] == PLAYER)
)

# 整回合最終是否由周天成得分
rally_last = (
    work.groupby(group_cols, as_index=False)
    .last()[group_cols + ['getpoint_player']]
)
attack_df = attack_df.merge(rally_last, on=group_cols, how='left', suffixes=('', '_rally_last'))
attack_df['rally_win'] = attack_df['getpoint_player_rally_last'] == PLAYER

summary = (
    attack_df.groupby('type')
    .agg(
        count=('type', 'size'),
        direct_active_win_rate=('direct_active_win', 'mean'),
        rally_win_rate=('rally_win', 'mean')
    )
    .reset_index()
    .sort_values('count', ascending=False)
)

print(summary)

92. 周天成應如何透過短拍數進攻、節奏加速策略避免進入長回合？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


rally_info = df.groupby(GROUP_COLS, as_index=False).agg(rally_length=('ball_round', 'max'))
rally_result = df.sort_values(GROUP_COLS + ['ball_round']).groupby(GROUP_COLS, as_index=False).last()[GROUP_COLS + ['getpoint_player']]
rally_info = pd.merge(rally_info, rally_result, on=GROUP_COLS, how='left')
rally_info['length_group'] = pd.cut(rally_info['rally_length'], bins=[0, 4, 10, float('inf')], labels=['短拍數(<=4)', '中拍數(5-10)', '長回合(>=11)'])

short_win_rallies = rally_info[(rally_info['length_group'].astype(str) == '短拍數(<=4)') & (rally_info['getpoint_player'] == PLAYER)][GROUP_COLS]
early_chou = pd.merge(df[(df['player'] == PLAYER) & (df['ball_round'] <= 4)].copy(), short_win_rallies, on=GROUP_COLS, how='inner')
early_chou['is_direct_score'] = early_chou['getpoint_player'].eq(PLAYER)

shot_summary = early_chou.groupby(['ball_round', 'type']).agg(
    使用次數=('type', 'size'),
    直接得分率=('is_direct_score', 'mean')
).reset_index().sort_values(['ball_round', '使用次數'], ascending=[True, False])

length_summary = rally_info.groupby('length_group', observed=True).agg(
    回合數=('rally_length', 'size'),
    周天成得分率=('getpoint_player', lambda s: (s == PLAYER).mean())
).reset_index()

print('不同回合長度下周天成得分率：')
print(length_summary)
print('\n周天成在短拍數得分回合中的前四拍球種：')
print(shot_summary)
print('\n建議：優先複製短拍數得分回合中常見的前四拍球種與攻擊節奏，減少進入長回合。')

fig, ax = plt.subplots(figsize=(9, 4))
length_summary.plot(x='length_group', y='周天成得分率', kind='bar', ax=ax, legend=False, color='#4C78A8')
ax.set_title('回合長度與周天成得分率')
ax.set_xlabel('回合長度')
ax.set_ylabel('得分率')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()

93. 當對手在關鍵分(18分以上)皆呈現非進攻狀態，周天成應如何調整球路設計來強化關鍵分得分率？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
ordered_df['next_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(-1)
ordered_df['next_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(-1)
ordered_df['next_landing_area'] = ordered_df.groupby(GROUP_COLS)['landing_area'].shift(-1)
ordered_df['next_getpoint_player'] = ordered_df.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

critical_non_attack = ordered_df[
    (ordered_df['player'] == OPPONENT) &
    ((ordered_df['CHOU Tien Chen_score'] >= 18) | (ordered_df['Kento MOMOTA_score'] >= 18)) &
    (~ordered_df['type'].isin(ATTACK_TYPES)) &
    (ordered_df['next_player'] == PLAYER)
].copy()
critical_non_attack['chou_response_depth'] = critical_non_attack['next_landing_area'].apply(depth_label)
critical_non_attack['chou_direct_score'] = critical_non_attack['next_getpoint_player'].eq(PLAYER)

summary = critical_non_attack.groupby(['next_type', 'chou_response_depth']).agg(
    回應次數=('next_type', 'size'),
    下一拍直接得分率=('chou_direct_score', 'mean')
).reset_index().sort_values(['下一拍直接得分率', '回應次數'], ascending=[False, False])

print('關鍵分時，對手非進攻狀態後周天成的回應效率：')
print(summary)
print('\n建議：對手關鍵分非進攻時，優先使用直接得分率較高的回應球種與落點深度，主動壓迫而不是等待失誤。')

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = summary.head(10).copy()
plot_df['策略'] = plot_df['next_type'] + '到' + plot_df['chou_response_depth']
ax.barh(plot_df['策略'][::-1], plot_df['下一拍直接得分率'][::-1], color='#54A24B')
ax.set_title('關鍵分對手非進攻後：周天成回應得分率')
ax.set_xlabel('下一拍直接得分率')
plt.tight_layout()

94. 當周天成連續失分三次以上時，應如何透過節奏調整與高成功率球路重建比賽掌控感？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


rally_result = df.sort_values(GROUP_COLS + ['ball_round']).groupby(GROUP_COLS, as_index=False).last()[GROUP_COLS + ['getpoint_player']]
rally_result['chou_scored'] = rally_result['getpoint_player'].eq(PLAYER)

streaks = []
loss_streak = 0
prev_key = None
for row in rally_result.itertuples(index=False):
    key = (row.match_id, row.set)
    if key != prev_key:
        loss_streak = 0
        prev_key = key
    streaks.append(loss_streak)
    loss_streak = 0 if row.chou_scored else loss_streak + 1
rally_result['pre_loss_streak'] = streaks
rally_result['state'] = np.where(rally_result['pre_loss_streak'] >= 3, '連續失分三次以上後', '其他')

chou_shots = pd.merge(df[df['player'] == PLAYER].copy(), rally_result[GROUP_COLS + ['state', 'pre_loss_streak']], on=GROUP_COLS, how='left')
chou_shots['is_direct_score'] = chou_shots['getpoint_player'].eq(PLAYER)
chou_shots['is_error'] = chou_shots['lose_reason'].notna() & chou_shots['getpoint_player'].eq(OPPONENT)

summary = chou_shots.groupby(['state', 'type']).agg(
    使用次數=('type', 'size'),
    直接得分率=('is_direct_score', 'mean'),
    失誤率=('is_error', 'mean')
).reset_index().sort_values(['state', '直接得分率', '使用次數'], ascending=[True, False, False])
rebuild_choices = summary[(summary['state'] == '連續失分三次以上後') & (summary['使用次數'] >= 3)].sort_values(['直接得分率', '失誤率'], ascending=[False, True])

print('連續失分三次以上後，周天成各球種效果：')
print(summary)
print('\n可作為重建掌控感的高成功率/低失誤選擇：')
print(rebuild_choices.head(8))
print('\n建議：失分潮後先用高成功率且低失誤的球種穩住，再逐步加入攻擊型球種。')

fig, ax = plt.subplots(figsize=(9, 4))
plot_df = rebuild_choices.head(8)
ax.bar(plot_df['type'], plot_df['直接得分率'], color='#59A14F')
ax.set_title('連續失分後可重建節奏的球種')
ax.set_xlabel('球種')
ax.set_ylabel('直接得分率')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

95. 比賽中若觀察到對手因壓力無法耐心打多拍，周天成可如何透過中場拉吊、穩定球路誘發對手躁進失誤？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
ordered_df['prev_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(1)
ordered_df['prev_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(1)
ordered_df['prev_landing_area'] = ordered_df.groupby(GROUP_COLS)['landing_area'].shift(1)
ordered_df['prev_landing_depth'] = ordered_df['prev_landing_area'].apply(depth_label)
rally_length = ordered_df.groupby(GROUP_COLS, as_index=False).agg(rally_length=('ball_round', 'max'))
ordered_df = pd.merge(ordered_df, rally_length, on=GROUP_COLS, how='left')

# 對手因壓力躁進失誤：對手自己失誤且周天成得分。
opponent_errors = ordered_df[
    (ordered_df['player'] == OPPONENT) &
    (ordered_df['lose_reason'].notna()) &
    (ordered_df['getpoint_player'] == PLAYER)
].copy()
opponent_errors['rally_length_group'] = pd.cut(opponent_errors['rally_length'], bins=[0, 4, 10, float('inf')], labels=['短', '中', '長'])

cause_summary = opponent_errors.groupby(['prev_type', 'prev_landing_depth', 'rally_length_group'], observed=True).agg(
    造成對手失誤次數=('type', 'size')
).reset_index().sort_values('造成對手失誤次數', ascending=False)

mid_control = opponent_errors[
    (opponent_errors['prev_landing_depth'] == '中場') &
    (opponent_errors['prev_type'].isin(CONTROL_TYPES))
]

print('周天成前一拍造成對手失誤的球種/落點深度/回合長度：')
print(cause_summary.head(15))
print(f"\n中場拉吊/穩定控球造成對手失誤次數: {len(mid_control)}")
print('\n建議：若對手無法耐心多拍，使用中場或前後場控球銜接，讓對手在中長回合中先失誤。')

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = cause_summary.head(10).copy()
plot_df['模式'] = plot_df['prev_type'].astype(str) + '到' + plot_df['prev_landing_depth'].astype(str) + '/' + plot_df['rally_length_group'].astype(str)
ax.barh(plot_df['模式'][::-1], plot_df['造成對手失誤次數'][::-1], color='#F28E2B')
ax.set_title('造成對手躁進失誤的前一拍模式')
ax.set_xlabel('次數')
plt.tight_layout()

95. 比賽中若觀察到對手因壓力無法耐心打多拍，周天成可如何透過中場拉吊、穩定球路誘發對手躁進失誤？

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_cols = ['match_id', 'set', 'rally']

work = df.sort_values(group_cols + ['ball_round']).copy()
work['next_player'] = work.groupby(group_cols)['player'].shift(-1)
work['next_type'] = work.groupby(group_cols)['type'].shift(-1)
work['rally_last_getpoint_player'] = work.groupby(group_cols)['getpoint_player'].transform('last')
work['rally_last_lose_reason'] = work.groupby(group_cols)['lose_reason'].transform('last')

mid_areas = list(range(5, 17))
stable_types = ['長球', '挑球', '平球', '網前球']
aggressive_types = ['殺球', '推撲球', '平球', '網前球']
error_reasons = ['出界', '掛網', '未過網', '犯規', '落點判斷失誤']

pattern_df = work[
    (work['player'] == PLAYER) &
    (work['hit_area'].isin(mid_areas)) &
    (work['type'].isin(stable_types)) &
    (work['next_player'] == OPP) &
    (work['next_type'].isin(aggressive_types))
].copy()

pattern_df['opponent_error_end'] = (
    (pattern_df['rally_last_getpoint_player'] == PLAYER) &
    (pattern_df['rally_last_lose_reason'].isin(error_reasons))
)

summary = (
    pattern_df.groupby('type')
    .agg(
        count=('type', 'size'),
        opponent_error_end_rate=('opponent_error_end', 'mean')
    )
    .reset_index()
    .sort_values('count', ascending=False)
)

print(summary)

96. 如果周天成發現對手能抓到他所有直線殺球，他應該如何調整殺球角度與變化？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


column_map = {
    1:'A',2:'B',3:'C',4:'D',5:'A',6:'B',7:'C',8:'D',9:'A',10:'B',11:'C',12:'D',
    13:'A',14:'B',15:'C',16:'D',17:'A',18:'B',19:'C',20:'D',21:'A',22:'B',23:'C',24:'D'
}
ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
ordered_df['next_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(-1)
ordered_df['next_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(-1)

smashes = ordered_df[(ordered_df['player'] == PLAYER) & (ordered_df['type'] == '殺球')].copy()
smashes['hit_col'] = smashes['hit_area'].map(column_map)
smashes['landing_col'] = smashes['landing_area'].map(column_map)
smashes = smashes[smashes['hit_col'].notna() & smashes['landing_col'].notna()].copy()
smashes['direction'] = np.where(smashes['hit_col'] == smashes['landing_col'], '直線殺球', '斜線/變線殺球')
smashes['direct_score'] = smashes['getpoint_player'].eq(PLAYER)
smashes['opponent_replied'] = smashes['next_player'].eq(OPPONENT)

summary = smashes.groupby('direction').agg(
    殺球次數=('type', 'size'),
    直接得分率=('direct_score', 'mean'),
    對手接回率=('opponent_replied', 'mean')
).reset_index()
reply_summary = smashes[smashes['opponent_replied']].groupby(['direction', 'next_type']).size().reset_index(name='對手回擊次數').sort_values(['direction', '對手回擊次數'], ascending=[True, False])

print('周天成直線 vs 斜線/變線殺球效果：')
print(summary)
print('\n對手接回後的回擊球種分布：')
print(reply_summary)
print('\n建議：若直線殺球被抓，降低直線比例，增加斜線/變線殺球與落點深度變化。')

fig, ax = plt.subplots(figsize=(7, 4))
summary.plot(x='direction', y=['直接得分率', '對手接回率'], kind='bar', ax=ax)
ax.set_title('直線 vs 斜線/變線殺球效果')
ax.set_xlabel('殺球方向')
ax.set_ylabel('比例')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()

97. 如果周天成在網前打點一直遭對手抓撲殺，他可以怎麼改變接網方式？

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']
FRONT_COURT = list(range(17, 25))
MID_COURT = list(range(5, 17))
BACK_COURT = [1, 2, 3, 4]
ATTACK_TYPES = ['殺球', '推撲球', '平球']
CONTROL_TYPES = ['長球', '挑球', '切球', '網前球', '接殺防守']

def depth_label(area):
    if area in FRONT_COURT:
        return '前場'
    if area in MID_COURT:
        return '中場'
    if area in BACK_COURT:
        return '後場'
    return '出界/其他'


ordered_df = df.sort_values(GROUP_COLS + ['ball_round']).copy()
ordered_df['next_player'] = ordered_df.groupby(GROUP_COLS)['player'].shift(-1)
ordered_df['next_type'] = ordered_df.groupby(GROUP_COLS)['type'].shift(-1)
ordered_df['next_getpoint_player'] = ordered_df.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 網前打點：周天成在前場 hit_area 或打網前球。
net_df = ordered_df[
    (ordered_df['player'] == PLAYER) &
    ((ordered_df['hit_area'].isin(FRONT_COURT)) | (ordered_df['type'] == '網前球'))
].copy()
net_df['landing_depth'] = net_df['landing_area'].apply(depth_label)
net_df['punished_by_attack'] = net_df['next_player'].eq(OPPONENT) & net_df['next_type'].isin(['推撲球', '殺球'])
net_df['punished_directly'] = net_df['punished_by_attack'] & net_df['next_getpoint_player'].eq(OPPONENT)

risk_summary = net_df.groupby(['type', 'landing_depth']).agg(
    使用次數=('type', 'size'),
    被對手撲殺攻擊率=('punished_by_attack', 'mean'),
    被直接得分率=('punished_directly', 'mean')
).reset_index().sort_values(['被對手撲殺攻擊率', '使用次數'], ascending=[False, False])

safe_summary = risk_summary[risk_summary['使用次數'] >= 5].sort_values(['被對手撲殺攻擊率', '被直接得分率'])

print('周天成網前打點後，被對手撲殺/攻擊的風險：')
print(risk_summary)
print('\n較安全的網前銜接或變化方式：')
print(safe_summary.head(8))
print('\n建議：避免單調短網前；增加推深、挑後場或更貼網低弧線，降低對手推撲球/殺球機會。')

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = risk_summary.head(10).copy()
plot_df['模式'] = plot_df['type'] + '到' + plot_df['landing_depth']
ax.barh(plot_df['模式'][::-1], plot_df['被對手撲殺攻擊率'][::-1], color='#E15759')
ax.set_title('網前打點後被對手攻擊風險')
ax.set_xlabel('被對手撲殺/殺球率')
plt.tight_layout()

98. 如果對手連續以平球壓中路，周天成應如何調整回球方向與節奏，避免被持續壓制？

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 定義中路區域
middle_areas = [2, 3, 6, 7, 10, 11, 14, 15, 18, 19, 22, 23]

# 2. 在完整資料上建立下一拍資訊
df_next = df.copy()
df_next['next_player'] = df_next.groupby(GROUP_COLS)['player'].shift(-1)
df_next['next_type'] = df_next.groupby(GROUP_COLS)['type'].shift(-1)
df_next['next_getpoint_player'] = df_next.groupby(GROUP_COLS)['getpoint_player'].shift(-1)

# 3. 找出對手以平球壓中路，且下一拍是周天成回擊的情況
pressure_df = df_next[
    (df_next['player'] == OPPONENT) &
    (df_next['type'] == '平球') &
    (df_next['landing_area'].isin(middle_areas)) &
    (df_next['next_player'] == PLAYER)
].copy()

# 4. 定義周天成回應是否成功
# 這裡用「下一拍沒有直接讓對手得分」作為保守成功定義
pressure_df['response_success'] = (
    pressure_df['next_getpoint_player'] != OPPONENT
)

# 5. 統計不同回球方式的使用次數與成功率
response_summary = pressure_df.groupby('next_type').agg(
    使用次數=('next_type', 'count'),
    成功次數=('response_success', 'sum')
).reset_index()

response_summary['成功率'] = response_summary['成功次數'] / response_summary['使用次數']
response_summary = response_summary.sort_values(
    ['成功率', '使用次數'],
    ascending=[False, False]
)

print("對手以平球壓中路時，周天成不同回球方式的表現：")
print(response_summary)

99. 若觀察到對手在多拍後移動變慢，周天成應如何透過前後場調動提高下一拍主動得分機會？

In [ ]:
import pandas as pd
import numpy as np

PLAYER = 'CHOU Tien Chen'

# 1. 定義前場與後場區域
front_areas = list(range(17, 25))
back_areas = list(range(1, 5))

# 2. 只看周天成自己的擊球
chou_df = df[df['player'] == PLAYER].copy()

# 3. 計算對手移動距離，作為對手移動負擔的代理條件
chou_df['opponent_move_distance'] = np.sqrt(
    chou_df['opponent_move_x'] ** 2 + chou_df['opponent_move_y'] ** 2
)

# 4. 用高移動距離 + 多拍作為「對手在多拍後移動變慢」的代理情境
move_threshold = chou_df['opponent_move_distance'].quantile(0.75)

proxy_df = chou_df[
    (chou_df['ball_round'] >= 8) &
    (chou_df['opponent_move_distance'] >= move_threshold)
].copy()

# 5. 定義前後場調動策略
def classify_strategy(area):
    if area in front_areas:
        return '調動到前場'
    elif area in back_areas:
        return '調動到後場'
    else:
        return '其他'

proxy_df['strategy_type'] = proxy_df['landing_area'].apply(classify_strategy)
proxy_df = proxy_df[proxy_df['strategy_type'] != '其他'].copy()

# 6. 定義直接得分
proxy_df['direct_score'] = (
    proxy_df['getpoint_player'] == PLAYER
)

# 7. 統計不同策略的使用次數與直接得分率
summary = proxy_df.groupby('strategy_type').agg(
    使用次數=('strategy_type', 'count'),
    直接得分次數=('direct_score', 'sum')
).reset_index()

summary['直接得分率'] = summary['直接得分次數'] / summary['使用次數']

print(f"高移動負擔門檻: {move_threshold:.4f}")
print("當對手在多拍後移動負擔較高時，周天成前後場調動策略的效果：")
print(summary)

100. 當對手偏好在網前搶撲時，周天成應如何調整網前落點深淺與後續連接球種？

In [ ]:
import pandas as pd

PLAYER = 'CHOU Tien Chen'
OPPONENT = 'Kento MOMOTA'
GROUP_COLS = ['match_id', 'set', 'rally']

# 1. 定義對手偏好網前搶撲的代理條件
#    這裡用：周天成打出網前球 / 發短球後，對手下一拍常以推撲球或殺球處理
short_shot_types = ['網前球', '發短球']
pressure_types = ['推撲球', '殺球']

# 2. 定義前場落點深淺
shallow_front_areas = [21, 22, 23, 24]  # 靠網較淺
deep_front_areas = [17, 18, 19, 20]     # 較深前場

# 3. 在完整資料上建立後續兩拍資訊
df_seq = df.copy()
df_seq['next_player'] = df_seq.groupby(GROUP_COLS)['player'].shift(-1)
df_seq['next_type'] = df_seq.groupby(GROUP_COLS)['type'].shift(-1)
df_seq['next_getpoint_player'] = df_seq.groupby(GROUP_COLS)['getpoint_player'].shift(-1)
df_seq['next2_player'] = df_seq.groupby(GROUP_COLS)['player'].shift(-2)
df_seq['next2_type'] = df_seq.groupby(GROUP_COLS)['type'].shift(-2)

# 4. 找出周天成打出短球後，對手下一拍有網前搶撲傾向的情況
target_df = df_seq[
    (df_seq['player'] == PLAYER) &
    (df_seq['type'].isin(short_shot_types)) &
    (df_seq['next_player'] == OPPONENT) &
    (df_seq['next_type'].isin(pressure_types))
].copy()

# 5. 定義網前落點深淺
def classify_front_depth(area):
    if area in shallow_front_areas:
        return '較淺前場'
    if area in deep_front_areas:
        return '較深前場'
    return '其他'

target_df['front_depth'] = target_df['landing_area'].apply(classify_front_depth)
target_df = target_df[target_df['front_depth'] != '其他'].copy()

# 6. 統計不同深淺下，對手搶撲後周天成常見的後續連接球種
followup_df = target_df[target_df['next2_player'] == PLAYER].copy()

followup_summary = (
    followup_df.groupby(['front_depth', 'next2_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['front_depth', 'count'], ascending=[True, False])
)

print("當對手偏好在網前搶撲時，周天成不同前場落點深淺下的後續連接球種分布：")
print(followup_summary)

# 7. 補充：不同深淺下，被對手下一拍直接得分的比例
target_df['punished_immediately'] = (
    target_df['next_getpoint_player'] == OPPONENT
)

punish_summary = target_df.groupby('front_depth').agg(
    次數=('front_depth', 'count'),
    被對手下一拍直接得分次數=('punished_immediately', 'sum')
).reset_index()

punish_summary['被直接得分比例'] = (
    punish_summary['被對手下一拍直接得分次數'] / punish_summary['次數']
)

print("\n不同前場落點深淺下，被對手下一拍直接得分的比例：")
print(punish_summary)

100. 當對手偏好在網前搶撲時，周天成應如何調整網前落點深淺與後續連接球種？

In [ ]:
if len(df) == 0:
    raise ValueError("df 為空")

PLAYER = 'CHOU Tien Chen'
OPP = 'Kento MOMOTA'
group_cols = ['match_id', 'set', 'rally']

work = df.sort_values(group_cols + ['ball_round']).copy()
work['prev_player'] = work.groupby(group_cols)['player'].shift(1)
work['prev_type'] = work.groupby(group_cols)['type'].shift(1)
work['prev_landing_area'] = work.groupby(group_cols)['landing_area'].shift(1)
work['next_player'] = work.groupby(group_cols)['player'].shift(-1)
work['next_type'] = work.groupby(group_cols)['type'].shift(-1)

front_zones = list(range(17, 25))

# 對手網前搶撲：對手在前場打推撲球
pounce_df = work[
    (work['player'] == OPP) &
    (work['type'] == '推撲球') &
    (work['hit_area'].isin(front_zones)) &
    (work['prev_player'] == PLAYER)
].copy()

def net_depth(area):
    if area in [21, 22, 23, 24]:
        return '貼網前(Row6)'
    elif area in [17, 18, 19, 20]:
        return '稍退一格(Row5)'
    else:
        return '非前場'

pounce_df['prev_net_depth'] = pounce_df['prev_landing_area'].apply(net_depth)

# 看搶撲後若回合還延續，周天成下一次自己的連接球種
pounce_df['next_is_ctc'] = pounce_df['next_player'] == PLAYER

depth_summary = (
    pounce_df.groupby('prev_net_depth')
    .size()
    .reset_index(name='count')
)

followup_summary = (
    pounce_df[pounce_df['next_is_ctc']]
    .groupby(['prev_net_depth', 'next_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['prev_net_depth', 'count'], ascending=[True, False])
)

print("上一拍網前落點深淺分布：")
print(depth_summary)
print("\n被搶撲後仍延續時，周天成後續連接球種：")
print(followup_summary)